# Data Cleaning and Preprocessing

**Master's thesis:** From Dyads to Networks: Vendor Lock-in and Systemic Vulnerability in German Public IT Procurement

**Author:** Marco Stoehr

**Notebook:** 1 of 5 (run in order; see README)

**Thesis sections:** 3.1, Appendix A

## Purpose
This notebook covers the first data cleaning steps, including general steps such as NA handling and the removal of non-informative entries, as well as steps specifically designed for the TED CAN data. For both the supplier and the buyer (contracting authority) side, the following steps are implemented:

1. Splitting of multi-supplier and multi-buyer observations
2. Entity resolution
3. Parent mapping

## Input
- `can_data/export_CAN_<YYYY>.csv`: TED CAN CSV files for 2009–2023, one per year (see README for download)

## Main Output
- `data_analysis/data_clean_final.csv`

## Checkpoints
- `data_checkpoints/`: intermediate files written after major steps, so that later steps can be rerun without repeating earlier ones

In [1]:
# import libraries
import pandas as pd
import networkx
import matplotlib.pyplot as plt
import os
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss
import numpy as np
from rapidfuzz import process, fuzz
import rapidfuzz
import networkx as nx
from collections import Counter
import unicodedata
from itertools import combinations
import random


# set display-options for pandas dataframes
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
# check version
print(networkx.__version__)
print(faiss.__version__)
print(rapidfuzz.__version__)

3.5
1.13.2
3.14.5


In [3]:
# import data
years = range(2009, 2024)
dfs = []
cols = [
    "ID_NOTICE_CAN", "YEAR", "CAE_NAME", "CAE_TYPE", "MAIN_ACTIVITY",
    "WIN_NAME", "WIN_COUNTRY_CODE", "B_CONTRACTOR_SME",
    "CPV", "TYPE_OF_CONTRACT", "VALUE_EURO", "AWARD_VALUE_EURO",
    "B_FRA_AGREEMENT", "B_FRA_CONTRACT",
    "B_ON_BEHALF", "TOP_TYPE", "NUMBER_OFFERS",
    "B_EU_FUNDS", "LOTS_NUMBER", "FRA_ESTIMATED", "CRIT_CODE",
    "B_SUBCONTRACTED", "TAL_LOCATION_NUTS", "NUMBER_AWARDS", "DT_AWARD",
    "INFO_UNPUBLISHED", "ISO_COUNTRY_CODE", "INFO_ON_NON_AWARD",
    "WIN_NATIONALID", "VALUE_EURO_FIN_1", "VALUE_EURO_FIN_2"]

dfs = []
for year in years:
    filepath = f"can_data/export_CAN_{year}.csv"
    df_year = pd.read_csv(filepath, usecols=cols, low_memory=False)
    df_year = df_year[df_year["ISO_COUNTRY_CODE"] == "DE"]
    df_year = df_year[
        df_year["CPV"].astype(str).str.startswith("30") |
        df_year["CPV"].astype(str).str.startswith("48") |
        df_year["CPV"].astype(str).str.startswith("72")
    ]
    df_year["YEAR_ID"] = year
    dfs.append(df_year)

df_panel = pd.concat(dfs, ignore_index=True)

# Combine all years
df_panel = pd.concat(dfs, ignore_index=True)
print(f"\nTotal cases: {len(df_panel)}")
print(f"Years covered: {df_panel['YEAR_ID'].unique()}")
print(f"Unique buyers (CAE): {df_panel['CAE_NAME'].nunique()}")
print(f"Unique suppliers (WIN): {df_panel['WIN_NAME'].nunique()}")


Total cases: 52800
Years covered: [2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022
 2023]
Unique buyers (CAE): 8033
Unique suppliers (WIN): 14211


In [4]:
# --- Missing values in key identifier columns ---
print(f"Missing WIN_NAME: {df_panel['WIN_NAME'].isna().sum()}")
print(f"Missing CAE_NAME: {df_panel['CAE_NAME'].isna().sum()}")

# Treat empty / whitespace-only strings as missing as well
win_blank = df_panel["WIN_NAME"].astype(str).str.strip().isin(["", "nan", "None", "-"])
win_missing = df_panel["WIN_NAME"].isna() | win_blank
print(f"Missing incl. blank strings: {win_missing.sum()}")

# --- Drop cases without an identifiable supplier ---
n_before = len(df_panel)
data_clean = df_panel[~win_missing].copy()
n_after = len(data_clean)
n_dropped = n_before - n_after

print(f"\nCases before: {n_before}")
print(f"Cases dropped (no WIN_NAME): {n_dropped} ({n_dropped / n_before:.2%})")
print(f"Cases remaining: {n_after}")

# --- Why are they missing? Profile of the dropped records ---
dropped = df_panel[win_missing]

print("\nDropped cases by year:")
print(dropped["YEAR_ID"].value_counts().sort_index())

print("\nDropped cases by INFO_ON_NON_AWARD:")
print(dropped["INFO_ON_NON_AWARD"].value_counts(dropna=False).head(10))

print("\nDropped cases by INFO_UNPUBLISHED:")
print(dropped["INFO_UNPUBLISHED"].value_counts(dropna=False).head(10))


Missing WIN_NAME: 5484
Missing CAE_NAME: 0
Missing incl. blank strings: 5484

Cases before: 52800
Cases dropped (no WIN_NAME): 5484 (10.39%)
Cases remaining: 47316

Dropped cases by year:
YEAR_ID
2009     88
2010     81
2011    100
2012    101
2013    102
2014     68
2015     34
2016    172
2017    287
2018    454
2019    649
2020    817
2021    759
2022    970
2023    802
Name: count, dtype: int64

Dropped cases by INFO_ON_NON_AWARD:
INFO_ON_NON_AWARD
NaN                         2345
PROCUREMENT_UNSUCCESSFUL    2052
PROCUREMENT_DISCONTINUED    1087
Name: count, dtype: int64

Dropped cases by INFO_UNPUBLISHED:
INFO_UNPUBLISHED
N    4832
Y     652
Name: count, dtype: int64


In [5]:
# --- Split the exclusion into two conceptually distinct steps ---

non_award = df_panel["INFO_ON_NON_AWARD"].isin(
    ["PROCUREMENT_UNSUCCESSFUL", "PROCUREMENT_DISCONTINUED"]
)

# Step 1: contracts that never resulted in an award (out of scope by definition)
step1 = df_panel[~non_award].copy()
print(f"Excluded as non-awards: {non_award.sum()}")

# Step 2: genuine item non-response on the supplier name
win_missing = (
    step1["WIN_NAME"].isna()
    | step1["WIN_NAME"].astype(str).str.strip().isin(["", "nan", "None", "-"])
)
data_clean = step1[~win_missing].copy()
print(f"Excluded as missing WIN_NAME: {win_missing.sum()} "
      f"({win_missing.sum() / len(df_panel):.2%} of raw sample)")
print(f"Final analysis sample: {len(data_clean)}")

# --- Is the missingness time-varying? (share, not absolute counts) ---
by_year = pd.DataFrame({
    "total": df_panel.groupby("YEAR_ID").size(),
    "non_award": df_panel[non_award].groupby("YEAR_ID").size(),
    "missing_win": step1[win_missing].groupby("YEAR_ID").size(),
}).fillna(0).astype(int)
by_year["pct_missing_win"] = (by_year["missing_win"] / by_year["total"] * 100).round(2)
by_year["pct_non_award"] = (by_year["non_award"] / by_year["total"] * 100).round(2)
print("\n", by_year)

# --- Do the dropped awards differ in contract value? ---
val = pd.to_numeric(step1["VALUE_EURO"], errors="coerce")
print("\nValue of records WITH supplier name:")
print(val[~win_missing].describe())
print("\nValue of records WITHOUT supplier name:")
print(val[win_missing].describe())

# --- Overlap: are missing names concentrated among unpublished notices? ---
print("\nMissing WIN_NAME x INFO_UNPUBLISHED:")
print(pd.crosstab(win_missing, step1["INFO_UNPUBLISHED"], dropna=False))

Excluded as non-awards: 3139
Excluded as missing WIN_NAME: 2345 (4.44% of raw sample)
Final analysis sample: 47316

          total  non_award  missing_win  pct_missing_win  pct_non_award
YEAR_ID                                                               
2009      1470          0           88             5.99           0.00
2010      1731          0           81             4.68           0.00
2011      1858          0          100             5.38           0.00
2012      1970          0          101             5.13           0.00
2013      2101          0          102             4.85           0.00
2014      2017          0           68             3.37           0.00
2015      2232          1           33             1.48           0.04
2016      2660         56          116             4.36           2.11
2017      3019        176          111             3.68           5.83
2018      3452        289          165             4.78           8.37
2019      4358        406      

In [6]:
# check unique values for buyers and suppliers
print(f"Remaining cases: {len(data_clean)}")
print(f"Unique buyers (CAE): {data_clean['CAE_NAME'].nunique()}")
print(f"Unique suppliers (WIN): {data_clean['WIN_NAME'].nunique()}")

Remaining cases: 47316
Unique buyers (CAE): 7605
Unique suppliers (WIN): 14211


In [7]:
data_clean.to_csv("data_checkpoints/data_start.csv")

In [8]:
data_clean = pd.read_csv("data_checkpoints/data_start.csv")

## Data Cleaning: Supplier Side

### Multi-supplier cases
Some award notices include more than one supplier, indicating that several companies were awarded. The most frequent approach of documenting such cases uses "---" as a seperator, but also numerical lists appear within the dataset. The following section splits such cases into several observations - one observation for the contract awarding authority and each supplier awarded with a contract.

In [9]:
# first handling cases with "---" (more than one supplier)
data_clean_ = data_clean
mask_multi = data_clean_['WIN_NAME'].str.contains('---', na=False)
print(f"Cases with '---': {mask_multi.sum()}")

Cases with '---': 515


In [10]:
# new column indicating splits
data_clean_['IS_SPLIT'] = False

# select cases with '---'
mask_multi = data_clean_['WIN_NAME'].str.contains('---', na=False)
df_single = data_clean_[~mask_multi].copy()
df_multi = data_clean_[mask_multi].copy()

# split
split_rows = []
for _, row in df_multi.iterrows():
    firms = [f.strip() for f in row['WIN_NAME'].split('---') if f.strip()]
    n = len(firms)
    for firm in firms:
        new_row = row.copy()
        new_row['WIN_NAME'] = firm
        new_row['VALUE_EURO'] = row['VALUE_EURO'] / n
        new_row['IS_SPLIT'] = True
        split_rows.append(new_row)

df_split = pd.DataFrame(split_rows)

# concatenate
data_clean_ = pd.concat([df_single, df_split], ignore_index=True)

print(f"Original cases: {mask_multi.sum()}")
print(f"New cases: {len(df_split)}")
print(f"Cases: {len(data_clean_)}")

Original cases: 515
New cases: 1387
Cases: 48188


In [11]:
# Splitting on '---' leaves residual dash-only placeholders behind:
#   '----'  -> '-'
#   '-----' -> '--'
# These are placeholders, not supplier names. They are removed later together
# with the other uninformative entries ("Keine Angabe", "entfällt", ...).
_placeholder = data_clean_['WIN_NAME'].str.fullmatch(r'[\s\-]*')
print(f"Dash-only placeholders introduced by the split: {_placeholder.sum()}")
print(data_clean_.loc[_placeholder, 'WIN_NAME'].value_counts().to_string())
assert _placeholder.sum() == 4, _placeholder.sum()

Dash-only placeholders introduced by the split: 4
WIN_NAME
-     2
--    2


In [12]:
# --- Detection of semicolon multi-supplier candidates (after the '---' split) ---
# Heuristic: a semicolon followed by a legal form or list numbering, intended to
# separate genuine multi-supplier entries from names that merely contain a semicolon.
SEMICOLON_DETECTION = r';.*(?:gmbh|mbh|ag|kg|se|inc|ltd|b\.v\.|plc|\d+\.)'

_names = (data_clean_['WIN_NAME'].dropna().str.strip()
          .value_counts().rename_axis('WIN_NAME').reset_index(name='n'))

_detected = _names.loc[
    _names['WIN_NAME'].str.lower().str.contains(
        SEMICOLON_DETECTION, case=False, regex=True, na=False)]

assert len(_detected) == 194, len(_detected)

print(f"Detected candidates: {len(_detected)} names / {_detected['n'].sum()} awards")
print(_detected.to_string(index=False))

Detected candidates: 194 names / 240 awards
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [13]:
# Manual classification of the 194 names detected above.
# Categories: 'split' | 'keep' | 'vertreter' | 'drop'
SEMICOLON_ACTIONS = {
    # ── KEEP (address, department, or single firm with descriptor after ";") ──
    "wehking; ernst gmbh & co. kg":                                             ("keep", "Wehking Ernst GmbH & Co. KG"),
    "PROSOZ Herten Softwareentwicklungs- und Beratungsgesellschaft für Gemeinden; Städte und Kreise mbH": ("keep", "PROSOZ Herten Softwareentwicklungs- und Beratungsgesellschaft für Gemeinden, Städte und Kreise mbH"),
    "logica deutschland gmbh & co.kg; humperdinckstr.1; 53773 hennef":          ("keep", "logica deutschland gmbh & co.kg"),
    "ibm deutschland gmbh; global business services":                           ("keep", "IBM Deutschland GmbH"),
    "klinikum der universität münchen; institut für notfallmedizin und medizinmanagement": ("keep", "klinikum der universität münchen"),
    "ricoh deutschland gmbh; business & service center düsseldorf":             ("keep", "Ricoh Deutschland GmbH"),
    "hays ag; immermannstr.65; 40210 düsseldorf":                               ("keep", "hays ag"),
    "t-systems international gmbh; computing & desktop services":               ("keep", "t-systems international gmbh"),
    "t-systems international gmbh; public sector & health care; segment länder und kommunen": ("keep", "t-systems international gmbh"),
    "t-systems international gmbh; segment länder und kommunen":                ("keep", "t-systems international gmbh"),
    "open text software gmbh; professional services":                           ("keep", "Open Text Software GmbH"),
    "novatec; sicherheit und logistik gmbh":                                    ("keep", "novatec sicherheit und logistik gmbh"),
    "networker; projektberatung gmbh":                                          ("keep", "networker projektberatung gmbh"),
    "xdot gmbh; ein unternehmen der convotis gmbh":                             ("keep", "xdot gmbh"),
    "mckinsey & company; inc.":                                                 ("keep", "mckinsey & company inc."),
    "mog-mbh; mittelständische organisationsgesellschaft mit beschränkter haftung; geschäftsführer reinhold decker": ("keep", "mog-mbh"),
    "schütze ag; seit 11.11.2000 nortal ag":                                    ("keep", "nortal ag"),
    "siemens ag; ger ic bt ost bln fss sec.":                                   ("keep", "siemens ag"),
    "toshiba tec germany imaging systems gmbh nl ost; vertriebsbüro magdeburg": ("keep", "toshiba tec germany imaging systems gmbh"),
    "sv saxonia verlag für recht; wirtschaft und kultur gmbh":                  ("keep", "sv saxonia verlag für recht, wirtschaft und kultur gmbh"),
    "technidata; it-service gmbh":                                              ("keep", "technidata it-service gmbh"),
    "price systems limited; zweigniederlassung rüsselsheim":                    ("keep", "price systems limited"),
    "bscd business solutions; consulting and development gmbh":                 ("keep", "bscd business solutions consulting and development gmbh"),
    "bscd business soltuions; consulting and development gmbh":                 ("keep", "bscd business solutions consulting and development gmbh"),
    "burg; software & service für die vermessung":                              ("keep", "burg software & service für die vermessung"),
    "businesstechnology consulting ag; öffentlicher sektor":                    ("keep", "businesstechnology consulting ag"),
    "cern; the european organization for nuclear research":                     ("keep", "cern"),
    "delphi; gesellschaft für forschung; beratung und projektentwicklung":      ("keep", "delphi gesellschaft für forschung beratung und projektentwicklung"),
    "urs corporation; federal services":                                        ("keep", "urs corporation"),
    "wvi prof.wermuth; verkehrsforschung und infrastrukturplanung gmbh":        ("keep", "wvi prof.wermuth verkehrsforschung und infrastrukturplanung gmbh"),
    "milch & zucker; talent acquisition & talent management company ag":        ("keep", "milch & zucker talent acquisition & talent management company ag"),
    "tu clausthal; institut für geik und markscheidewesen":                     ("keep", "tu clausthal"),
    "pixelpark ag; pixelpark agentur köln; stefan schopp":                      ("keep", "pixelpark ag"),
    "towers; perrin; forster & crosby; inc.":                                   ("keep", "towers perrin forster & crosby inc."),
    "int.ras.pdv gesellschaft für schulung; dokumentation und multimedia mbh":  ("keep", "int.ras.pdv gesellschaft für schulung dokumentation und multimedia mbh"),
    "inforsacom informationssysteme gmbh; neu isenburg":                        ("keep", "inforsacom informationssysteme gmbh"),
    "in-time; it consulting center gmbh":                                       ("keep", "in-time it consulting center gmbh"),
    "igepa group; e. michaelis & co. (gmbh & co. kg)":                         ("keep", "igepa group e. michaelis & co. gmbh & co. kg"),
    "herausgebergemeinschaft wertpapiermitteilungen; keppler; lehmann gmbh & co. kg": ("keep", "herausgebergemeinschaft wertpapiermitteilungen keppler lehmann gmbh & co. kg"),
    "hochschule ostwestfalen-lippe; fachbereich bauingenieurwesen":             ("keep", "hochschule ostwestfalen-lippe"),
    "konica minolta business solutions deutschland gmbh; langenhagen":          ("keep", "Konica Minolta Business Solutions Deutschland GmbH"),
    "konica minolta; business solutions deutschland gmbh; direktvertrieb region süd; niederlassung münchen": ("keep", "Konica Minolta Business Solutions Deutschland GmbH"),
    "logica deutschland gmbh & co. kg; hamborner str.53; 40472 düsseldorf":    ("keep", "logica deutschland gmbh & co. kg"),
    "land bremen; vertreten durch den senator für finanzen als träger der osci-leitstelle": ("keep", "land bremen"),
    "babiel gmbh; düsseldorf":                                                  ("keep", "babiel gmbh"),
    "becom informationssysteme gmbh; konrad-zuse-str.14; 58239 schwerte":       ("keep", "becom informationssysteme gmbh"),
    "bechtle gmbh; it-systemhaus magdeburg":                                    ("keep", "bechtle gmbh"),
    "barthauer software gmbh; geoinformatik wasserwirtschaft":                  ("keep", "barthauer software gmbh"),
    "firma toshiba; tec germany imaging system gmbh nl":                        ("keep", "toshiba tec germany imaging systems gmbh"),
    "firma siemens ag; deutschland; healthcare sector":                         ("keep", "siemens ag"),
    "em-faktor; die social profit agentur gmbh":                                ("keep", "em-faktor die social profit agentur gmbh"),
    "eumetsys; c/o ernst basler + partner gmbh":                                ("keep", "eumetsys"),
    "electricity coordinating center; ltd":                                     ("keep", "electricity coordinating center ltd"),
    "emerging portfolio fund research; inc. – epfr global":                     ("keep", "emerging portfolio fund research inc."),
    "fa. j.fuhrmann büro-center; niederlassung der toshiba tec germany gmbh":  ("keep", "toshiba tec germany gmbh"),
    "schweers; kemps & schuhmann unternehmensberatung gmbh & co. kg":          ("keep", "schweers kemps & schuhmann unternehmensberatung gmbh & co. kg"),
    "allen systems group; inc.":                                                ("keep", "allen systems group inc."),
    "amazon web services; inc":                                                 ("keep", "amazon web services inc."),
    "intex solutions; inc.":                                                    ("keep", "intex solutions inc."),
    "lewtan technologies; inc.":                                                ("keep", "lewtan technologies inc."),
    "levi;ray & shoup inc":                                                     ("keep", "levi ray & shoup inc."),
    "levi; ray & shoup; inc. deutschland":                                      ("keep", "levi ray & shoup inc. deutschland"),
    "krollpfeiffer; nl der abe gmbh":                                           ("keep", "krollpfeiffer nl der abe gmbh"),
    "aral ag; aral card-service":                                               ("keep", "aral ag"),
    "micro focus; vertreten durch computacenter ag & co. ohg":                  ("keep", "computacenter ag & co. ohg"),
    "corporate express deutschland gmbh & co; vertriebs kg":                   ("keep", "corporate express deutschland gmbh & co. vertriebs kg"),
    "chg - meridian; deutsche computer leasing ag":                             ("keep", "CHG-Meridian AG"),
    "infoman ag bietergem; infoman ag bearing point gmbh":                      ("keep", "infoman ag"),
    "team weber gmbh; xerox vertragspartner":                                   ("keep", "team weber gmbh"),
    "dns:net internet service gmbh; hrb11714ff ag frankfurt (oder); sitz der gesellschaft: bernau bei berlin": ("keep", "dns:net internet service gmbh"),
    "3 zuschlagspartner: fa. usu ag; möglingen; fa. dr. seidel; unterhaching; fa. aracom; augsburg": ("drop", None),
    "3 zuschlagspartner: fa. hörn software; kürnbach; fa. götzfried ag; wiesbaden; fa. dv-ratio; leutenbach": ("drop", None),
    "12 zuschlagspartner: fa. lupus; nufringen; fa. friedrich; stuttgart; fa. gfi-it; stuttgart; fa. edv machold; stuttgart; fa. logica; stuttgart; fa. dr. seidel; unterhaching; fa. dv-ratio; leutenbach; fa. scs; stuttgart; fa. consist; kiel; fa. dr. karb; stuttgart; elke müller; filderstadt; fa. hays; stuttgart": ("drop", None),

    # ── VERTRETER (Bietergemeinschaft – keep named representative only) ────────
    "Bietergemeinschaft SINC GmbH und alphaBit GmbH; vertreten durch alphaBit GmbH": ("vertreter", "alphaBit GmbH"),
    "bietergemeinschaft aios-imtb-infora; vertreten durch aios gmbh":           ("vertreter", "aios gmbh"),
    "bietergemeinschaft gruler & cleve consulting; vertreten durch die gruler & cleve consulting gmbh": ("vertreter", "gruler & cleve consulting gmbh"),
    "bietergemeinschaft itomig gmbh und lanedo gmbh; vertreten durch itomig gmbh": ("vertreter", "itomig gmbh"),
    "bietergemeinschaft treorbis/q_perior ag; vertreten durch die treorbis fba gmbh": ("vertreter", "treorbis fba gmbh"),
    "bietergemeinschaft sparkassen rating und risikosysteme gmbh/dsv gmbh; vertreten durch sparkassen rating und risikosysteme gmbh": ("vertreter", "sparkassen rating und risikosysteme gmbh"),
    "tata consultancy services tcs; representing the consortium formed by tata consultancy services tcs; i.r.i.s.; ctg belgium and ctg luxembourg": ("vertreter", "tata consultancy services"),
    "siemens it solutions and services s.a./n.v.; representing siemens it solutions and services s.a./n.a. and trasys s.a. consortium": ("vertreter", "siemens it solutions and services"),
    "trasys s.a.; representing the aetos consortium formed by trasys s.a. (leader); amplexor n.v. and everis spain s.l.": ("vertreter", "trasys s.a."),
    "intrasoft international s.a.; representing the consortium formed by intrasoft international s.a. and siveco romania s.a.": ("vertreter", "intrasoft international s.a."),
    "teamplan gmbh als bietergemeinschaft mit ludes architekten; recklinghausen": ("vertreter", "teamplan gmbh"),
    "bietergemeinschaft mit bevollmächtigten hbt gmbh; stadthausbrücke 3; 20355 hamburg (+ teilnehmer gisworks gbr und traffgo road gmbh)": ("vertreter", "hbt gmbh"),
    "arge leitstellenplanung; c/o. d.i.e. projelt gmbh":                        ("vertreter", "d.i.e. projelt gmbh"),
    "arbeitsgemeinschaft intec/cgs; bv cgs gmbh":                               ("vertreter", "cgs gmbh"),
    "bietergemeinschaft dresearch digital media systems gmbh (vertreter); edv beratung hempel; schütze consulting informationssysteme gmbh": ("vertreter", "dresearch digital media systems gmbh"),
    "consortium of sgi aviation services (leader); bureau veritas; senasa and egis avia": ("vertreter", "sgi aviation services"),
    "consortium between egis avia (leader); bureau veritas; caa international; sgi aviation services and lufthansa consulting": ("vertreter", "egis avia"),
    "cronos bietergemeinschaft; bestehend aus cronos unternehmensberatung gmbh als vertreter der bietergemeinschaft; cronos billing consulting gmbh; crm consulting ag; smart utility consulting gmbh; cronos services gmbh; coneco consulting gmbh; cronos information management consulting gmbh und cronos consulting gmbh": ("vertreter", "cronos unternehmensberatung gmbh"),
    "bietergemeinschaft itec systems ag; düsseldorf und init ag für digitale kommunikation; berlin": ("vertreter", "init ag für digitale kommunikation"),

    # ── SPLIT (genuine multi-supplier) ────────────────────────────────────────
    "Siemens AG; Siemens IT Solutions and Services":                            ("split", ["Siemens AG", "Siemens IT Solutions and Services"]),
    "Allgeier Enterprise Service AG; Bietergemeinschaft Allgeier Enterprise Service AG/Allgeier Experts SE": ("split", ["Allgeier Enterprise Service AG", "Allgeier Experts SE"]),
    "ricoh deutschland gmbh; ricoh it services & solutions":                    ("split", ["Ricoh Deutschland GmbH", "Ricoh IT Services & Solutions"]),
    "IBM Deutschland GmbH; IDS Scheer GmbH; SAP AG":                           ("split", ["IBM Deutschland GmbH", "IDS Scheer GmbH", "SAP AG"]),
    "bechtle gmbh; systemhaus magdeburg":                                       ("split", ["bechtle gmbh", "systemhaus magdeburg"]),
    "orange business services sa; colt technology services gmbh":              ("split", ["orange business services sa", "colt technology services gmbh"]),
    "bridgingit; pta; ctb; apsec; norcom":                                      ("split", ["bridgingit", "pta", "ctb", "apsec", "norcom"]),
    "microstaxx gmbh; cancom gmbh; bechtle gmbh & co. kg; think about it":     ("split", ["microstaxx gmbh", "cancom gmbh", "bechtle gmbh & co. kg", "think about it gmbh"]),
    "msg systems ag; acg automation consulting group gmbh; information works unternehmensberatung gmbh; information bridge gmbh; viadee unternehmensberatung gmbh": ("split", ["msg systems ag", "acg automation consulting group gmbh", "information works unternehmensberatung gmbh", "information bridge gmbh", "viadee unternehmensberatung gmbh"]),
    "msg systems ag; pass it consulting dipl. inf. g. rienecker gmbh & co kg; steria mummert consulting ag; bearing point gmbh": ("split", ["msg systems ag", "pass it consulting dipl. inf. g. rienecker gmbh & co kg", "steria mummert consulting ag", "bearing point gmbh"]),
    "sap deutschland ag & co. kg; strike consulting gmbh; steria mummert consulting ag": ("split", ["sap deutschland ag & co. kg", "strike consulting gmbh", "steria mummert consulting ag"]),
    "csc deutschland solutions gmbh; vega deutschland gmbh; steria mummert consulting ag": ("split", ["csc deutschland solutions gmbh", "vega deutschland gmbh", "steria mummert consulting ag"]),
    "controllit ag; acg automation consulting group gmbh; carmao gmbh; bearingpoint gmbh": ("split", ["controllit ag", "acg automation consulting group gmbh", "carmao gmbh", "bearingpoint gmbh"]),
    "codesense gmbh; it-designers gmbh; logica deutschland gmbh & co. kg":     ("split", ["codesense gmbh", "it-designers gmbh", "logica deutschland gmbh & co. kg"]),
    "comdok gmbh; sankt augustin (lose 1 und 3). universum kommunikation und medien ag; berlin (los 2).": ("split", ["comdok gmbh", "universum kommunikation und medien ag"]),
    "opitz consulting gmbh; 51647 gummersbach; promatis software gmbh; 76275 ettlingen; virtual7 gmbh; 76185 karlsruhe": ("split", ["opitz consulting gmbh", "promatis software gmbh", "virtual7 gmbh"]),
    "adesso ag; steria mummert consulting ag; metorit gmbh; hays ag; logica deutschland gmbh & co. kg; codesense gmbh": ("split", ["adesso ag", "steria mummert consulting ag", "metorit gmbh", "hays ag", "logica deutschland gmbh & co. kg", "codesense gmbh"]),
    "adesso ag; metafinanz gmbh; consensit gmbh":                               ("split", ["adesso ag", "metafinanz gmbh", "consensit gmbh"]),
    "adesso ag; logica deutschland gmbh & co. kg; consensit gmbh; metorit gmbh; codesense gmbh; ibm deutschland gmbh": ("split", ["adesso ag", "logica deutschland gmbh & co. kg", "consensit gmbh", "metorit gmbh", "codesense gmbh", "ibm deutschland gmbh"]),
    "adesso ag; codesense gmbh; innovas gmbh":                                  ("split", ["adesso ag", "codesense gmbh", "innovas gmbh"]),
    "adesso ag; cassini gmbh; mid gmbh; metafinanz gmbh; steria mummert consulting ag; csc deutschland solutions gmbh": ("split", ["adesso ag", "cassini gmbh", "mid gmbh", "metafinanz gmbh", "steria mummert consulting ag", "csc deutschland solutions gmbh"]),
    "it score; codesense gmbh; adesso ag; metafinanz gmbh; cassini gmbh; ibm deutschland gmbh": ("split", ["it score", "codesense gmbh", "adesso ag", "metafinanz gmbh", "cassini gmbh", "ibm deutschland gmbh"]),
    "andrena objects ag; goetzpartners management consultants gmbh; nttdata deutschland gmbh; sap deutschland ag & co. kg": ("split", ["andrena objects ag", "goetzpartners management consultants gmbh", "nttdata deutschland gmbh", "sap deutschland ag & co. kg"]),
    "andrena objects ag; nttdata deutschland gmbh; zeb/rolfes.schierenbeck.associates gmbh": ("split", ["andrena objects ag", "nttdata deutschland gmbh", "zeb/rolfes.schierenbeck.associates gmbh"]),
    "ehealth experts gmbh; jan hueter; ctb gmbh; geos mbh; systhemis ag; gulp consulting services gmbh": ("split", ["ehealth experts gmbh", "jan hueter", "ctb gmbh", "geos mbh", "systhemis ag", "gulp consulting services gmbh"]),
    "bg alphabit gmbh; sinc gmbh":                                              ("split", ["alphabit gmbh", "sinc gmbh"]),
    "bietergemeinschaft scoop software gmbh; transver gmbh; scoop gmbh":       ("split", ["scoop software gmbh", "transver gmbh", "scoop gmbh"]),
    "bietergemeinschaft accantec consulting ag; bde gmbh und c1 fincon gmbh":  ("split", ["accantec consulting ag", "bde gmbh", "c1 fincon gmbh"]),
    "bietergemeinschaft: fernbach financial software s.a.; universalowner sarl; institut für operations research": ("split", ["fernbach financial software s.a.", "universalowner sarl", "institut für operations research"]),
    "bietergemeinschaft exccon gmbh; ncc management consulting gmbh und ncc solutions gmbh": ("split", ["exccon gmbh", "ncc management consulting gmbh", "ncc solutions gmbh"]),
    "bietergemeinschaft codia software gmbh; meppen; und d.velop ag; gescher": ("split", ["codia software gmbh", "d.velop ag"]),
    "bietergemeinschaft booz & company gmbh; compugroup medical ag; kococonnector ag": ("split", ["booz & company gmbh", "compugroup medical ag", "kococonnector ag"]),
    "bietergemeinschaft; bestehend aus eset deutschland gmbh und eset spol.s.r.o.": ("split", ["eset deutschland gmbh", "eset spol.s.r.o."]),
    "bietergemeinschaft: siemens ag; siemens it solutions and services und mid gmbh": ("split", ["siemens ag", "siemens it solutions and services", "mid gmbh"]),
    "bietergemeinschaft aus siemens it solutions and services gmbh; fujitsu technology solutions gmbh und logica deutschland gmbh & co. kg": ("split", ["siemens it solutions and services gmbh", "fujitsu technology solutions gmbh", "logica deutschland gmbh & co. kg"]),
    "bietergemeinschaft arbeit und leben e.v.; landesarbeitsgemeinschaft berlin und sirup agentur für neue medien gmbh & co.kg": ("split", ["arbeit und leben e.v.", "sirup agentur für neue medien gmbh & co.kg"]),
    "fmb blickle gmbh; villingen-schwenningen und festo didactic gmbh & co. kg; esslingen": ("split", ["fmb blickle gmbh", "festo didactic gmbh & co. kg"]),
    "konica minolta gmbh (los 1); ricoh gmbh (lose 2 und 3)":                  ("split", ["Konica Minolta Business Solutions Deutschland GmbH", "Ricoh Deutschland GmbH"]),
    "los 1: gisa gmbh; pdv-systeme gmbh":                                      ("split", ["gisa gmbh", "pdv-systeme gmbh"]),
    "los 1: isoft health gmbh; los 2: siemens healthcare sektor":              ("split", ["isoft health gmbh", "siemens ag"]),
    "los 1; 3; 4 hewlwtt-packard gmbh; los 2 systrade gmbh":                   ("split", ["Hewlett-Packard GmbH", "systrade gmbh"]),
    "los 1 u. 3. fa. siemens; essen; los 2: fa. modutec; au bei freiburg":     ("split", ["siemens ag", "modutec gmbh"]),
    "1. ibs bürosysteme gmbh; 2. horn gmbh":                                   ("split", ["IBS Bürosysteme GmbH & Co. KG", "horn gmbh"]),
    "1. fujitsu technology solutions gmbh; 2. hewlett packard gmbh":           ("split", ["fujitsu technology solutions gmbh", "Hewlett-Packard GmbH"]),
    "1. dell halle gmbh; 2.fujitsu technology solutions gmbh; 3. hewelett-packard gmbh": ("split", ["Dell Halle GmbH", "fujitsu technology solutions gmbh", "Hewlett-Packard GmbH"]),
    "1.) accenture gmbh; 2.) bearingpoint gmbh; 3.) d-fine gmbh":              ("split", ["accenture gmbh", "bearingpoint gmbh", "d-fine gmbh"]),
    "1. pcs beratungskontor ag; 2. bearingpoint gmbh; 3. ib-bank systems gmbh": ("split", ["pcs beratungskontor ag", "bearingpoint gmbh", "ib-bank systems gmbh"]),
    "(1) phi partners global ltd. (german branch); (2) msggillardon ag; (3) devnet gmbh.": ("split", ["phi partners global ltd.", "msggillardon ag", "devnet gmbh"]),
    "(1) phi partners global ltd. (german branch); (2) it professionals ag; (3) allgeier experts go gmbh.": ("split", ["phi partners global ltd.", "it professionals ag", "allgeier experts go gmbh"]),
    "(1) phi partners global ltd. (german branch); (2) devnet gmbh; (3) d-fine gmbh.": ("split", ["phi partners global ltd.", "devnet gmbh", "d-fine gmbh"]),
    "teil-los 3.1 – 16.000 pt: hansevision gmbh; teil-los 3.2 – 16.000 pt: entserv deutschland gmbh": ("split", ["hansevision gmbh", "entserv deutschland gmbh"]),
    "teil-los 1.1 – 8.000 pt: fujitsu technology solutions gmbh; teil-los 1.2 – 8.000 pt: hansevision gmbh; teil-los 1.3 – 8.000 pt: bewerbergemeinschaft allgeier experts": ("split", ["fujitsu technology solutions gmbh", "hansevision gmbh", "allgeier experts"]),
    "für los 1: on line datensysteme gmbh (granatenstraße 19-20; 13409 berlin); für los 2 und 3: bechtle gmbh & co. kg (tharandter straße 13; 01159 dresden)": ("split", ["on line datensysteme gmbh", "bechtle gmbh & co. kg"]),
    "gisa gmbh; pdv-systeme gmbh; init ag":                                    ("split", ["gisa gmbh", "pdv-systeme gmbh", "init ag für digitale kommunikation"]),

    # ── DROP (unresolvable large consortia, "diverse:", Losen-Listen ohne Volumen) ─
    "diverse: clientfocus gmbh; csc deutschland solutions gmbh; q_perior ag (vormals esprit consulting ag); hays ag; logica deutschland gmbh & co. kg.; norcom information technology ag; reutax ag": ("drop", None),
    "diverse: bridgingit gmbh; computacenter ag & co. ohg; logica deutschland gmbh & co. kg.; reutax ag; sd&t ag; tüv rheinland consulting gmbh": ("drop", None),
    "diverse: bridgingit gmbh; capgemini outsourcing services gmbh; ciber ag; computacenter ag & co. ohg; csc deutschland solutions gmbh; q_perior ag (vormals esprit consulting ag); gulp consulting service gmbh; hays ag; i.s.a.dresden gmbh & co. kg.; ibm deutschland gmbh; logica deutschland gmbh & co. kg.; norcom information technology ag; reutax ag; schmidt & partner gmbh; sps consulting & services gmbh": ("drop", None),
    "diverse: accenture gmbh; bearing point gmbh; bluecarat ag; bridgingit gmbh; csc deutschland solutions gmbh; q_perior ag (vormals esprit consulting ag); hays ag; ibm deutschland gmbh; logica deutschland gmbh & co. kg.; norcom information technology ag; reutax ag; seven principles ag; steria mummert consulting ag": ("drop", None),
    "european dynamics luxembourg s.a.; 45. av j.f. kennedy; l-9053 ettelbrück luxembourg in bietergemeinschaft mit european dynamics s.a. 209; kifissias ave. & arkadiou str. 151 24; maroussi; athen; griechenland": ("drop", None),
}

In [14]:
# --- Apply SEMICOLON_ACTIONS ---
# keep      -> single supplier, descriptor after ";" removed
# vertreter -> consortium with a named representative; award attributed to that firm
# split     -> genuine multi-supplier award; one row per firm, value divided by n
# drop      -> unresolvable listing, record removed

actions_lower = {k.lower().strip(): v for k, v in SEMICOLON_ACTIONS.items()}
assert len(actions_lower) == len(SEMICOLON_ACTIONS), (
    "case-insensitive key collision", len(SEMICOLON_ACTIONS) - len(actions_lower))

_win_key = data_clean_["WIN_NAME"].str.lower().str.strip()

# Every mapping key must match at least one record.
_unused = set(actions_lower) - set(_win_key)
assert not _unused, sorted(_unused)

mask_affected = _win_key.isin(actions_lower)
df_unaffected = data_clean_[~mask_affected].copy()
df_affected   = data_clean_[mask_affected].copy()

new_rows = []
n_keep = n_vertreter = n_split_src = n_dropped = 0

for _, row in df_affected.iterrows():
    action, payload = actions_lower[row["WIN_NAME"].lower().strip()]

    if action == "drop":
        n_dropped += 1
        continue

    if action in ("keep", "vertreter"):
        new_row = row.copy()
        new_row["WIN_NAME"] = payload
        new_rows.append(new_row)
        n_keep += (action == "keep")
        n_vertreter += (action == "vertreter")

    elif action == "split":
        n = len(payload)
        n_split_src += 1
        for firm in payload:
            new_row = row.copy()
            new_row["WIN_NAME"]   = firm
            new_row["VALUE_EURO"] = row["VALUE_EURO"] / n
            new_row["IS_SPLIT"]   = True
            new_rows.append(new_row)

df_fixed    = pd.DataFrame(new_rows)
n_before    = len(data_clean_)
data_clean_ = pd.concat([df_unaffected, df_fixed], ignore_index=True)

assert len(data_clean_) == n_before - mask_affected.sum() + len(df_fixed)
assert not data_clean_["WIN_NAME"].str.lower().str.strip().isin(actions_lower).any()

print(f"Affected records:   {mask_affected.sum()}")
print(f"  kept/cleaned:     {n_keep}")
print(f"  representative:   {n_vertreter}")
print(f"  split:            {n_split_src} records -> {len(df_fixed) - n_keep - n_vertreter} rows")
print(f"  dropped:          {n_dropped}")
print(f"Cases: {n_before} -> {len(data_clean_)}")

Affected records:   192
  kept/cleaned:     88
  representative:   31
  split:            65 records -> 190 rows
  dropped:          8
Cases: 48188 -> 48305


In [15]:
# The mapping must be a subset of the detection above; nothing else may enter it.
_keys = {k.lower().strip() for k in SEMICOLON_ACTIONS}
_hits = set(_detected['WIN_NAME'].str.lower())

assert len(_keys) == 151, len(_keys)
assert _keys <= _hits, sorted(_keys - _hits)

_unmapped = _detected.loc[~_detected['WIN_NAME'].str.lower().isin(_keys)]
print(f"Classified: {len(_keys)} | not classified: {len(_unmapped)} names / {_unmapped['n'].sum()} awards")

Classified: 151 | not classified: 39 names / 48 awards


In [16]:
print(_unmapped.to_string(index=False))

In [17]:
# --- Drop framework award notices listing all admitted suppliers ---
# These notices enumerate the complete roster of suppliers admitted to a framework
# agreement instead of the parties to an individual award; in some cases more than
# fifty firms are listed. No award can be attributed to a single supplier, so the
# records are removed. Detection is structural: six or more parties, i.e. at least
# five separators, in entries not already classified in SEMICOLON_ACTIONS.

already_handled = {k.lower().strip() for k in SEMICOLON_ACTIONS}

mask_large_consortium = (
    ~data_clean_["WIN_NAME"].str.lower().str.strip().isin(already_handled)
    & (data_clean_["WIN_NAME"].str.count(";") >= 5)
)

# Reviewed state as of manual inspection.
assert mask_large_consortium.sum() == 10, mask_large_consortium.sum()

print(f"Framework roster entries removed: {mask_large_consortium.sum()}")
print(data_clean_.loc[mask_large_consortium, "WIN_NAME"]
      .str.slice(0, 90).to_string())

n_before = len(data_clean_)
data_clean_ = data_clean_.loc[~mask_large_consortium].copy().reset_index(drop=True)

assert len(data_clean_) == n_before - 10
print(f"\nRemaining cases: {len(data_clean_)}")

Framework roster entries removed: 10
1261    3 Zuschlagspartner: Fa. Logica; Stuttgart; Fa....
7651    (1) Bechtle GmbH & Co. KG in Bonn; (2) CompuSa...
7652    (1) Bechtle GmbH & Co. KG in Bonn; (2) CompuSa...
7653    (1) Bechtle GmbH & Co. KG in Bonn; (2) CompuSa...
7654    (1) CompuSafe Data System AG in München; (2) C...
9708    (1) 7P Solutions & Consulting AG in Köln; (2) ...
9709    (1) 4Soft GmbH in München; (2) 7P Solutions & ...
9710    (1) 7P Solutions & Consulting AG in Köln; (2) ...
9711    (1) Accenture GmbH in Kronberg im Taunus; (2) ...
9712    (1) 42Virtual Business Services GmbH in Wien; ...

Remaining cases: 48295


## Entity Resolution - Supplier side
Entity Resolution followed a four step procedure:


**I. Normalisation.** Names are reduced to their distinctive core: lowercasing, transliteration of umlauts and diacritics, removal of legal forms and sector-typical generic tokens (*Solutions*, *Systems*, *IT*, *Deutschland*). Such tokens appear across nearly all suppliers and would dominate similarity scores while carrying no discriminating information. Sorting the remaining tokens alphabetically additionally renders word order irrelevant.

**II. Blocking.** An all-pairs comparison would scale quadratically and is therefore infeasible. Using character-n-gram TF-IDF vectors and a FAISS nearest-neighbour search, only the ten most plausible candidates are retained per name. This step does not decide identity; it reduces the search space.

**III. Scoring.** The actual, more expensive string similarity is computed only within these candidate pairs. Pairs above a fixed threshold are treated as matches. The threshold governs the trade-off between erroneously merging distinct firms and failing to capture spelling variants.

**IV. Clustering.** Matches are treated as transitive: the pairs form a graph whose connected components each represent one real-world actor. The canonical name is the most frequent raw spelling within a component — the empirically most common variant rather than a constructed normal form.


In [18]:
# step 1: preprocessing
LEGAL_FORMS = r'\b(gmbh|ag|kg|ohg|gbr|mbh|co|ug|se|ev|e\.v\.|ltd|inc|bv|nv|sarl)\b'

# using stopwords which proved to be misleading because of generic tokens which dominate
# similarity-scores
IT_STOPWORDS = {
    # domain specific
    'solutions', 'solution', 'services', 'service', 'systems', 'system',
    'consulting', 'unternehmensberatung', 'beratung', 'management',
    'technology', 'technologies', 'tech', 'digital', 'software',
    'it', 'ict', 'data', 'net', 'network', 'networks', 'informatik',
    'information', 'informationstechnologie', 'informationssysteme',
    'computer', 'datentechnik', 'computersysteme', 'systemhaus',
    'wirtschaftspruefungsgesellschaft'
    # geographic
    'deutschland', 'germany', 'europe', 'europa', 'international',
    # generic
    'gruppe', 'group', 'holding', 'partner', 'partners',
}

def normalize_name_v2(name):
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    name = name.replace('ä','ae').replace('ö','oe').replace('ü','ue').replace('ß','ss')
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode()
    name = re.sub(LEGAL_FORMS, '', name)
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    tokens = [t for t in name.split() if t not in IT_STOPWORDS and len(t) > 1]
    return ' '.join(tokens).strip()


data_clean_['WIN_NAME_NORM'] = data_clean_['WIN_NAME'].apply(normalize_name_v2)

data_clean_['WIN_NAME_NORM_SORTED'] = data_clean_['WIN_NAME_NORM'].apply(
    lambda x: ' '.join(sorted(x.split())) if isinstance(x, str) else x
)
unique_names_sorted = data_clean_['WIN_NAME_NORM_SORTED'].dropna().unique()

In [19]:
# Step II: TF-IDF + FAISS blocking on token-sorted normalized names
unique_names_sorted = data_clean_['WIN_NAME_NORM_SORTED'].dropna().unique()

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X = vec.fit_transform(unique_names_sorted).toarray().astype('float32')

faiss.normalize_L2(X)
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
D, I = index.search(X, k=10)  # Top-10 candidates per name

In [20]:
# Step III: similarity scoring with rapidfuzz
THRESHOLD = 88

pairs = []
for i, (dists, idxs) in enumerate(zip(D, I)):
    for dist, j in zip(dists, idxs):
        if i >= j:
            continue
        score = fuzz.token_sort_ratio(unique_names_sorted[i], unique_names_sorted[j])
        if score >= THRESHOLD:
            pairs.append((unique_names_sorted[i], unique_names_sorted[j], score))

In [21]:
# Step IV: clustering with networkx
G = nx.Graph()
G.add_edges_from([(a, b) for a, b, _ in pairs])

mapping = {}
for component in nx.connected_components(G):
    # Get all rows belonging to this cluster via WIN_NAME_NORM_SORTED
    mask = data_clean_['WIN_NAME_NORM_SORTED'].isin(component)

    # Count by lowercased raw name to collapse case variants
    counts = Counter(
        data_clean_.loc[mask, 'WIN_NAME']
        .str.strip().str.lower()
    )
    canonical_lower = counts.most_common(1)[0][0]

    # Recover representative raw name with original capitalization
    canonical_raw = (
        data_clean_.loc[
            data_clean_['WIN_NAME'].str.strip().str.lower() == canonical_lower,
            'WIN_NAME'
        ].mode()[0]
    )
    for norm_name in component:
        mapping[norm_name] = canonical_raw

# Apply mapping via WIN_NAME_NORM_SORTED
data_clean_['WIN_NAME_CANONICAL'] = (
    data_clean_['WIN_NAME_NORM_SORTED']
    .map(mapping)
    .fillna(data_clean_['WIN_NAME'].str.strip().str.lower())
)

print(f"Unique canonical names: {data_clean_['WIN_NAME_CANONICAL'].nunique()}")

Unique canonical names: 11095


### False positive check

In [22]:
# remove the "#" of the following two lines to inspect clusters
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_rows', None)


# check of the clusters
validation_df = (
    data_clean_[['WIN_NAME', 'WIN_NAME_NORM', 'WIN_NAME_CANONICAL']]
    .drop_duplicates()
    .groupby('WIN_NAME_CANONICAL')
    .agg(
        variations=('WIN_NAME', lambda x: list(x.unique())),
        number_of_variations=('WIN_NAME', 'nunique')
    )
    .reset_index()
    .query('number_of_variations > 1')          
    .sort_values('number_of_variations', ascending=False)
)

print(f"Cluster with >1 variations: {len(validation_df)}")
print(f"Of which >=5 variations:  {(validation_df['number_of_variations'] >= 5).sum()}")

# manually checking all clusters with 5 or more variations
validation_df.query('number_of_variations >= 5').sort_values('number_of_variations', ascending=False)

Cluster with >1 variations: 1629
Of which >=5 variations:  186


,WIN_NAME_CANONICAL,variations,number_of_variations
199,Computacenter AG & Co. oHG,"[Computacenter AG & Co. oHG, Computacenter, Co...",49
553,SAP Deutschland SE & Co. KG,"[SAP Deutschland AG & Co. KG, SAP Deutschland ...",28
496,Oracle Deutschland B.V. & Co. KG,"[Oracle Deutschland GmbH, Oracle Deutschland G...",26
185,Canon Deutschland GmbH,"[Canon Deutschland GmbH, CSC Deutschland Solut...",23
424,Konica Minolta Business Solutions Deutschland ...,[Konica Minolta Business Solutions Deutschland...,20
...,...,...,...
240,DiGiTEXX Gesellschaft für digitale Bürosysteme...,[Firma Digitexx Gesellschaft für digitale Büro...,5
573,Senacor FCS GmbH,"[Senacor FSG, Senacor ISG GmbH, Senacor FSG Gm...",5
331,H&G Hansen & Gieraths EDV Vertriebsgesellschaf...,[H&G Hansen & Gieraths EDV Vertriebsgesellscha...,5
183,Cancom Deutschland GmbH,"[Cancom Deutschland GmbH, CANCOM Deutschland G...",5


In [23]:
# manual fixes for all clusters containing 5 or more variations
manual_fixes = {
    # Canon cluster: remove false positives
    # CSC, HP, CA, CalCon, HMM, Calcon, Software AG are all different companies
    'CSC Deutschland Solutions GmbH': 'CSC Deutschland Solutions GmbH',
    'CSC Deutschland GmbH': 'CSC Deutschland GmbH',
    'HP Deutschland GmbH': 'HP Deutschland GmbH',
    'CA Deutschland GmbH': 'CA Deutschland GmbH',
    'Calcon Deutschland GmbH': 'Calcon Deutschland GmbH',
    'HMM Deutschland GmbH': 'HMM Deutschland GmbH',
    'CalCon Deutschland GmbH': 'CalCon Deutschland GmbH',
    'CalCon Deutschland AG': 'CalCon Deutschland AG',
    'csc deutschland solutions gmbh': 'CSC Deutschland Solutions GmbH',
    'CSC Deutschland Solutions Gmbh': 'CSC Deutschland Solutions GmbH',
    'T-Systems Deutschland GmbH': 'T-Systems Deutschland GmbH',
    'CSC Deutschland GmBH': 'CSC Deutschland GmbH',
    'CA-Deutschland GmbH': 'CA Deutschland GmbH',
    'CSC Deutschland': 'CSC Deutschland GmbH',
    'HP Deutschland': 'HP Deutschland GmbH',
    'Software AG Deutschland GmbH': 'Software AG Deutschland GmbH',
    'Cale Deutschland GmbH': 'Cale Deutschland GmbH',


    # --- Atos Information Technology: remove Athos/ATOSS ---
    'Athos Unternehmensberatung GmbH': 'Athos Unternehmensberatung GmbH',
    'ATHOS Unternehmensberatung GmbH': 'Athos Unternehmensberatung GmbH',
    'ATHOS GmbH': 'Athos Unternehmensberatung GmbH',
    'ATOSS Software AG': 'ATOSS Software AG',
    'ATOSS Software AG ': 'ATOSS Software AG',
    'Atos IT Solutions a n d Services GmbH': 'Atos Information Technology GmbH',
    'ATOS IT Solutions GmbH': 'Atos Information Technology GmbH',
    'Atos IT Solutions & Services GmbH': 'Atos Information Technology GmbH',
    'Atos IT Solutions Service GmbH': 'Atos Information Technology GmbH',
    'AToS GmbH': 'Atos Information Technology GmbH',
    'ATOS GmbH': 'Atos Information Technology GmbH',
    'Atos GmbH': 'Atos Information Technology GmbH',
    'Atos': 'Atos Information Technology GmbH',
    'Atos Information Technology Gmbh': 'Atos Information Technology GmbH',
    'ATOS Information Technology GmbH': 'Atos Information Technology GmbH',
    'Atos information Technology GmbH': 'Atos Information Technology GmbH',
    'ATOS IT Solutions&Service GmbH': 'Atos Information Technology GmbH',
    'Atos IT Information Technology GmbH': 'Atos Information Technology GmbH',
    'Atos Holding Germany 1 GmbH': 'Atos Information Technology GmbH',

    # --- NEC: keep only NEC variants ---
    'Infotec Deutschland GmbH': 'Infotec Deutschland GmbH',
    'Fa. Infotec Deutschland GmbH': 'Infotec Deutschland GmbH',
    'Ocè Deutschland GmbH': 'Océ Deutschland GmbH',
    'Oce-Deutschland GmbH': 'Océ Deutschland GmbH',
    'Océ-Deutschland GmbH': 'Océ Deutschland GmbH',
    'OCE Deutschland GmbH': 'Océ Deutschland GmbH',
    'OCÉ Deutschland GmbH': 'Océ Deutschland GmbH',
    'OCE-Deutschland GmbH': 'Océ Deutschland GmbH',
    'Oce Deutschland GmbH': 'Océ Deutschland GmbH',
    'EMC Deutschland GmbH': 'EMC Deutschland GmbH',
    'Infoprint Solutions Deutschland GmbH': 'Infoprint Solutions Deutschland GmbH',
    'Infor (Deutschland) GmbH': 'Infor GmbH',
    'Infor GmbH': 'Infor GmbH',
    'NEc Deutschland GmbH': 'NEC Deutschland GmbH',
    'Software One Deutschland GmbH': 'SoftwareONE Deutschland GmbH',

    # --- MR Datentechnik: remove false positives ---
    'KMS Vertrieb und Services AG': 'KMS Vertrieb und Service AG',
    'KMS Vertrieb und Service AG': 'KMS Vertrieb und Service AG',
    'iS Software Vertrieb und Service GmbH': 'iS Software Vertrieb und Service GmbH',
    'IS Software Vertrieb und Service GmbH': 'iS Software Vertrieb und Service GmbH',
    'K&P Computer Service- und Vertriebs-GmbH': 'K&P Computer Service- und Vertriebs-GmbH',
    'K&P Computer Service- und Vertriebs- GmbH': 'K&P Computer Service- und Vertriebs-GmbH',
    'MR-Datentechnik; Vertrieb und Service GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR-Datentechnik Vertriebs- und Service GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR Datentechnik Service- und Vertriebs GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR-Datentechnik Vertriebs-und Service GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR Datentechnik Vertriebs und Service GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR Datentechnik Vertriebs-und Service GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR Datentechnik Vertriebs- und Service Gmbh': 'MR Datentechnik Vertriebs- und Service GmbH',
    'R Datentechnik Vertriebs- und Service GmbH': 'MR Datentechnik Vertriebs- und Service GmbH',
    'MR Datentechnik Vertriebs- und Service GmbH ': 'MR Datentechnik Vertriebs- und Service GmbH',

    # NTT DATA cluster: remove false positives
    
    # Riso, TIS, NTS, RIB, Tieto, Ticos, Data Vison are different companies
    'Riso Deutschland GmbH': 'Riso Deutschland GmbH',
    'RISO Deutschland GmbH': 'Riso Deutschland GmbH',
    'RISO (Deutschland) GmbH': 'Riso Deutschland GmbH',
    'TIS Deutschland GmbH': 'TIS Deutschland GmbH',
    'NTS Deutschland GmbH': 'NTS Deutschland GmbH',
    'RIB Deutschland GmbH': 'RIB Deutschland GmbH',
    'Tieto Deutschland GmbH': 'Tieto Deutschland GmbH',
    'Ticos Systems Deutschland GmbH': 'Ticos Systems Deutschland GmbH',
    'Data Vison Deutschland GmbH': 'Data Vison Deutschland GmbH',

    # EntServ is HP Enterprise Services → legitimate NTT predecessor, keep
    'EntServ Deutschland GmbH': 'NTT DATA Deutschland GmbH',
    'EntServ DEutschland GmbH': 'NTT DATA Deutschland GmbH',
    'EntServ-Deutschland GmbH (HP)': 'NTT DATA Deutschland GmbH',
    'entserv deutschland gmbh': 'NTT DATA Deutschland GmbH',

    # --- KPMG: split into individual firms ---
    'BDO AG Wirtschaftsprüfungsgesellschaft': 'BDO AG Wirtschaftsprüfungsgesellschaft',
    'KPMG AG Wirtschaftsprügungsgesellschaft': 'KPMG AG Wirtschaftsprüfungsgesellschaft',
    'Baker Tilly GmbH & Co. KG Wirtschaftsprüfungsgesellschaft': 'Baker Tilly Wirtschaftsprüfungsgesellschaft GmbH',
    'Baker Tilly Wirtschaftsprüfungsgesellschaft GmbH': 'Baker Tilly Wirtschaftsprüfungsgesellschaft GmbH',
    'Deloitte GmbH Wirtschaftsprüfungsgesellschaft': 'Deloitte GmbH Wirtschaftsprüfungsgesellschaft',
    'AIOS GmbH Wirtschaftsprüfungsgesellschaft': 'AIOS GmbH Wirtschaftsprüfungsgesellschaft',
    'KPMG AG Wirtschaftprüfungsgesellschaft': 'KPMG AG Wirtschaftsprüfungsgesellschaft',
    'EY GmbH Wirtschaftsprüfungsgesellschaft': 'EY GmbH Wirtschaftsprüfungsgesellschaft',
    'ETL AG Wirtschaftsprüfungsgesellschaft': 'ETL AG Wirtschaftsprüfungsgesellschaft',
    'Mazars GmbH & Co KG Wirtschaftsprüfungsgesellschaft': 'Mazars GmbH & Co. KG Wirtschaftsprüfungsgesellschaft',

    # --- Atos IT Solutions and Services: keep together, remove uncertain case ---
    'ATOS IT Solutions and Services GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos IT-Solutions and Services GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos IT Solutions an d Services GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos IT Solutions ans Services GmbH': 'Atos IT Solutions and Services GmbH',
    'ATOS IT-Solutions and Services GmbH': 'Atos IT Solutions and Services GmbH',
    'ATOS IT Solution and Services GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos IT Solutions and Service GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos IT Solutions an Services GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos IT Solution and Services GmbH': 'Atos IT Solutions and Services GmbH',
    'Atos Information and Technology GmbH': 'Atos Information and Technology GmbH',  # kept separate

    # --- Vision Consulting: remove false positives ---
    'it vision GmbH': 'it vision GmbH',
    'Avision GmbH': 'Avision GmbH',
    'VISION Consulting Group GmbH & Co. KG': 'Vision Consulting GmbH & Co. KG',
    'Vision Consulting Group GmbH + Co.KG': 'Vision Consulting GmbH & Co. KG',
    'Vision Consulting Group GmbH & Co. KG': 'Vision Consulting GmbH & Co. KG',
    'Vision Consulting Group GmbH & Co.KG': 'Vision Consulting GmbH & Co. KG',
    'Vision Consulting GmbH & Co.KG': 'Vision Consulting GmbH & Co. KG',
    'Vision Consulting Gmbh & Co.KG': 'Vision Consulting GmbH & Co. KG',
    'Vision Consulting GmbH': 'Vision Consulting GmbH & Co. KG',

    # --- CONET: remove econet ---
    'econet GmbH': 'econet GmbH',
    'Conet Solutions GmbH': 'CONET Solutions GmbH',
    'Conet': 'CONET Solutions GmbH',
    'Conet Solutions': 'CONET Solutions GmbH',
    'CONET Solutions': 'CONET Solutions GmbH',
    'CONET GmbH': 'CONET Solutions GmbH',
    'CONET Services GmbH': 'CONET Solutions GmbH',
    'Conet Services GmbH': 'CONET Solutions GmbH',

    # --- Saveco/Savecom: split ---
    'Savecom datentechnik e. K': 'Savecom Datentechnik e.K.',
    'Savecom datentechnik e. K.': 'Savecom Datentechnik e.K.',
    'Savecom datentechnik e.K.': 'Savecom Datentechnik e.K.',
    'Savecom Datentechnik e. K.': 'Savecom Datentechnik e.K.',
    'Savecon Datentechnik e. K.': 'Savecom Datentechnik e.K.',
    'Savecom datentechnik e. K. ': 'Savecom Datentechnik e.K.',
    'Savecom Datentechnik': 'Savecom Datentechnik e.K.',

    # --- T.S. Team-Software: remove false positives ---
    'Data-Team GmbH': 'Data-Team GmbH',
    'Data-team GmbH': 'Data-Team GmbH',
    'DATA team GmbH': 'Data-Team GmbH',
    'NTeam GmbH': 'NTeam GmbH',
    'nteam GmbH': 'NTeam GmbH',
    'Team Technology GmbH': 'Team Technology GmbH',
    'T.S. Team-Software GmbH': 'T. S. Team-Software GmbH',

    # --- Esri: remove SER Solutions ---
    'SER Solutions Deutschland GmbH': 'SER Solutions Deutschland GmbH',
    'ESRI Deutschland GmbH': 'Esri Deutschland GmbH',
    'Esri Deutschland': 'Esri Deutschland GmbH',
    'Esri Detschland GmbH': 'Esri Deutschland GmbH',
    'Esri Deuschland GmbH': 'Esri Deutschland GmbH',
    'ESRi Deutschland GmbH': 'Esri Deutschland GmbH',
    'Esri Deutschland GmbH ': 'Esri Deutschland GmbH',

    # --- Dimension Data: remove dimensional ---
    'dimensional GmbH': 'dimensional GmbH',
    'DIMENSION DATA GERMANY AG & Co.KG': 'Dimension Data Germany AG & Co. KG',
    'Dimension Data Germany AG & Co KG': 'Dimension Data Germany AG & Co. KG',
    'DIMENSION DATA Germany AG & Co.KG': 'Dimension Data Germany AG & Co. KG',
    'Dimension Data Germany AG & Co.KG': 'Dimension Data Germany AG & Co. KG',
    'Dimension-Data': 'Dimension Data Germany AG & Co. KG',
    'Dimension Data Germany AG&Co. KG': 'Dimension Data Germany AG & Co. KG',

    # --- Infora: keep only Infora, separate Infor ---
    'info-tech solutions GmbH & Co. KG': 'info-tech solutions GmbH & Co. KG',
    'Info Tech GmbH': 'Info Tech GmbH',
    'Info GmbH': 'Info GmbH',
    'INFORA GmbH': 'Infora GmbH',
    'Înfora GmbH': 'Infora GmbH',
    'INFORA': 'Infora GmbH',

    # --- UNIT4: remove false positives ---
    'Business IT Nord GmbH': 'Business IT Nord GmbH',
    'L und M Business IT': 'L und M Business IT GmbH',
    'L und M Business IT GmbH': 'L und M Business IT GmbH',
    'Unit 4 Business Software GmbH': 'UNIT4 Business Software GmbH',
    'UNIT4 Business GmbH': 'UNIT4 Business Software GmbH',
    'Unit4 Business Software GmbH': 'UNIT4 Business Software GmbH',

    # --- Q-SOFT: remove false positives ---
    '4Soft GmbH': '4Soft GmbH',
    'iSOFT': 'iSOFT',
    'j&soft GmbH': 'j&s-soft GmbH',
    'j&s-soft GmbH': 'j&s-soft GmbH',
    'J&S-soft GmbH': 'j&s-soft GmbH',
    'Q-Soft GmbH': 'Q-SOFT GmbH',

    # --- IT Media Consult: remove Media Consulta ---
    'Media Consulta GmbH': 'Media Consulta GmbH',
    'media consulta International Holding AG': 'Media Consulta GmbH',
    'ITMedia Consult AG': 'IT Media Consult AG',
    'IT-Media Consult AG': 'IT Media Consult AG',
    'IT Media Consult': 'IT Media Consult AG',
    'ITMedia Consult': 'IT Media Consult AG',

    # --- BGS: remove false positives ---
    'Management Beratungs Gesellschaft mbH': 'Management Beratungs Gesellschaft mbH',
    'Computer-Beratungs-Gesellschaft mbH': 'Computer-Beratungs-Gesellschaft mbH',
    'HSP Beratungsgesellschaft mbH': 'HSP Beratungsgesellschaft mbH',
    'BGS Beratungsgesellschaft': 'BGS Beratungsgesellschaft Software Systemplanung AG',
    'Fa. BGS Beratungsgesellschaft Software Systemplanung AG': 'BGS Beratungsgesellschaft Software Systemplanung AG',
    'BGS Beratungsgesellschaft Software Syste': 'BGS Beratungsgesellschaft Software Systemplanung AG',

    # --- i.syde: remove TÜV ---
    'TÜV Informationstechnik GmbH': 'TÜV Informationstechnik GmbH',
    'TÜV Informationstechnik': 'TÜV Informationstechnik GmbH',
    'ITS Informationstechnik Service GmbH': 'ITS Informationstechnik Service GmbH',
    'i.Syde Informationstechnik GmbH': 'i.syde Informationstechnik GmbH',
    'i.syde Informationstechnik mbH': 'i.syde Informationstechnik GmbH',
    'i.syde Informationstechnik': 'i.syde Informationstechnik GmbH',

    # --- OBS Büro-Centrum: remove Wolbers ---
    'Wolbers GmbH Bürocentrum': 'Wolbers GmbH Bürocentrum',
    'Bürocentrum Wolbers GmbH': 'Wolbers GmbH Bürocentrum',
    'Bürozentrum Wolbers GmbH': 'Wolbers GmbH Bürocentrum',
    'OBS Büro Centrum GmbH': 'OBS Büro-Centrum GmbH',
    'OBS Bürocentrum GmbH': 'OBS Büro-Centrum GmbH',

    # --- kommIT: remove false positives ---
    'GIb Gesellschaft für Informationstechnik und -beratung mbH': 'GIb Gesellschaft für Informationstechnik und -beratung mbH',
    'intec Gesellschaft für Informationstechnik mbH': 'intec Gesellschaft für Informationstechnik mbH',
    'kommIT Gesellschaft für Informationstechnik GmbH': 'kommIT Gesellschaft für Informationstechnik mbH',
    'KommIT; Gesellschaft für Informationstechnik': 'kommIT Gesellschaft für Informationstechnik mbH',
    'KommIT Gesellschaft für Informationstechnik mbH': 'kommIT Gesellschaft für Informationstechnik mbH',

    # --- Planon: remove false positives ---
    'PlanB. GmbH': 'PlanB. GmbH',
    'Data-Plan Computer Consulting GmbH': 'Data-Plan Computer Consulting GmbH',
    'DATA-Plan Computer Consulting GmbH': 'Data-Plan Computer Consulting GmbH',
    'EPLAN GMBH & CO KG': 'EPLAN GmbH & Co. KG',
    'plano solutions gmbh': 'plano solutions GmbH',

    # --- bbt Barth: remove Bürotechnik Bartel ---
    'Bürotechnik Bartel': 'Bürotechnik Bartel',
    'Barth Büro Technik GmbH & Co. KG': 'bbt Barth Büro Technik GmbH & Co. KG',
    'bbt Barth Bürotechnik GmbH & Co. KG': 'bbt Barth Büro Technik GmbH & Co. KG',
    'bbt Barth Büro Technik GmbH & Co.KG': 'bbt Barth Büro Technik GmbH & Co. KG',
    'bbt Barth Büro Technik GmbH &Co. KG': 'bbt Barth Büro Technik GmbH & Co. KG',

    # --- Comparex Deutschland: remove Compart ---
    'Compart Deutschland GmbH': 'Compart Deutschland GmbH',
    'COMPAREX Deutschland AG': 'Comparex Deutschland AG',
    'Comparex Deutschland GmbH': 'Comparex Deutschland AG',
    'COMPAREX Deutschland GAG': 'Comparex Deutschland AG',
    'Compaerx Deutschland AG': 'Comparex Deutschland AG',

    # --- EWERK: remove U-Werk ---
    'U-Werk GbR': 'U-Werk GbR',
    'Ewerk IT GmbH': 'EWERK DIGITAL GmbH',
    'EWERK IT GmbH': 'EWERK DIGITAL GmbH',
    'EWERK Digital GmbH': 'EWERK DIGITAL GmbH',
    'EWERK Consulting GmbH': 'EWERK DIGITAL GmbH',

    # --- Engineering ITS: remove false positives ---
    'AL-Engineering AG': 'AL-Engineering AG',
    'Software Engineering GmbH': 'Software Engineering GmbH',
    'ISE Information Systems Engineering GmbH': 'ISE Information Systems Engineering GmbH',
    'iD-Engineering GmbH': 'iD-Engineering GmbH',
    'Engineering ITS GmbH': 'Engineering ITS AG',

    # --- VegaSystems/Pegasystems: split ---
    'Fa. Pegasystems GmbH': 'Pegasystems GmbH',
    'Pegasystems Ltd': 'Pegasystems GmbH',
    'Pegasystems Ltd.': 'Pegasystems GmbH',
    'Pegasystems GmbH': 'Pegasystems GmbH',

    # --- matrix Systems: remove MTRIX ---
    'MTRIX GmbH': 'MTRIX GmbH',
    'matrix technology AG': 'matrix technology AG',
    'matrix technology GmbH': 'matrix technology AG',
    'matrix Systems Consulting GmbH': 'matrix Systems & Consulting GmbH',

    # --- Cerner: remove false positives ---
    'Acer Deutschland GmbH': 'Acer Deutschland GmbH',
    'Acer Deutschland GmbH.': 'Acer Deutschland GmbH',
    'DAC Group Deutschland GmbH': 'DAC Group Deutschland GmbH',
    'Cerner HS Deutschland GmbH': 'Cerner Deutschland GmbH',

    # --- COPRA System: remove Compra ---
    'Compra e. K.': 'Compra e.K.',
    'compra e.K.': 'Compra e.K.',
    'Copra System GmbH': 'COPRA System GmbH',
    'COPRA SYSTEM GMBH': 'COPRA System GmbH',

    # codia cluster: remove Coda (different company – Coda GmbH is ERP vendor)
    'Coda GmbH': 'Coda GmbH',
    'CODA GmbH': 'Coda GmbH',

    # V-BC.de / Vogtland Bürocenter: same owner, merge into one canonical name
    'Vogtland Bürocenter Heiko Protze': 'V-BC.de, Inhaber Heiko Protze',
    'Vogtland-Bürocenter Inh. Heiko Protze': 'V-BC.de, Inhaber Heiko Protze',
    'Vogtland -Bürocenter Heiko Protze': 'V-BC.de, Inhaber Heiko Protze',
    'Vogtland-Bürocenter, Inh. Heiko Protze': 'V-BC.de, Inhaber Heiko Protze',
    'Vogtland Bürocenter Heiko Protze, V-BC.de': 'V-BC.de, Inhaber Heiko Protze',
    'Vogtland-Bürocenter': 'V-BC.de, Inhaber Heiko Protze',
    'Firma Vogtland - Bürocenter': 'V-BC.de, Inhaber Heiko Protze',
    'Vogtland Bürocenter': 'V-BC.de, Inhaber Heiko Protze',

    # Unisys: remove Scisys and Unify (different companies)
    'Scisys Deutschland GmbH': 'Scisys Deutschland GmbH',
    'SCISYS Deutschland GmbH': 'Scisys Deutschland GmbH',
    'Unify Deutschland GmbH & Co. KG': 'Unify Deutschland GmbH & Co. KG',
    'unisys Deutschland GmbH': 'Unisys Deutschland GmbH',

    # INFOMA: remove infoman ag (different company – Infoman AG is unrelated)
    'infoman ag': 'infoman ag',

    # iProCon: remove false positives (RosCon, ProCon, PROCON IT are different companies)
    'RosCon GmbH': 'RosCon GmbH',
    'rocon GmbH': 'rocon GmbH',
    'ProCon GmbH & Co.KG': 'ProCon GmbH & Co.KG',
    'RosCon Unternehmensberatung GmbH': 'RosCon Unternehmensberatung GmbH',
    'PROCON IT GmbH': 'PROCON IT GmbH'

}

data_clean_['WIN_NAME_CANONICAL'] = (
    data_clean_['WIN_NAME']
    .map(manual_fixes)
    .fillna(data_clean_['WIN_NAME_CANONICAL'])
)

hits = data_clean_['WIN_NAME'].map(manual_fixes).notna().sum()
print(f"Adjusted rows: {hits}")

Adjusted rows: 706


In [24]:
# check again
validation_df = (
    data_clean_[['WIN_NAME', 'WIN_NAME_NORM', 'WIN_NAME_CANONICAL']]
    .drop_duplicates()
    .groupby('WIN_NAME_CANONICAL')
    .agg(
        variations=('WIN_NAME', lambda x: list(x.unique())),
        number_of_variations=('WIN_NAME', 'nunique')
    )
    .reset_index()
    .query('number_of_variations > 1')          
    .sort_values('number_of_variations', ascending=False)
)

print(f"Of which >=5 variations:  {(validation_df['number_of_variations'] >= 5).sum()}")
validation_df.query('number_of_variations >= 5').sort_values('number_of_variations', ascending=False)

Of which >=5 variations:  166


,WIN_NAME_CANONICAL,variations,number_of_variations
221,Computacenter AG & Co. oHG,"[Computacenter AG & Co. oHG, Computacenter, Co...",49
614,SAP Deutschland SE & Co. KG,"[SAP Deutschland AG & Co. KG, SAP Deutschland ...",28
549,Oracle Deutschland B.V. & Co. KG,"[Oracle Deutschland GmbH, Oracle Deutschland G...",26
469,Konica Minolta Business Solutions Deutschland ...,[Konica Minolta Business Solutions Deutschland...,20
11004,wittler visuelle einrichtungen GmbH,"[Wittler visuelle Einrichtungen GmbH, Wittler ...",18
...,...,...,...
404,IDS Scheer AG,"[IDS Scheer AG, IDS-Scheer AG, IDS Scheer Cons...",5
254,Datagroup Stuttgart GmbH,"[Datagroup Stuttgart GmbH, DATATGROUP Stuttgar...",5
1097,alphaBit GmbH,"[alphaBit GmbH, Alphabit GmbH, alphabit GmbH, ...",5
40,3M Medica Zweigniederlassung der 3M Deutschlan...,[3M Medica – Zweigniederlassung der 3M Deutsch...,5


In [25]:
# remove cases without actual suppliers (e.g. "keine angabe")
remove_patterns = ['keine angabe', 'wurde am', 'aufgehoben', 'losweise', 'sicherstellung des wettbewerbs']

mask_remove = data_clean_['WIN_NAME_CANONICAL'].str.lower().apply(
    lambda x: any(p in x for p in remove_patterns) if isinstance(x, str) else False
)

print(f"removed cases: {mask_remove.sum()}")
data_clean_ = data_clean_[~mask_remove].copy()
print(f"remaining cases: {len(data_clean_)}")

removed cases: 213
remaining cases: 48082


In [26]:
# check effect 
print(f"Remaining cases: {len(data_clean_)}")
print(f"Unique buyers (CAE): {data_clean_['CAE_NAME'].nunique()}")
print(f"Unique suppliers (WIN_NAME_CANONICAL): {data_clean_['WIN_NAME_CANONICAL'].nunique()}")

Remaining cases: 48082
Unique buyers (CAE): 7603
Unique suppliers (WIN_NAME_CANONICAL): 11157


In [27]:
# random check
validation_df = (
    data_clean_[['WIN_NAME', 'WIN_NAME_NORM', 'WIN_NAME_CANONICAL']]
    .drop_duplicates()
    .groupby('WIN_NAME_CANONICAL')
    .agg(
        variations=('WIN_NAME', lambda x: list(x.unique())),
        number_of_variations=('WIN_NAME', 'nunique')
    )
    .reset_index()
    .query('number_of_variations > 1')          
    .sort_values('number_of_variations', ascending=False)
)
validation_df.sample(30, random_state=42)

,WIN_NAME_CANONICAL,variations,number_of_variations
6640,kühn & weyh software gmbh,"[kühn & weyh Software GmbH, Kühn & Weyh Softwa...",2
265,Deutsche Telekom Security GmbH,"[Deutsche Telekom Securitiy GmbH, Deutsche Tel...",2
7725,nexus ag,"[Nexus AG, NEXUS AG]",2
2553,ces it-systemhaus gmbh,"[CES IT-Systemhaus GmbH, CES IT-SYstemhaus GmbH]",2
186,CODA Computer und Programme GmbH Erfurt,"[Coda Computer & Programme GmbH Erfurt, CODA C...",3
8226,pit-cup gmbh,"[pit-cup GmbH, Pit-cup GmbH]",2
380,Hans Engels GmbH,"[Hans Engels, Hans Engels GmbH, Fa. Hans Engel...",4
10851,webit! gesellschaft für neue medien mbh,"[webit! Gesellschaft für neue Medien mbH, Webi...",2
10575,v-bc.de,"[V-BC.de, v-bc.de, V-BC.de ]",3
6823,lowteq gmbh,"[LOWTeq GmbH, Lowteq GmbH, LOWteq GmbH]",3


In [28]:
# save data 
data_clean_.to_csv("data_checkpoints/data_supplier_er.csv")

In [29]:
data_clean_ = pd.read_csv("data_checkpoints/data_supplier_er.csv")

# check effect 
print(f"Remaining cases: {len(data_clean_)}")
print(f"Unique buyers (CAE): {data_clean_['CAE_NAME'].nunique()}")
print(f"Unique suppliers (WIN_NAME_CANONICAL): {data_clean_['WIN_NAME_CANONICAL'].nunique()}")

Remaining cases: 48082
Unique buyers (CAE): 7603
Unique suppliers (WIN_NAME_CANONICAL): 11157


### False negative check

In [30]:
# --- Spot-check: find suspiciously similar canonical names ---

canonical_names = data_clean_["WIN_NAME_CANONICAL"].unique().tolist()

# Check near-identical pairs (very high similarity) that were NOT merged
suspicious_pairs = []
for i, name_a in enumerate(canonical_names):
    for name_b in canonical_names[i+1:]:
        score = fuzz.token_sort_ratio(name_a.lower(), name_b.lower())
        if score >= 95:  # Near-identical after lowercasing
            suspicious_pairs.append((name_a, name_b, score))

suspicious_df = pd.DataFrame(suspicious_pairs, columns=["name_a", "name_b", "score"])
print(suspicious_df.sort_values("score", ascending=False).to_string(index=False))

                                                                                          name_a                                                                                          name_b  score
                                                                  ahd hellweg data gmbh & co. kg                                                                 ahd  hellweg data gmbh & co. kg 100.00
                                                                                     cancom gmbh                                                                                    cancom  gmbh 100.00
                                                          media markt leipzig höfe am brühl gmbh                                                          media markt gmbh leipzig höfe am brühl 100.00
                                                                          concat ag solutions it                                                                          concat ag it solutions 100.00


In [31]:
# How many of the around 600 pairs involve canonical names with substantial award counts?
threshold_awards = 5  # only care about suppliers that matter for network analysis

canonical_counts = data_clean_.groupby('WIN_NAME_CANONICAL').size()

substantial_pairs = []
for _, row in suspicious_df.iterrows():
    n_a = canonical_counts.get(row['name_a'], 0)
    n_b = canonical_counts.get(row['name_b'], 0)
    if n_a >= threshold_awards and n_b >= threshold_awards:
        substantial_pairs.append((row['name_a'], row['name_b'], row['score'], n_a, n_b))

substantial_df = pd.DataFrame(substantial_pairs, columns=['name_a', 'name_b', 'score', 'n_a', 'n_b'])
print(f"Pairs where both have >= {threshold_awards} awards: {len(substantial_df)}")
print(f"Pairs where both have >= 10 awards: {len(substantial_df[(substantial_df['n_a']>=10) & (substantial_df['n_b']>=10)])}")
print(f"Pairs where both have >= 20 awards: {len(substantial_df[(substantial_df['n_a']>=20) & (substantial_df['n_b']>=20)])}")
print(substantial_df.sort_values(['n_a','n_b'], ascending=False).head(20).to_string(index=False))

Pairs where both have >= 5 awards: 32
Pairs where both have >= 10 awards: 10
Pairs where both have >= 20 awards: 1
                             name_a                              name_b  score  n_a  n_b
  fujitsu technology solutions gmbh  Fujitsu Technologie Solutions GmbH  95.52  612   23
  fujitsu technology solutions gmbh    fujitsu technology solution gmbh  98.46  612   12
  fujitsu technology solutions gmbh  Fujitsu Technology Solutiuons GmbH  98.51  612    6
       t-systems international gmbh       t-systems  international gmbh 100.00  496    8
                        cancom gmbh                        cancom  gmbh 100.00  444   12
              bechtle gmbh & co. kg                bechtle gmbh & co kg  97.56  404   12
              bechtle gmbh & co. kg                 bechtle gmbh co. kg  95.00  404   12
              bechtle gmbh & co. kg               bechtle gmbh + co. kg  95.24  404    6
              acp it solutions gmbh               acp it-solutions gmbh  95.24   80 

In [32]:
canonical_names = data_clean_['WIN_NAME_CANONICAL'].unique().tolist()

# Full pairwise scoring without blocking
pairs = []
for i, name_a in enumerate(canonical_names):
    # process.extract to compare name_a against all others efficiently
    results = process.extract(
        name_a,
        canonical_names[i+1:],
        scorer=fuzz.token_sort_ratio,
        score_cutoff=88
    )
    for name_b, score, _ in results:
        pairs.append((name_a, name_b, score))

print(f"Total pairs above threshold: {len(pairs)}")

# Run full pairwise scoring for diagnostic purposes only
full_pairs_df = pd.DataFrame(pairs, columns=['name_a', 'name_b', 'score'])

# Filter to cases where BOTH names have substantial award counts
canonical_counts = data_clean_.groupby('WIN_NAME_CANONICAL').size()
full_pairs_df['n_a'] = full_pairs_df['name_a'].map(canonical_counts)
full_pairs_df['n_b'] = full_pairs_df['name_b'].map(canonical_counts)

# Only look at pairs that actually matter for network analysis
substantial = full_pairs_df[
    (full_pairs_df['n_a'] >= 5) &
    (full_pairs_df['n_b'] >= 5)
].sort_values('score', ascending=False)

print(f"Substantial unmatched pairs: {len(substantial)}")
print(substantial.to_string(index=False))

Total pairs above threshold: 1938
Substantial unmatched pairs: 92
                                                   name_a                                                      name_b  score  n_a  n_b
                             t-systems international gmbh                               t-systems  international gmbh 100.00  496    8
                                              cancom gmbh                                                cancom  gmbh 100.00  444   12
                      bechtle it-systemhaus gmbh & co. kg                         bechtle gmbh & co. kg it-systemhaus 100.00   21    9
                      bechtle gmbh & co. kg it systemhaus                         bechtle it systemhaus gmbh & co. kg 100.00    5    6
                                             it-haus gmbh                                               it-haus  gmbh 100.00   21   11
                                alpha computer sales gmbh                                 alpha computer   sales gmbh 100.00

In [33]:
# Separate score-100 pairs by name length as safety filter
score_100 = full_pairs_df[full_pairs_df['score'] == 100].copy()
score_100['min_length'] = score_100[['name_a', 'name_b']].apply(
    lambda r: min(len(r['name_a']), len(r['name_b'])), axis=1
)
# Flag short names for manual review
print(score_100[score_100['min_length'] < 20][['name_a', 'name_b', 'n_a', 'n_b']])
print(f"\nLong-name score-100 pairs (safe to auto-merge): {(score_100['min_length'] >= 20).sum()}")

                   name_a               name_b  n_a  n_b
20    bechtle gmbh berlin  bechtle berlin gmbh    1    1
202          it-haus gmbh        it-haus  gmbh   21   11
799      coda gmbh erfurt     coda erfurt gmbh    1    1
965           cancom gmbh         cancom  gmbh  444   12
1059      hsm gmbh co. kg     hsm gmbh  co. kg    2    2
1605       ratiodata gmbh      ratiodata  gmbh    2    1
1853          rednet gmbh         rednet  gmbh   39    1

Long-name score-100 pairs (safe to auto-merge): 31


In [34]:
# --- Second-level manual fixes: false negatives from full pairwise scoring ---

# Pairs to explicitly exclude from merging (false positives identified above)
FALSE_POSITIVE_PAIRS = {
    ('Canon Deutschland GmbH', 'Crayon Deutschland GmbH'),
    ('Canon Deutschland GmbH', 'Cancom Deutschland GmbH'),
    ('KPMG AG Wirtschaftsprüfungsgesellschaft', 'BDO AG Wirtschaftsprüfungsgesellschaft'),
    ('Unisys Deutschland GmbH', 'Scisys Deutschland GmbH'),
    ('versatel deutschland gmbh', '1&1 versatel deutschland gmbh'),
    ('CA Deutschland GmbH', 'CSC Deutschland GmbH'),
    ('adiccon gmbh', 'audicon gmbh'),
    ('volavis gmbh', 'polavis gmbh'),
    ('blu systems gmbh', 'b1 systems gmbh'),
    ('campus computersysteme gmbh', 'atm computersysteme gmbh'),
    ('aura computersysteme gmbh', 'atm computersysteme gmbh'),
    ('aura computersysteme gmbh', 'campus computersysteme gmbh'),
    ('arvato systems gmbh', 'arvato systems s4m gmbh'),
    ('itz informationstechnologie gmbh', 'sps informationstechnologie gmbh'),
    ('becom informationssysteme gmbh', 'bents informationssysteme gmbh'),
    ('disy informationssysteme gmbh', 'bents informationssysteme gmbh'),
    ('vianova technologies gmbh', 'zana technologies gmbh'),
    ('ml public services gmbh', 'iso public services gmbh'),
    ('isec7 gmbh', 'ise gmbh'),
    ('ise gmbh', 'iserv gmbh'),
    ('next gmbh', 'nect gmbh'),
    ('concat ag', 'condat ag'),
    ('udf consulting ag', 'uhb consulting ag'),
    ('web commerce gmbh', 'red commerce gmbh'),
    ('com software gmbh', 'cib software gmbh'),
    ('ccp software gmbh', 'com software gmbh'),
    ('ccp software gmbh', 'cib software gmbh'),
    ('vph gmbh & co. kg', 'pco gmbh & co. kg'),
    ('cronos unternehmensberatung gmbh', 'ft consult unternehmensberatung gmbh'),
    ('imtb consulting gmbh', 'itsm consulting gmbh'),
    ('compuware gmbh', 'campusware gmbh'),
    ('dmi gmbh & co. kg', 'dmt gmbh & co kg'),
    ('bechtle gmbh it-systemhaus weimar', 'bechtle gmbh it-systemhaus ulm'),
    ('Ricoh Deutschland GmbH', 'Riso Deutschland GmbH'),
    ('acp it solutions gmbh', 'cancom it solutions gmbh')
}

# Case-insensitive lookup
FALSE_POSITIVE_PAIRS_LOWER = {
    (a.lower(), b.lower()) for a, b in FALSE_POSITIVE_PAIRS
}

def is_false_positive(a, b):
    return (
        (a.lower(), b.lower()) in FALSE_POSITIVE_PAIRS_LOWER or
        (b.lower(), a.lower()) in FALSE_POSITIVE_PAIRS_LOWER
    )

# Build graph from substantial pairs + score-100 pairs, excluding false positives
G2 = nx.Graph()

# Add score-100 pairs (all safe to merge based on length check above)
for _, row in score_100[score_100['min_length'] >= 20].iterrows():
    if not is_false_positive(row['name_a'], row['name_b']):
        G2.add_edge(row['name_a'], row['name_b'])

# Also add score-100 short pairs that are clearly fine
safe_short_100 = [
    ('cancom gmbh', 'cancom  gmbh'),
    ('it-haus gmbh', 'it-haus  gmbh'),
    ('rednet gmbh', 'rednet  gmbh'),
]
for a, b in safe_short_100:
    G2.add_edge(a, b)

# Add substantial pairs (n >= 5 on both sides), excluding false positives
for _, row in substantial.iterrows():
    if not is_false_positive(row['name_a'], row['name_b']):
        G2.add_edge(row['name_a'], row['name_b'])

# Add the three pairs that were missed due to case mismatch
missing_pairs = [
    ('fujitsu technology solutions gmbh', 'Fujitsu Technologie Solutions GmbH'),
    ('Raid Media Sytems GmbH', 'raid media systems gmbh'),
    ('ProCom Datensysteme GmbH', 'pro-com DATENSYSTEME GmbH'),
]
for a, b in missing_pairs:
    G2.add_edge(a, b)

# Apply second-level clustering
mapping2 = {}
for component in nx.connected_components(G2):
    mask = data_clean_['WIN_NAME_CANONICAL'].isin(component)
    counts = Counter(
        data_clean_.loc[mask, 'WIN_NAME_CANONICAL']
        .str.strip().str.lower()
    )
    canonical_lower = counts.most_common(1)[0][0]
    canonical_raw = (
        data_clean_.loc[
            data_clean_['WIN_NAME_CANONICAL'].str.strip().str.lower() == canonical_lower,
            'WIN_NAME_CANONICAL'
        ].mode()[0]
    )
    for name in component:
        mapping2[name] = canonical_raw

data_clean_['WIN_NAME_CANONICAL'] = (
    data_clean_['WIN_NAME_CANONICAL']
    .map(mapping2)
    .fillna(data_clean_['WIN_NAME_CANONICAL'])
)

print(f"Unique canonical names after second-level fix: {data_clean_['WIN_NAME_CANONICAL'].nunique()}")

Unique canonical names after second-level fix: 11077


In [35]:
# check again
# How many of the 600 pairs involve canonical names with substantial award counts?
threshold_awards = 5  # only care about suppliers that matter for network analysis

canonical_counts = data_clean_.groupby('WIN_NAME_CANONICAL').size()

substantial_pairs = []
for _, row in suspicious_df.iterrows():
    n_a = canonical_counts.get(row['name_a'], 0)
    n_b = canonical_counts.get(row['name_b'], 0)
    if n_a >= threshold_awards and n_b >= threshold_awards:
        substantial_pairs.append((row['name_a'], row['name_b'], row['score'], n_a, n_b))

substantial_df = pd.DataFrame(substantial_pairs, columns=['name_a', 'name_b', 'score', 'n_a', 'n_b'])
print(f"Pairs where both have >= {threshold_awards} awards: {len(substantial_df)}")
print(f"Pairs where both have >= 10 awards: {len(substantial_df[(substantial_df['n_a']>=10) & (substantial_df['n_b']>=10)])}")
print(f"Pairs where both have >= 20 awards: {len(substantial_df[(substantial_df['n_a']>=20) & (substantial_df['n_b']>=20)])}")
print(substantial_df.sort_values(['n_a','n_b'], ascending=False).head(20).to_string(index=False))

Pairs where both have >= 5 awards: 0
Pairs where both have >= 10 awards: 0
Pairs where both have >= 20 awards: 0
Empty DataFrame
Columns: [name_a, name_b, score, n_a, n_b]
Index: []


In [36]:
# check
print(f"Remaining cases: {len(data_clean_)}")
print(f"Unique buyers (CAE): {data_clean_['CAE_NAME'].nunique()}")
print(f"Unique suppliers (WIN_NAME_CANONICAL): {data_clean_['WIN_NAME_CANONICAL'].nunique()}")

Remaining cases: 48082
Unique buyers (CAE): 7603
Unique suppliers (WIN_NAME_CANONICAL): 11077


In [37]:
data_clean_.to_csv("data_checkpoints/data_supplier_er.csv")

In [38]:
data_clean_ = pd.read_csv("data_checkpoints/data_supplier_er.csv")

## Check residual separator artefacts found in WIN_NAME_CANONICAL

In [39]:
# --- Residual separator artefacts in WIN_NAME_CANONICAL: diagnostic only ---
freq = (data_clean_['WIN_NAME_CANONICAL']
        .value_counts()
        .rename_axis('name')
        .reset_index(name='n'))

CONSORT_RE = (r'(bietergemeinschaft|arbeitsgemeinschaft|\barge\b|vertreten durch'
              r'|bestehend aus|bewerberpool|^diverse:|^\d+[\.\)]\s)')
BRANCH_RE  = (r'(zweigniederlassung|niederlassung|\bnl\b|geschäftsstelle|geschäftsbereich'
              r'|standort|it[- ]?systemhaus|vertriebszentrum|competence\s*-?\s*center'
              r'|\boffice\b|büro|\bbsc\b|direct sales|abt\.|filiale)')
ADDRESS_RE = r'(\d{5}|str\.|straße|strasse|\bweg\b|allee|platz|\bd-\d)'
PERSON_RE  = r'(\bherr|\bfrau|z\.\s?hd|zu händen|inh\.|inhaber|dipl\.|\bdr\.|e\.\s?k\.)'

def classify(name: str) -> str:
    low = name.lower()
    if ';' not in name:
        return 'consortium_no_semicolon' if re.search(CONSORT_RE, low) else 'clean'
    if re.search(CONSORT_RE, low):
        return 'consortium_semicolon'
    tail = low.split(';', 1)[1].strip(' ;')
    if tail == '':
        return 'empty_tail'
    if re.search(BRANCH_RE, tail):
        return 'branch'
    if re.search(ADDRESS_RE, tail):
        return 'address'
    if re.search(PERSON_RE, tail):
        return 'person'
    return 'unclear'

freq['bucket'] = freq['name'].map(classify)

summary = (freq.groupby('bucket')
                .agg(distinct_names=('name', 'size'), awards=('n', 'sum'))
                .sort_values('awards', ascending=False))
print(summary.to_string())

for b in ['unclear', 'consortium_semicolon', 'consortium_no_semicolon',
          'person', 'address', 'branch', 'empty_tail']:
    total = (freq['bucket'] == b).sum()
    if total:
        sub = freq.loc[freq['bucket'] == b, ['name', 'n']].head(15)
        print(f"\n--- {b} (showing {len(sub)} of {total}) ---")
        print(sub.to_string(index=False))

                         distinct_names  awards
bucket                                         
clean                             10753   47654
consortium_no_semicolon             187     252
unclear                              50      64
branch                               43      54
empty_tail                           11      18
address                              14      16
consortium_semicolon                 14      15
person                                5       9

--- unclear (showing 15 of 50) ---
                                                                    name  n
                                     sony europe limited; zn deutschland  8
                               xerox vertragspartner; hans-peter-schmitz  3
                               music factory studio sales; marc ebermann  3
                                                  msg systems ag; berlin  2
    hrms consult partnerschaft burghoff u. faßheber; unternehmensberater  2
                            

In [40]:
# --- Step 1 (preview): head-truncation candidates for the mechanical buckets ---
# Read-only. Rule under test: the firm is the segment before the first ";".

MECHANICAL = ['empty_tail', 'branch', 'address', 'person']

cand = freq.loc[freq['bucket'].isin(MECHANICAL)].copy()
cand['head'] = (cand['name'].str.split(';', n=1).str[0]
                            .str.strip()
                            .str.rstrip(' ,-'))

# A ";" inside brackets breaks the head-truncation rule: the head is a fragment,
# not a firm name. Route those to the manual list instead.
cand['head_ok'] = cand['head'].str.count(r'\(') == cand['head'].str.count(r'\)')
manual_residual = cand.loc[~cand['head_ok'], ['bucket', 'name', 'head', 'n']]
assert len(manual_residual) == 1, "bracket-guard hit count changed"
cand = cand.loc[cand['head_ok']].drop(columns='head_ok')

print("--- excluded, handled manually ---")
print(manual_residual.to_string(index=False))

# Existing canonical names (bucket 'clean'), keyed case-insensitively.
# If several canonical spellings share a key, keep the most frequent one.
clean_names = freq.loc[freq['bucket'] == 'clean'].sort_values('n', ascending=False)
existing = (clean_names.assign(key=clean_names['name'].str.lower().str.strip())
                       .drop_duplicates('key')
                       .set_index('key')['name'])

cand['target'] = cand['head'].str.lower().str.strip().map(existing)
cand['merges'] = cand['target'].notna()

print(f"\nCandidates: {len(cand)} distinct names, {cand['n'].sum()} awards")
print(f"  merge into an existing cluster: "
      f"{cand['merges'].sum()} names, {cand.loc[cand['merges'], 'n'].sum()} awards")
print(f"  no existing match (new singleton): "
      f"{(~cand['merges']).sum()} names, {cand.loc[~cand['merges'], 'n'].sum()} awards")

print("\n--- would merge into an existing cluster ---")
print(cand.loc[cand['merges'], ['name', 'head', 'target', 'n']]
          .sort_values('n', ascending=False).to_string(index=False))

print("\n--- no existing match ---")
print(cand.loc[~cand['merges'], ['bucket', 'name', 'head', 'n']]
          .sort_values('n', ascending=False).to_string(index=False))

--- excluded, handled manually ---
 bucket                                                                      name                                                    head  n
address raber + märcker (jetzt konica minolta it solutions gmbh; 70499 stuttgart) raber + märcker (jetzt konica minolta it solutions gmbh  1

Candidates: 72 distinct names, 96 awards
  merge into an existing cluster: 60 names, 81 awards
  no existing match (new singleton): 12 names, 15 awards

--- would merge into an existing cluster ---
                                                                                                 name                                                                        head                                                                      target  n
                                                                 metafinanz-informationssysteme gmbh;                                         metafinanz-informationssysteme gmbh                                         metafinan

In [41]:
# --- Step 1: head truncation for mechanical separator artefacts ---
# Rule: where a semicolon separates the firm from a branch, an address, a contact
# person, or an empty remainder, the firm is the segment before the first semicolon.
# Awards are neither duplicated nor dropped; only WIN_NAME_CANONICAL changes.

MECHANICAL_BUCKETS = ['empty_tail', 'branch', 'address', 'person']

# Head is not a usable firm name -> left for the parent mapping step.
HEAD_TRUNCATION_EXCLUDE = {
    'raber + märcker (jetzt konica minolta it solutions gmbh; 70499 stuttgart)',
}

_freq = (data_clean_['WIN_NAME_CANONICAL']
         .value_counts().rename_axis('name').reset_index(name='n'))
_freq['bucket'] = _freq['name'].map(classify)

cand = _freq.loc[_freq['bucket'].isin(MECHANICAL_BUCKETS)
                 & ~_freq['name'].isin(HEAD_TRUNCATION_EXCLUDE)].copy()
cand['head'] = (cand['name'].str.split(';', n=1).str[0].str.strip().str.rstrip(' ,-'))

_clean = _freq.loc[_freq['bucket'] == 'clean'].sort_values('n', ascending=False)
_existing = (_clean.assign(key=_clean['name'].str.lower().str.strip())
                   .drop_duplicates('key').set_index('key')['name'])

cand['target'] = cand['head'].str.lower().str.strip().map(_existing)
# No existing cluster -> the truncated head becomes the canonical form itself.
cand['new_name'] = cand['target'].fillna(cand['head'])

HEAD_TRUNCATION_MAP = dict(zip(cand['name'], cand['new_name']))

# Reviewed state as of manual inspection; fails loudly if upstream steps change.
assert len(HEAD_TRUNCATION_MAP) == 72, len(HEAD_TRUNCATION_MAP)
assert cand['target'].notna().sum() == 60, cand['target'].notna().sum()
assert cand['n'].sum() == 96, cand['n'].sum()

data_sep_fixed = data_clean_.copy()
data_sep_fixed['WIN_NAME_CANONICAL'] = (
    data_sep_fixed['WIN_NAME_CANONICAL'].replace(HEAD_TRUNCATION_MAP)
)

assert len(data_sep_fixed) == len(data_clean_)
assert not data_sep_fixed['WIN_NAME_CANONICAL'].isin(HEAD_TRUNCATION_MAP).any()

n_changed = (data_clean_['WIN_NAME_CANONICAL'] != data_sep_fixed['WIN_NAME_CANONICAL']).sum()
print(f"Awards with rewritten canonical name: {n_changed}")
print(f"Distinct canonical names: {data_clean_['WIN_NAME_CANONICAL'].nunique()} "
      f"-> {data_sep_fixed['WIN_NAME_CANONICAL'].nunique()}")

Awards with rewritten canonical name: 96
Distinct canonical names: 11077 -> 11017


In [42]:
# --- Step 2 (preview): remaining 'unclear' separator cases ---
# Read-only.

_freq2 = (data_sep_fixed['WIN_NAME_CANONICAL']
          .value_counts().rename_axis('name').reset_index(name='n'))
_freq2['bucket'] = _freq2['name'].map(classify)

unclear = _freq2.loc[_freq2['bucket'] == 'unclear'].sort_values('n', ascending=False)
print(f"unclear: {len(unclear)} names, {unclear['n'].sum()} awards\n")
print(unclear[['name', 'n']].to_string(index=False))

unclear: 50 names, 64 awards

                                                                                                                   name  n
                                                                                    sony europe limited; zn deutschland  8
                                                                              music factory studio sales; marc ebermann  3
                                                                              xerox vertragspartner; hans-peter-schmitz  3
                                                                                                 msg systems ag; berlin  2
                                                   hrms consult partnerschaft burghoff u. faßheber; unternehmensberater  2
                                                                              HRMS Consult; PertG Burghoff und Faßheber  2
                                                                                  dr. jörg bowitz; gutachten 

In [43]:
# --- Step 2: manual resolution of 'unclear' separator cases ---
# Each entry was inspected individually. Three sub-rules apply:
#   head  -> semicolon separates firm from location, division, owner or dept code
#   tail  -> the firm is in the second segment (head is a role or trade name)
#   join  -> the semicolon is punctuation inside a single firm name

UNCLEAR_FIXES = {
    # head: firm + location / branch
    'sony europe limited; zn deutschland': 'sony europe limited',
    'msg systems ag; berlin': 'msg systems ag',
    'bibliotheca rfid library systems gmbh; reutlingen': 'bibliotheca rfid library systems gmbh',
    'müller apparatebau gmbh; kranzberg': 'müller apparatebau gmbh',
    'everis spain s.l.u.; succursale en belgique': 'everis spain s.l.u.',
    'computacenter ag & co. ohg; berlin': 'computacenter ag & co. ohg',
    'fnt gmbh; ellwangen': 'fnt gmbh',
    'materna gmbh; dortmund': 'materna gmbh',
    'westhouse consulting gmbh; garching bei münchen': 'westhouse consulting gmbh',
    'quest software gmbh; köln': 'quest software gmbh',
    'ada das systemhaus gmbh; münster': 'ada das systemhaus gmbh',
    'morgenstern ag; reutlingen': 'morgenstern ag',
    'canon business center münchen west; poing': 'canon business center münchen west',

    # head: firm + division / internal department code
    'atos it solutions and services gmbh; ais ger si sol pht ps': 'atos it solutions and services gmbh',
    'siemens ag; ger h ba sl3': 'siemens ag',
    't-systems international gmbh; public & health': 't-systems international gmbh',
    't-systems international gmbh; it-fachvertrieb süd/südwest': 't-systems international gmbh',
    'canon deutschland gmbh; cbc sales – stuttgart': 'canon deutschland gmbh',
    'sag gmbh; gf cegit': 'sag gmbh',
    'fa. 3m deutschland; bibliothekssysteme': '3m deutschland',
    'pro video handelsgesellschaft mbh berlin; broadcast-und konferenztechnik': 'pro video handelsgesellschaft mbh berlin',
    'werner tantzky gmbh; intelligente dokumenten lösungen': 'werner tantzky gmbh',
    'zimmer & lange; it & busines solutions': 'zimmer & lange',
    'köhle und schenk gbr; opencall callcenter': 'köhle und schenk gbr',
    'berger-eppmann & siebert gbr; createam softwareentwicklung': 'berger-eppmann & siebert gbr',
    'frankfurt school of finance & management ggmbh; center for practical quantitative finance': 'frankfurt school of finance & management ggmbh',
    'institut für angewandte informatik e. v.; an-institut der universität leipzig': 'institut für angewandte informatik e. v.',

    # head: firm + owner / occupation descriptor
    'music factory studio sales; marc ebermann': 'music factory studio sales',
    'enviprot; waldemar siedlok': 'enviprot',
    'naumann büroausstattung komplett; enrico naumann': 'naumann büroausstattung komplett',
    'dr. jörg bowitz; gutachten + analytik': 'dr. jörg bowitz',
    'hrms consult partnerschaft burghoff u. faßheber; unternehmensberater': 'hrms consult partnerschaft burghoff u. faßheber',
    'HRMS Consult; PertG Burghoff und Faßheber': 'hrms consult partnerschaft burghoff u. faßheber',

    # tail: the firm is the second segment
    'xerox vertragspartner; hans-peter-schmitz': 'hans-peter-schmitz',
    'office- und kopiermanagement 2000; schramm & städtler ohg': 'schramm & städtler ohg',

    # join: semicolon is punctuation inside one firm name
    'a.s.t.i. - agentur für computersysteme; -technik und innovation gmbh':
        'a.s.t.i. - agentur für computersysteme, -technik und innovation gmbh',
    'srp gesellschaft für stadt-; u. regionalplanung mb': 'srp gesellschaft für stadt- u. regionalplanung mb',
    'gms management solutions; s.l.': 'gms management solutions s.l.',
    'fraunhofer-institut für kommunikation; informationsverarbeitung und ergonomie fkie':
        'fraunhofer-institut für kommunikation, informationsverarbeitung und ergonomie fkie',
    'ressourcenmangel nickel; minack; neumann gbr': 'ressourcenmangel nickel, minack, neumann gbr',

    # manual: corrupted string (firm name glued to its address)
    'sophos ltdthe pentagon; abingdon science park': 'sophos ltd',
}

_freq2 = (data_sep_fixed['WIN_NAME_CANONICAL']
          .value_counts().rename_axis('name').reset_index(name='n'))
_freq2['bucket'] = _freq2['name'].map(classify)

# All keys must still be present, and all must come from the 'unclear' bucket.
_unclear = set(_freq2.loc[_freq2['bucket'] == 'unclear', 'name'])
assert set(UNCLEAR_FIXES) <= _unclear, sorted(set(UNCLEAR_FIXES) - _unclear)
assert len(UNCLEAR_FIXES) == 41, len(UNCLEAR_FIXES)

# Snap resolved names onto an existing canonical spelling where one exists.
_clean = _freq2.loc[_freq2['bucket'] == 'clean'].sort_values('n', ascending=False)
_existing = (_clean.assign(key=_clean['name'].str.lower().str.strip())
                   .drop_duplicates('key').set_index('key')['name'])
UNCLEAR_MAP = {k: _existing.get(v.lower().strip(), v) for k, v in UNCLEAR_FIXES.items()}

data_unclear_fixed = data_sep_fixed.copy()
data_unclear_fixed['WIN_NAME_CANONICAL'] = (
    data_unclear_fixed['WIN_NAME_CANONICAL'].replace(UNCLEAR_MAP)
)

assert len(data_unclear_fixed) == len(data_sep_fixed)
assert not data_unclear_fixed['WIN_NAME_CANONICAL'].isin(UNCLEAR_MAP).any()

n_changed = (data_sep_fixed['WIN_NAME_CANONICAL'] != data_unclear_fixed['WIN_NAME_CANONICAL']).sum()
print(f"Awards with rewritten canonical name: {n_changed}")
print(f"Distinct canonical names: {data_sep_fixed['WIN_NAME_CANONICAL'].nunique()} "
      f"-> {data_unclear_fixed['WIN_NAME_CANONICAL'].nunique()}")
print(f"Remaining 'unclear': {len(_unclear) - len(UNCLEAR_FIXES)} names")

Awards with rewritten canonical name: 55
Distinct canonical names: 11017 -> 10996
Remaining 'unclear': 9 names


In [44]:
# --- Step 3 (preview): decompose the consortium cases ---
# Read-only.
CONSORT_MARKER = (r'\b(bietergemeinschaft|arbeitsgemeinschaft|arge|bewerberpool)\b'
                  r'|\bvertreten durch\b|\bbestehend aus\b|\bmitglied der\b'
                  r'|\bbevollmächtigter vertreter\b|^\s*diverse:|^\s*\d+[\.\)]\s')
LEGAL_FORM = (r'\b(gmbh|mbh|bmbh|ag|kg|ohg|gbr|se|e\.\s?v\.|ggmbh|ltd|limited'
              r'|s\.l\.|b\.v\.|inc|plc|partg|kgaa)\b')
LIST_SEP = r'(\bund\b|&|/|,|;)'

_freq3 = (data_unclear_fixed['WIN_NAME_CANONICAL']
          .value_counts().rename_axis('name').reset_index(name='n'))
_freq3['bucket'] = _freq3['name'].map(classify)

consort = _freq3.loc[_freq3['bucket'].str.startswith('consortium')
                     | (_freq3['bucket'] == 'unclear')].copy()

def sub_bucket(name: str) -> str:
    low = name.lower()
    # metadata in parentheses, firm named outside
    if re.search(r'\([^)]*(bietergemeinschaft|arge|arbeitsgemeinschaft)[^)]*\)', low):
        return 'member_note'
    stripped = re.sub(CONSORT_MARKER, ' ', low)
    stripped = re.sub(r'\s+', ' ', stripped).strip(' -.,;/')
    n_forms = len(re.findall(LEGAL_FORM, stripped))
    has_sep = bool(re.search(LIST_SEP, stripped))
    if n_forms == 0:
        return 'no_legal_form'
    if n_forms == 1 and not has_sep:
        return 'prefix_single'
    return 'multi_firm'

consort['sub'] = consort['name'].map(sub_bucket)

print(consort.groupby('sub').agg(distinct_names=('name', 'size'), awards=('n', 'sum'))
             .sort_values('awards', ascending=False).to_string())

for s in ['multi_firm', 'no_legal_form', 'prefix_single', 'member_note']:
    total = (consort['sub'] == s).sum()
    if total:
        sub = consort.loc[consort['sub'] == s, ['name', 'n']].sort_values('n', ascending=False).head(20)
        print(f"\n--- {s} (showing {len(sub)} of {total}) ---")
        print(sub.to_string(index=False))

               distinct_names  awards
sub                                  
multi_firm                114     142
no_legal_form              45      54
member_note                37      48
prefix_single              14      32

--- multi_firm (showing 20 of 114) ---
                                                                                                                                                         name  n
                                                                                                                    bietergemeinschaft freikom gbr / bvs gmbh  6
                                                                                                          ARGE IP Systems GmbH und Kalinowski Consulting GmbH  6
                                                                                                               Bietergemeinschaft alphaBit GmbH und SINC GmbH  4
                                                                              bietergeme

In [45]:
# --- Step 3a (preview): consortium entries naming exactly one firm ---
# Read-only. Two-stage rule:
#   1. If a representative is named explicitly -> that firm.
#   2. Otherwise strip consortium markers and parenthetical notes.
LEAD_RE = (r'(?:vertreten durch|bevollmächtigter vertreter der bietergemeinschaft ist'
           r'|konsortialführer)\s*(?:die|der|das|den|herrn|frau)?\s*(.+)$')

# Named a second member -> genuine multi-firm case, handled in step 3b.
STEP3A_EXCLUDE = {
    'Bluecarat AG (in Bietergemeinschaft mit Beta Systems Software AG; Berlin)',
    'complion ag (bietergemeinschaft mit licetus gmbh & co. kg)',
}

def resolve_single_consortium(name: str) -> str:
    m = re.search(LEAD_RE, name, flags=re.I)
    s = m.group(1) if m else name
    # drop parenthetical consortium notes
    s = re.sub(r'\s*\([^)]*(?:bietergemeinschaft|arge|arbeitsgemeinschaft)[^)]*\)', '', s, flags=re.I)
    # drop leading / trailing consortium markers and list numbering
    s = re.sub(r'^\s*(?:firma\s+)?(?:bietergemeinschaft|arbeitsgemeinschaft|arge)\b[\s:,.-]*', '', s, flags=re.I)
    s = re.sub(r'[\s,.-]*\b(?:bietergemeinschaft|arbeitsgemeinschaft|arge)\b\s*$', '', s, flags=re.I)
    s = re.sub(r'^\s*\d+[\.\)]\s*', '', s)
    return re.sub(r'\s+', ' ', s).strip(' -.,;()')

step3a = consort.loc[consort['sub'].isin(['member_note', 'prefix_single'])
                     & ~consort['name'].isin(STEP3A_EXCLUDE)].copy()
step3a['resolved'] = step3a['name'].map(resolve_single_consortium)

_clean3 = _freq3.loc[_freq3['bucket'] == 'clean'].sort_values('n', ascending=False)
_existing3 = (_clean3.assign(key=_clean3['name'].str.lower().str.strip())
                     .drop_duplicates('key').set_index('key')['name'])
step3a['target'] = step3a['resolved'].str.lower().str.strip().map(_existing3)
step3a['merges'] = step3a['target'].notna()

print(f"Step 3a candidates: {len(step3a)} names, {step3a['n'].sum()} awards")
print(f"  merge into existing cluster: {step3a['merges'].sum()}")
print(f"  new singleton:               {(~step3a['merges']).sum()}\n")
print(step3a[['sub', 'name', 'resolved', 'target', 'n']]
      .sort_values(['sub', 'n'], ascending=[True, False]).to_string(index=False))

Step 3a candidates: 49 names, 76 awards
  merge into existing cluster: 19
  new singleton:               30

          sub                                                                                                                                                          name                                                                                                   resolved                                                      target  n
  member_note                                                                                                             Oliver Wyman AG (Mitglied der Bietergemeinschaft)                                                                                            Oliver Wyman AG                                                         NaN  6
  member_note                                                                                         Oliver Wyman GmbH (Bevollmächtigter Vertreter der Bietergemeinschaft)                                    

In [46]:
# --- Step 3a: consortium entries naming exactly one firm ---
# Two-stage rule: if a representative is named explicitly, that firm counts;
# otherwise consortium markers and parenthetical notes are stripped.
LEAD_RE = (r'(?:vertreten durch|bevollmächtigte[rn]? vertreter(?:in)? ist'
           r'|konsortialführer)\s*(?:die|der|das|den|herrn|frau)?\s*(.+)$')

STEP3A_EXCLUDE = {   # name a second member -> handled by the blanket rule
    'Bluecarat AG (in Bietergemeinschaft mit Beta Systems Software AG; Berlin)',
    'complion ag (bietergemeinschaft mit licetus gmbh & co. kg)',
}

def resolve_single_consortium(name: str) -> str:
    m = re.search(LEAD_RE, name, flags=re.I)
    s = m.group(1) if m else name
    s = re.sub(r'\s*\([^)]*(?:bietergemeinschaft|arge|arbeitsgemeinschaft)[^)]*\)', '', s, flags=re.I)
    s = re.sub(r'^\s*(?:die|der|das|den)?\s*bevollmächtigte[rn]?\s+', '', s, flags=re.I)
    s = re.sub(r'^\s*(?:firma\s+)?(?:bietergemeinschaft|arbeitsgemeinschaft|arge)\b[\s:,.-]*', '', s, flags=re.I)
    s = re.sub(r'[\s,.-]*\b(?:bietergemeinschaft|arbeitsgemeinschaft|arge)\b\s*$', '', s, flags=re.I)
    s = re.sub(r'^\s*\d+[\.\)]\s*', '', s)
    return re.sub(r'\s+', ' ', s).strip(' -.,;()')

_f = (data_unclear_fixed['WIN_NAME_CANONICAL']
      .value_counts().rename_axis('name').reset_index(name='n'))
_f['bucket'] = _f['name'].map(classify)
_c = _f.loc[_f['bucket'].str.startswith('consortium') | (_f['bucket'] == 'unclear')].copy()
_c['sub'] = _c['name'].map(sub_bucket)

step3a = _c.loc[_c['sub'].isin(['member_note', 'prefix_single'])
                & ~_c['name'].isin(STEP3A_EXCLUDE)].copy()
step3a['resolved'] = step3a['name'].map(resolve_single_consortium)

_cl = _f.loc[_f['bucket'] == 'clean'].sort_values('n', ascending=False)
_ex = (_cl.assign(key=_cl['name'].str.lower().str.strip())
          .drop_duplicates('key').set_index('key')['name'])
step3a['new_name'] = (step3a['resolved'].str.lower().str.strip().map(_ex)
                      .fillna(step3a['resolved']))

# Reviewed state; fails loudly if any upstream step changes.
assert len(step3a) == 49, len(step3a)
assert step3a['n'].sum() == 76, step3a['n'].sum()
assert (step3a['resolved'].str.strip() != '').all()

STEP3A_MAP = dict(zip(step3a['name'], step3a['new_name']))

data_3a_fixed = data_unclear_fixed.copy()
data_3a_fixed['WIN_NAME_CANONICAL'] = (
    data_3a_fixed['WIN_NAME_CANONICAL'].replace(STEP3A_MAP)
)

assert len(data_3a_fixed) == len(data_unclear_fixed)
assert not data_3a_fixed['WIN_NAME_CANONICAL'].isin(STEP3A_MAP).any()

n_changed = (data_unclear_fixed['WIN_NAME_CANONICAL'] != data_3a_fixed['WIN_NAME_CANONICAL']).sum()
print(f"Awards with rewritten canonical name: {n_changed}")
print(f"Distinct canonical names: {data_unclear_fixed['WIN_NAME_CANONICAL'].nunique()} "
      f"-> {data_3a_fixed['WIN_NAME_CANONICAL'].nunique()}")
print(f"Snapped onto an existing cluster: {step3a['new_name'].isin(_ex.values).sum()}")

Awards with rewritten canonical name: 76
Distinct canonical names: 10996 -> 10976
Snapped onto an existing cluster: 20


In [47]:
# --- Canonical names consisting only of dashes / punctuation: read-only ---
_names = (data_3a_fixed['WIN_NAME_CANONICAL']
          .value_counts().rename_axis('name').reset_index(name='n'))

DASH_ONLY = r'^[\s\-–—]+$'
hit = _names.loc[_names['name'].str.match(DASH_ONLY)]

print(f"dash-only names: {len(hit)} names, {hit['n'].sum()} rows")
if len(hit):
    print(hit[['name', 'n']].to_string(index=False))

dash-only names: 2 names, 4 rows
name  n
   -  2
  --  2


In [48]:
# --- Separator cleanup: final state ---
# Consortium entries naming several members are left unresolved (see limitation).
# Seven rows do not name a supplier: one records a cancelled procedure in the
# supplier field instead of the structured flags, two carry the placeholder
# "entfällt", and four consist of a separator character only, left behind when
# the surrounding text was removed. All slipped past the upstream filters, which
# check the structured non-award flags and WIN_NAME for emptiness. They are
# dropped here, at the first point in the pipeline where the supplier field
# itself is inspected.
NOT_A_SUPPLIER_NAME = {
    'kein auftrag; da teilaufhebung der vergabe',   # cancelled procedure
    'entfällt',                                     # placeholder, item nonresponse
}
DASH_ONLY = r'^[\s\-–—]+$'                          # separator remnant, no name left

_drop = (data_3a_fixed['WIN_NAME_CANONICAL'].isin(NOT_A_SUPPLIER_NAME)
         | data_3a_fixed['WIN_NAME_CANONICAL'].str.match(DASH_ONLY))
assert _drop.sum() == 7, _drop.sum()
data_sep_final = data_3a_fixed.loc[~_drop].copy()

assert len(data_sep_final) == 48075, len(data_sep_final)
assert data_sep_final['WIN_NAME_CANONICAL'].nunique() == 10972
assert data_sep_final['WIN_NAME_CANONICAL'].notna().all()
assert not data_sep_final['WIN_NAME_CANONICAL'].str.match(DASH_ONLY).any()

In [49]:
# --- Separator cleanup: documentation summary ---
# Read-only. Recomputes every figure from the objects above; nothing is hard-coded
# except the frozen totals in the asserts.

def _rows(mapping, frame):
    """Supplier-award pairs affected by a name mapping, measured on `frame`."""
    return frame['WIN_NAME_CANONICAL'].isin(mapping).sum()

resolved = pd.DataFrame([
    ('1  head truncation (mechanical)', len(HEAD_TRUNCATION_MAP),
     _rows(HEAD_TRUNCATION_MAP, data_clean_)),
    ('2  manual resolution (ambiguous)', len(UNCLEAR_MAP),
     _rows(UNCLEAR_MAP, data_sep_fixed)),
    ('3a consortium, single firm named', len(STEP3A_MAP),
     _rows(STEP3A_MAP, data_unclear_fixed)),
], columns=['step', 'names', 'rows'])

_final = (data_sep_final['WIN_NAME_CANONICAL']
          .value_counts().rename_axis('name').reset_index(name='n'))
_final['bucket'] = _final['name'].map(classify)

# Consortium entries naming several members: not decomposed (see limitation).
_left_consortium = _final.loc[_final['bucket'].str.startswith('consortium')
                              | (_final['bucket'] == 'unclear')]

# Names carrying a semicolon that no rule addressed.
_left_semicolon = _final.loc[_final['name'].str.contains(';')
                             & ~_final.index.isin(_left_consortium.index)]

left = pd.DataFrame([
    ('consortium, several members named', len(_left_consortium), _left_consortium['n'].sum()),
    ('other residual semicolon names',    len(_left_semicolon),  _left_semicolon['n'].sum()),
], columns=['case', 'names', 'rows'])

n_rows   = len(data_sep_final)
n_before = len(data_clean_)

print("RESOLVED")
print(resolved.to_string(index=False))
print(f"{'total':<34}{resolved['names'].sum():>7}{resolved['rows'].sum():>7}"
      f"   ({resolved['rows'].sum() / n_rows:.2%} of supplier-award pairs)")

print("\nLEFT UNCHANGED")
print(left.to_string(index=False))

print(f"\nDROPPED (no supplier named): {n_before - n_rows} rows")

_names_before = data_clean_['WIN_NAME_CANONICAL'].nunique()
_names_after  = data_sep_final['WIN_NAME_CANONICAL'].nunique()

print(f"\nEFFECT   distinct canonical names: {_names_before} -> {_names_after} "
      f"({_names_after - _names_before:+d})")
print(f"CONTROL  rows: {n_before} -> {n_rows} "
      f"({n_before - n_rows} dropped, none added - names are rewritten, not merged away)")

# Frozen state; fails if any step above changes.
assert resolved['names'].sum() == 162, resolved['names'].sum()
assert resolved['rows'].sum() == 227, resolved['rows'].sum()
assert n_before - n_rows == 7, n_before - n_rows

RESOLVED
                            step  names  rows
 1  head truncation (mechanical)     72    96
2  manual resolution (ambiguous)     41    55
3a consortium, single firm named     49    76
total                                 162    227   (0.47% of supplier-award pairs)

LEFT UNCHANGED
                             case  names  rows
consortium, several members named    160   199
   other residual semicolon names      1     1

DROPPED (no supplier named): 7 rows

EFFECT   distinct canonical names: 11077 -> 10972 (-105)
CONTROL  rows: 48082 -> 48075 (7 dropped, none added - names are rewritten, not merged away)


In [50]:
data_sep_final.to_csv("data_checkpoints/data_supplier_er_two.csv")

In [51]:
data_clean_ = pd.read_csv("data_checkpoints/data_supplier_er_two.csv")

## Parent Mapping - Supplier Side

In [52]:
# parent-mapping: most relevant suppliers, ranked by award count
# VALUE_EURO is not usable (implausible magnitudes) and is not shown here.
top_n = 200

freq = (data_clean_
        .groupby('WIN_NAME_CANONICAL')
        .size()
        .rename('n_awards')
        .sort_values(ascending=False)
        .reset_index())

print(f"Unique suppliers: {len(freq)}")
print(f"\nTop {top_n} suppliers by n_awards:")
print(freq.head(top_n).to_string(index=False))

Unique suppliers: 10972

Top 200 suppliers by n_awards:
                                         WIN_NAME_CANONICAL  n_awards
                                 Computacenter AG & Co. oHG       769
                          fujitsu technology solutions gmbh       722
                                               bechtle gmbh       694
                                     Ricoh Deutschland GmbH       603
                               t-systems international gmbh       511
                         SVA System Vertrieb Alexander GmbH       506
                                                cancom gmbh       456
                                       IBM Deutschland GmbH       447
                                      bechtle gmbh & co. kg       442
                                          Papier Union GmbH       355
                                SAP Deutschland SE & Co. KG       354
                                    Lyreco Deutschland GmbH       334
                                  

In [53]:
# Check for HP/Hewlett-Packard variants in canonical names
hp_mask = freq["WIN_NAME_CANONICAL"].str.contains(
    r"hewlett|packard|\bhpe\b|\bhp\b",
    case=False, regex=True
)
print(freq[hp_mask].to_string(index=False))

                               WIN_NAME_CANONICAL  n_awards
                             Hewlett-Packard GmbH       162
                       Hewlett Packard Enterprise        11
                              HP Deutschland GmbH        10
                        hp schmitz x digital gmbh         6
                 hewlett packard deutschland gmbh         2
                             hp schmitz x digital         1
                                   hp indigo b.v.         1
                           hp finanz service gmbh         1
           hewlett-packard international bank plc         1
                 hewlett-packard deutschland gmbh         1
hewlett packard enterprise / hewlett-packard gmbh         1


In [54]:
# --- Parent mapping: explicit dictionary --------------------------------------
# Every key is a canonical name observed in the data (see assert below).
# Only corporate families with more than one spelling are listed; single-spelling
# firms need no entry. The long tail is handled by the regex rules that follow.

PARENT_MAPPING = {
    # Bechtle
    'bechtle gmbh':                                     'Bechtle AG',
    'bechtle gmbh & co. kg':                            'Bechtle AG',
    'bechtle ag':                                       'Bechtle AG',
    'bechtle gmbh it-systemhaus':                       'Bechtle AG',
    'bechtle gmbh & co. kg it-systemhaus neckarsulm':   'Bechtle AG',
    'Bechtle direct GmbH':                              'Bechtle AG',
    'Bechtle Nürnberg IT-Systemhaus GmbH':              'Bechtle AG',
    'bechtle it-systemhaus gmbh & co. kg':              'Bechtle AG',
    'Bechtle direct Public Sector GmbH':                'Bechtle AG',
    'bechtle gmbh & co.kg':                             'Bechtle AG',
    'bechtle gmbh it-systemhaus weimar':                'Bechtle AG',

    # Deutsche Telekom (incl. T-Systems, Detecon)
    't-systems international gmbh':                     'Deutsche Telekom AG',
    'Telekom Deutschland GmbH':                         'Deutsche Telekom AG',
    'Deutsche Telekom Business Solutions GmbH':         'Deutsche Telekom AG',
    'T-Systems Enterprise Services GmbH':               'Deutsche Telekom AG',
    'detecon international gmbh':                       'Deutsche Telekom AG',

    # Cancom
    'cancom gmbh':                                      'Cancom SE',
    'cancom on line gmbh':                              'Cancom SE',
    'Cancom Deutschland GmbH':                          'Cancom SE',

    # Atos
    'Atos Information Technology GmbH':                 'Atos SE',
    'Atos IT Solutions and Services GmbH':              'Atos SE',

    # adesso
    'adesso se':                                        'adesso SE',
    'adesso ag':                                        'adesso SE',

    # Sopra Steria
    'sopra steria se':                                  'Sopra Steria SE',
    'sopra steria gmbh':                                'Sopra Steria SE',
    'steria mummert consulting ag':                     'Sopra Steria SE',

    # Materna
    'Materna Information & Communications SE':          'Materna SE',
    'materna gmbh':                                     'Materna SE',

    # NTT
    'NTT DATA Deutschland GmbH':                        'NTT Group',
    'ntt germany ag & co. kg':                          'NTT Group',
    'Dimension Data Germany AG & Co. KG':               'NTT Group',

    # SVA
    'SVA System Vertrieb Alexander GmbH':               'SVA GmbH',
    'sva gmbh':                                         'SVA GmbH',

    # Rednet
    'rednet ag':                                        'Rednet AG',
    'rednet gmbh':                                      'Rednet AG',

    # ITMediaConsult
    'ITMediaConsult AG':                                'ITMediaConsult AG',
    'IT Media Consult AG':                              'ITMediaConsult AG',

    # SPIRIT/21
    'spirit/21 gmbh':                                   'SPIRIT/21 AG',
    'spirit/21 ag':                                     'SPIRIT/21 AG',

    # Comparex
    'Comparex AG':                                      'Comparex AG',
    'Comparex Deutschland AG':                          'Comparex AG',

    # PricewaterhouseCoopers
    'PricewaterhouseCoopers GmbH WPG':                  'PricewaterhouseCoopers GmbH WPG',
    'pricewaterhousecoopers gmbh wirtschaftsprüfungsgesellschaft':
                                                        'PricewaterhouseCoopers GmbH WPG',

    # H&G Hansen & Gieraths
    'H&G Hansen & Gieraths IT Solutions GmbH':          'H&G Hansen & Gieraths',
    'H&G Hansen & Gieraths EDV Vertriebsgesellschaft mbH':
                                                        'H&G Hansen & Gieraths',

    # Konica Minolta
    'Konica Minolta Business Solutions Deutschland GmbH':
                                                        'Konica Minolta Business Solutions Deutschland GmbH',
    'Konica Minolta':                                   'Konica Minolta Business Solutions Deutschland GmbH',

    # MR Datentechnik
    'MR Datentechnik Vertriebs- und Service GmbH':      'MR Datentechnik',
    'mr datentechnik gmbh':                             'MR Datentechnik',

    # Dell
    'dell gmbh':                                        'Dell GmbH',
    'Dell Halle GmbH':                                  'Dell GmbH',

    # Intertronic
    'intertronic computer gmbh':                        'Intertronic GmbH',
    'intertronic it gmbh':                              'Intertronic GmbH',

    # MEGWARE
    'MEGWARE Computer Vertrieb und Service GmbH':       'MEGWARE Computer GmbH',
    'megware computer gmbh':                            'MEGWARE Computer GmbH',

    # Datagroup
    'datagroup consulting services gmbh':               'Datagroup SE',
    'Datagroup Stuttgart GmbH':                         'Datagroup SE',

    # DXC (CSC merged into DXC in 2017)
    'DXC Technology Deutschland GmbH':                  'DXC Technology Deutschland GmbH',
    'CSC Deutschland Solutions GmbH':                   'DXC Technology Deutschland GmbH',

    # CONET
    'CONET Solutions GmbH':                             'CONET Technologies Holding GmbH',
    'CONET Business Consultants GmbH':                  'CONET Technologies Holding GmbH',

    # V-BC.de
    'v-bc.de':                                          'V-BC.de',
    'V-BC.de, Inhaber Heiko Protze':                    'V-BC.de',

    # Hewlett-Packard (HP Schmitz X Digital is a different firm and is excluded)
    'Hewlett-Packard GmbH':                             'Hewlett-Packard',
    'Hewlett Packard Enterprise':                       'Hewlett-Packard',
    'HP Deutschland GmbH':                              'Hewlett-Packard',
    'hewlett packard deutschland gmbh':                 'Hewlett-Packard',
    'hewlett-packard deutschland gmbh':                 'Hewlett-Packard',
    'hewlett packard enterprise / hewlett-packard gmbh': 'Hewlett-Packard',
    'hp indigo b.v.':                                   'Hewlett-Packard',
    'hp finanz service gmbh':                           'Hewlett-Packard',
    'hewlett-packard international bank plc':           'Hewlett-Packard',
}

# Every key must exist in the data; a stale key would fail silently under .map().
_present = set(data_clean_['WIN_NAME_CANONICAL'].unique())
_dead = sorted(k for k in PARENT_MAPPING if k not in _present)
assert not _dead, _dead
assert len(PARENT_MAPPING) == 72, len(PARENT_MAPPING)

data_clean_['WIN_NAME_FINAL'] = (data_clean_['WIN_NAME_CANONICAL']
                                 .map(PARENT_MAPPING)
                                 .fillna(data_clean_['WIN_NAME_CANONICAL']))

_hit = data_clean_['WIN_NAME_CANONICAL'].isin(PARENT_MAPPING)
print(f"Keys: {len(PARENT_MAPPING)} | parents: {len(set(PARENT_MAPPING.values()))}")
print(f"Rows mapped to a parent: {_hit.sum()}")
print(f"Distinct names: {data_clean_['WIN_NAME_CANONICAL'].nunique()} "
      f"-> {data_clean_['WIN_NAME_FINAL'].nunique()}")
print("\nAwards per parent:")
print(data_clean_.loc[_hit].groupby('WIN_NAME_FINAL').size()
      .sort_values(ascending=False).to_string())

Keys: 72 | parents: 25
Rows mapped to a parent: 7758
Distinct names: 10972 -> 10925

Awards per parent:
WIN_NAME_FINAL
Bechtle AG                                            1724
Deutsche Telekom AG                                    798
SVA GmbH                                               609
Cancom SE                                              582
Comparex AG                                            334
Konica Minolta Business Solutions Deutschland GmbH     315
Atos SE                                                301
adesso SE                                              266
CONET Technologies Holding GmbH                        261
ITMediaConsult AG                                      257
Dell GmbH                                              254
Materna SE                                             246
Sopra Steria SE                                        195
Hewlett-Packard                                        190
Rednet AG                                              

In [55]:
# --- Residual check after the first mapping pass: ranked by award count ---
# Read-only. VALUE_EURO is not usable and is not consulted here.
_res = (data_clean_.groupby('WIN_NAME_FINAL').size()
        .rename('n_awards').sort_values(ascending=False).reset_index())
_res['mapped'] = _res['WIN_NAME_FINAL'].isin(PARENT_MAPPING.values())

print(f"Unique WIN_NAME_FINAL: {len(_res)}")
print("\nTop 100 not yet mapped to a parent:")
print(_res.loc[~_res['mapped']].head(100).to_string(index=False))

Unique WIN_NAME_FINAL: 10925

Top 100 not yet mapped to a parent:
                                    WIN_NAME_FINAL  n_awards  mapped
                        Computacenter AG & Co. oHG       769   False
                 fujitsu technology solutions gmbh       722   False
                            Ricoh Deutschland GmbH       603   False
                              IBM Deutschland GmbH       447   False
                                 Papier Union GmbH       355   False
                       SAP Deutschland SE & Co. KG       354   False
                           Lyreco Deutschland GmbH       334   False
                                           hays ag       331   False
                 Papyrus Deutschland GmbH & Co. KG       270   False
                               think about it gmbh       197   False
                        Microsoft Deutschland GmbH       195   False
                  Oracle Deutschland B.V. & Co. KG       190   False
                                   CH

In [56]:
# --- Per-brand sweep: find every spelling belonging to a known parent ---
# Read-only. Basis for the regex rules; also surfaces name collisions.
BRAND_PROBES = {
    'Bechtle AG':          r'\bbechtle\b',
    'Deutsche Telekom AG': r'^deutsche telekom|^telekom deutschland|\bt-systems\b|\bdetecon\b',
    'Cancom SE':           r'\bcancom\b',
    'Atos SE':             r'\batos\b',
    'adesso SE':           r'\badesso\b',
    'Sopra Steria SE':     r'\bsopra\b|\bsteria\b',
    'Materna SE':          r'\bmaterna\b',
    'Datagroup SE':        r'\bdatagroup\b',
    'msg systems ag':      r'\bmsg\b',
    'Capgemini':           r'\bcapgemini\b',
    'Deloitte':            r'\bdeloitte\b',
    'KPMG':                r'\bkpmg\b',
    'CGI':                 r'\bcgi\b|\blogica\b',
}

_names = (data_clean_['WIN_NAME_FINAL']
          .value_counts().rename_axis('name').reset_index(name='n'))

for parent, probe in BRAND_PROBES.items():
    hit = _names.loc[_names['name'].str.contains(probe, case=False, regex=True)]
    print(f"\n=== {parent}  ({len(hit)} names, {hit['n'].sum()} rows) ===")
    print(hit.to_string(index=False))


=== Bechtle AG  (143 names, 2203 rows) ===
                                                                                                                                                                                                   name    n
                                                                                                                                                                                             Bechtle AG 1724
                                                                                                                                                                         bechtle gmbh it-systemhaus ulm   29
                                                                                                                                                                                                bechtle   27
                                                                                                                                        

In [57]:
# --- Names matching more than one brand: excluded from regex mapping ---
# A multi-firm entry cannot be assigned to a single parent. Consistent with the
# separator-cleanup decision to leave such entries unresolved.
_probe_hits = pd.DataFrame(
    {p: _names['name'].str.contains(rx, case=False, regex=True)
     for p, rx in BRAND_PROBES.items()},
    index=_names.index)

_names['n_brands'] = _probe_hits.sum(axis=1)
ambiguous = _names.loc[_names['n_brands'] > 1]

print(f"Ambiguous (multi-brand) names: {len(ambiguous)}, {ambiguous['n'].sum()} rows")
print(ambiguous[['name', 'n', 'n_brands']].sort_values('n_brands', ascending=False)
      .to_string(index=False))

Ambiguous (multi-brand) names: 6, 6 rows
                                                                                                                                                                                                                     name  n  n_brands
                                                            1.) accenture gmbh 2.) adesso ag 3.) bearing point gmbh 4.) bietergemeinschaft deloitte consulting gmbh und deloitte gmbh 5.) q-perior ag 6.) sopra steria se  1         3
                                                                                     protechnology gmbh, allgeier it services gmbh, bechtle  gmbh, controlware gmbh, datagroup hamburg gmbh, t-systems international gmbh  1         3
                                                                                                                          cgi deutschland ltd. & co. kg/kpmg ag/pricewaterhousecoopers ag/sqs software quality systems ag  1         2
                                   

In [58]:
# --- Parent mapping, second pass: regex rules for decentralised groups --------
# Applied to canonical names not covered by the explicit PARENT_MAPPING.
# Patterns are anchored with word boundaries: "\bt-systems\b" does not match
# "it-systems", "\blogica\b" does not match "logicalis".
# Idempotent: WIN_NAME_FINAL is rebuilt from WIN_NAME_CANONICAL on every run.

REGEX_PARENT_RULES = [
    (r'\bbechtle\b',                                  'Bechtle AG'),
    (r'^deutsche telekom|^telekom deutschland'
     r'|\bt-systems\b|\bdetecon\b',                   'Deutsche Telekom AG'),
    (r'\bcancom\b',                                   'Cancom SE'),
    (r'\batos\b',                                     'Atos SE'),
    (r'\badesso\b',                                   'adesso SE'),
    (r'\bsopra\b|\bsteria\b',                         'Sopra Steria SE'),
    (r'\bmaterna\b',                                  'Materna SE'),
    (r'\bdatagroup\b',                                'Datagroup SE'),
    (r'\bmsg\b',                                      'msg systems ag'),
    (r'\bcapgemini\b',                                'Capgemini Deutschland GmbH'),
    (r'\bdeloitte\b',                                 'Deloitte GmbH Wirtschaftsprüfungsgesellschaft'),
    (r'\bkpmg\b',                                     'KPMG AG Wirtschaftsprüfungsgesellschaft'),
    (r'\bcgi\b|\blogica\b',                           'CGI Deutschland B.V. & Co. KG'),
    (r'\bconet\b',                                    'CONET Technologies Holding GmbH'),
    (r'\bcomparex\b',                                 'Comparex AG'),
    (r'\bfujitsu\b',                                  'Fujitsu Technology Solutions GmbH'),
    (r'\bntt\b|\bdimension data\b',                   'NTT Group'),
    (r'\bhewlett\b|\bpackard\b',                      'Hewlett-Packard'),
]

# Worldline was spun off from Atos in 2014 and is an independent company.
REGEX_EXCLUDE = r'\bworldline\b'

# --- Candidate names: everything the explicit dictionary does not cover -------
_cand = (pd.Series(data_clean_['WIN_NAME_CANONICAL'].unique(), name='name')
         .to_frame())
_cand = _cand.loc[~_cand['name'].isin(PARENT_MAPPING)].reset_index(drop=True)

_match = pd.DataFrame({parent: _cand['name'].str.contains(rx, case=False, regex=True)
                       for rx, parent in REGEX_PARENT_RULES}, index=_cand.index)

_cand['n_rules'] = _match.sum(axis=1)
_cand['excluded'] = _cand['name'].str.contains(REGEX_EXCLUDE, case=False, regex=True)

# Consortium entries name several firms; the regex rules would attribute the
# whole award to whichever brand happens to be listed. Excluded, consistent
# with the separator cleanup, which left these entries unresolved.
_cand['bucket'] = _cand['name'].map(classify)
_cand['is_consortium'] = (_cand['bucket'].str.startswith('consortium')
                          | _cand['bucket'].eq('unclear'))

# A name matching several brand rules names several firms and cannot be
# assigned to a single parent either.
_ambiguous = _cand.loc[_cand['n_rules'] > 1]

_ok = _cand.loc[(_cand['n_rules'] == 1)
                & ~_cand['excluded']
                & ~_cand['is_consortium']]

REGEX_MAP = dict(zip(_ok['name'], _match.loc[_ok.index].idxmax(axis=1)))

_rule_hit = _cand['n_rules'] == 1
print(f"Names matching exactly one rule:      {_rule_hit.sum()}")
print(f"  minus Worldline exclusion:          {(_rule_hit & _cand['excluded']).sum()}")
print(f"  minus consortium entries:           {(_rule_hit & _cand['is_consortium']).sum()}")
print(f"Names mapped by regex:                {len(REGEX_MAP)}")

# --- Apply --------------------------------------------------------------------
data_clean_['WIN_NAME_FINAL'] = (data_clean_['WIN_NAME_CANONICAL']
                                 .map(PARENT_MAPPING)
                                 .fillna(data_clean_['WIN_NAME_CANONICAL'].map(REGEX_MAP))
                                 .fillna(data_clean_['WIN_NAME_CANONICAL']))

# Worldline is independent of Atos and must never be folded into it.
_wl = data_clean_['WIN_NAME_CANONICAL'].str.contains(r'\bworldline\b', case=False)
assert data_clean_.loc[_wl, 'WIN_NAME_FINAL'].ne('Atos SE').all(), "Worldline mapped to Atos"

_hit = (data_clean_['WIN_NAME_CANONICAL'].isin(PARENT_MAPPING)
        | data_clean_['WIN_NAME_CANONICAL'].isin(REGEX_MAP))

print(f"\nRows mapped to a parent: {_hit.sum()}")
print(f"Distinct names: {data_clean_['WIN_NAME_CANONICAL'].nunique()} "
      f"-> {data_clean_['WIN_NAME_FINAL'].nunique()}")
print("\nAwards per parent:")
print(data_clean_.loc[_hit].groupby('WIN_NAME_FINAL').size()
      .sort_values(ascending=False).to_string())

print("\n--- consortium names excluded from regex mapping ---")
print(_cand.loc[_cand['is_consortium'] & (_cand['n_rules'] == 1), ['name']]
      .to_string(index=False))

Names matching exactly one rule:      376
  minus Worldline exclusion:          1
  minus consortium entries:           10
Names mapped by regex:                365

Rows mapped to a parent: 10284
Distinct names: 10972 -> 10566

Awards per parent:
WIN_NAME_FINAL
Bechtle AG                                            2197
Deutsche Telekom AG                                    948
Fujitsu Technology Solutions GmbH                      808
Cancom SE                                              714
SVA GmbH                                               609
Comparex AG                                            342
Atos SE                                                327
Konica Minolta Business Solutions Deutschland GmbH     315
adesso SE                                              275
CONET Technologies Holding GmbH                        264
ITMediaConsult AG                                      257
Dell GmbH                                              254
Materna SE                   

In [59]:
# check regex-based mappings
# --- Regex mappings where the brand is not the leading token: read-only ---
_chk = pd.Series(REGEX_MAP).rename('parent').to_frame().rename_axis('name').reset_index()
_lead = {p: rx for rx, p in REGEX_PARENT_RULES}
_chk['leading'] = [bool(re.match(_lead[p], n, flags=re.I))
                   for n, p in zip(_chk['name'], _chk['parent'])]

sus = _chk.loc[~_chk['leading']]
print(f"{len(sus)} of {len(_chk)} mapped names do not start with the brand token\n")
print(sus.sort_values('parent').to_string(index=False))

35 of 365 mapped names do not start with the brand token

                                                                          name                                        parent  leading
                                        science + computing ag an atos company                                       Atos SE    False
                                                 bull gmbh – atos technologies                                       Atos SE    False
                                                        bull — an atos company                                       Atos SE    False
atocris consortium (leader: atos belgium s.a., member: c.r.i. luxembourg s.a.)                                       Atos SE    False
               c.r.i. luxembourg s.a. (consortium partner to leader atos s.a.)                                       Atos SE    False
                                                            firma bechtle gmbh                                    Bechtle AG    False
    

### Validation of the regex mappings

Regex assignments were checked by inspecting all matches in which the brand token is
not the leading element of the name (35 of 365), the class most likely to produce false
positives. 29 are correct: `fa.` and `firma` prefixes, group subsidiaries (Bull,
science + computing, B&W Deloitte), and one consortium whose named leader is the parent.
Six are multi-firm entries without a consortium marker, attributed to the single listed
brand, including one in which the named contractor is explicitly not that brand
(`c.r.i. luxembourg s.a. (consortium partner to leader atos s.a.)`). They are left
unchanged: each occurs once, together less than 0.02 % of supplier-award pairs, and
correcting them by hand would replace a reproducible procedure with a list of
incidental findings.

In [60]:
# ── Final summary statistics after parent mapping ─────────────────────────────
# VALUE_EURO is not usable (implausible magnitudes) and is not reported.
# Rows are supplier-award pairs: consortium awards contribute one row per member.

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Supplier-award pairs (rows): {len(data_clean_):>10,}")
print(f"Unique buyers (CAE_NAME):    {data_clean_['CAE_NAME'].nunique():>10,}")
print(f"Unique WIN_NAME_CANONICAL:   {data_clean_['WIN_NAME_CANONICAL'].nunique():>10,}")
print(f"Unique WIN_NAME_FINAL:       {data_clean_['WIN_NAME_FINAL'].nunique():>10,}")

print("\n" + "=" * 60)
print("SPLIT FLAGS")
print("=" * 60)
print(f"IS_SPLIT = True:             {data_clean_['IS_SPLIT'].sum():>10,}")
print(f"IS_SPLIT = False:            {(~data_clean_['IS_SPLIT']).sum():>10,}")

print("\n" + "=" * 60)
print("PARENT MAPPING COVERAGE")
print("=" * 60)
_expl  = data_clean_['WIN_NAME_CANONICAL'].isin(PARENT_MAPPING)
_rgx   = data_clean_['WIN_NAME_CANONICAL'].isin(REGEX_MAP) & ~_expl
_unmap = ~(_expl | _rgx)
print(f"Explicitly mapped rows:      {_expl.sum():>10,}")
print(f"Regex-mapped rows:           {_rgx.sum():>10,}")
print(f"Unmapped rows:               {_unmap.sum():>10,}")
print(f"Coverage:                    {(_expl | _rgx).mean():>10.1%}")

print("\n" + "=" * 60)
print("TOP 10 SUPPLIERS BY AWARD COUNT (WIN_NAME_FINAL)")
print("=" * 60)
print(data_clean_.groupby('WIN_NAME_FINAL').size()
      .rename('n_awards').sort_values(ascending=False).head(10).to_string())

print("\n" + "=" * 60)
print("TEMPORAL COVERAGE")
print("=" * 60)
print(f"Year range: {data_clean_["YEAR"].min()} - {data_clean_["YEAR"].max()}")
print(data_clean_["YEAR"].value_counts().sort_index().to_string())

DATASET OVERVIEW
Supplier-award pairs (rows):     48,075
Unique buyers (CAE_NAME):         7,602
Unique WIN_NAME_CANONICAL:       10,972


Unique WIN_NAME_FINAL:           10,566

SPLIT FLAGS
IS_SPLIT = True:                  1,561
IS_SPLIT = False:                46,514

PARENT MAPPING COVERAGE
Explicitly mapped rows:           7,758
Regex-mapped rows:                2,526
Unmapped rows:                   37,791
Coverage:                         21.4%

TOP 10 SUPPLIERS BY AWARD COUNT (WIN_NAME_FINAL)
WIN_NAME_FINAL
Bechtle AG                           2197
Deutsche Telekom AG                   948
Fujitsu Technology Solutions GmbH     808
Computacenter AG & Co. oHG            769
Cancom SE                             714
SVA GmbH                              609
Ricoh Deutschland GmbH                603
IBM Deutschland GmbH                  447
Papier Union GmbH                     355
SAP Deutschland SE & Co. KG           354

TEMPORAL COVERAGE
Year range: 2009 - 2023
YEAR
2009    1380
2010    1652
2011    1752
2012    1869
2013    2033
2014    1945
2015    2214
2016    2505
2017    2801
2018    3084
2019    3809
2020  

In [61]:
data_clean_.to_csv("data_checkpoints/data_supplier_pr.csv")

In [62]:
data_clean_ = pd.read_csv("data_checkpoints/data_supplier_pr.csv")
data_clean_.columns

Index(['Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1',
       'Unnamed: 0', 'ID_NOTICE_CAN', 'YEAR', 'CAE_NAME', 'ISO_COUNTRY_CODE',
       'CAE_TYPE', 'MAIN_ACTIVITY', 'B_ON_BEHALF', 'TYPE_OF_CONTRACT',
       'TAL_LOCATION_NUTS', 'B_FRA_AGREEMENT', 'FRA_ESTIMATED',
       'B_FRA_CONTRACT', 'CPV', 'LOTS_NUMBER', 'VALUE_EURO',
       'VALUE_EURO_FIN_1', 'VALUE_EURO_FIN_2', 'B_EU_FUNDS', 'TOP_TYPE',
       'CRIT_CODE', 'NUMBER_AWARDS', 'INFO_ON_NON_AWARD', 'INFO_UNPUBLISHED',
       'WIN_NAME', 'WIN_NATIONALID', 'WIN_COUNTRY_CODE', 'B_CONTRACTOR_SME',
       'NUMBER_OFFERS', 'AWARD_VALUE_EURO', 'B_SUBCONTRACTED', 'DT_AWARD',
       'YEAR_ID', 'IS_SPLIT', 'WIN_NAME_NORM', 'WIN_NAME_NORM_SORTED',
       'WIN_NAME_CANONICAL', 'WIN_NAME_FINAL'],
      dtype='str')

# Buyer Side

### Buyer-side identifiers: checked, not used

`ID_NOTICE_CAN` identifies the notice rather than the award, so it is not unique
across the 48,075 supplier-award pairs. To test whether the raw exports offer a
stable identifier or a geographic blocking key for buyer-side entity resolution,
`CAE_NATIONALID` and `B_MULTIPLE_CAE` were merged back in from the yearly source
files and the granularity of `TAL_LOCATION_NUTS` was inspected. Neither signal is
usable: the national ID is populated for only 1,485 of 48,075 rows (3.1 %), and
NUTS codes are inconsistent in granularity (two to five characters for most rows,
concatenated multi-region strings for joint procurements) while frequent buyers
such as ITZBund appear across up to 59 distinct NUTS values, which would make the
region a source of false negatives rather than a reliable blocking key. Buyer-side
consolidation therefore proceeds on names alone; the two merged columns remain in
the frame but are not used in any later step.

In [63]:
# Check if ID_NOTICE_CAN is truly unique in data_clean_
n_rows = len(data_clean_)
n_unique = data_clean_['ID_NOTICE_CAN'].nunique()

print(f"Rows in data_clean_:          {n_rows}")
print(f"Unique ID_NOTICE_CAN values:  {n_unique}")
print(f"IDs are unique: {n_rows == n_unique}")

Rows in data_clean_:          48075
Unique ID_NOTICE_CAN values:  31209
IDs are unique: False


In [64]:
#adding additional columns
EXTRA_COLS = ['CAE_NATIONALID', 'B_MULTIPLE_CAE']

ids_needed = set(data_clean_['ID_NOTICE_CAN'])
raw_chunks = []

for year in range(2009, 2024):
    filepath = f"can_data/export_CAN_{year}.csv"
    if not os.path.exists(filepath):
        print(f"Warning: {filepath} not found, skipping.")
        continue

    # Check which extra cols are actually present in this file
    available_cols = pd.read_csv(filepath, nrows=0).columns.tolist()
    cols_present = [c for c in EXTRA_COLS if c in available_cols]
    missing = [c for c in EXTRA_COLS if c not in available_cols]
    if missing:
        print(f"{year}: columns not in file, will be NaN: {missing}")

    # Read chunkwise, filter to DE and needed IDs immediately
    year_rows = []
    for chunk in pd.read_csv(filepath, usecols=['ID_NOTICE_CAN', 'ISO_COUNTRY_CODE'] + cols_present,
                             low_memory=False, chunksize=50000):
        filtered = chunk[
            (chunk['ISO_COUNTRY_CODE'] == 'DE') &
            (chunk['ID_NOTICE_CAN'].isin(ids_needed))
        ]
        year_rows.append(filtered)

    year_df = pd.concat(year_rows, ignore_index=True)
    raw_chunks.append(year_df)
    print(f"{year}: done, kept {len(year_df)} matching rows")

raw_all = pd.concat(raw_chunks, ignore_index=True)
print(f"\nTotal matching rows: {len(raw_all)}")

# Deduplicate on raw side (one row per notice)
raw_filtered = raw_all.drop_duplicates(subset='ID_NOTICE_CAN')
print(f"After dedup: {len(raw_filtered)} rows")

# Merge into data_clean_
data_clean_ = data_clean_.merge(
    raw_filtered[['ID_NOTICE_CAN'] + cols_present],
    on='ID_NOTICE_CAN',
    how='left'
)

print(f"\nFinal shape: {data_clean_.shape}")
print("\nCoverage of new columns:")
print(data_clean_[cols_present].notna().sum())

2009: done, kept 1403 matching rows
2010: done, kept 1679 matching rows
2011: done, kept 1761 matching rows
2012: done, kept 1894 matching rows
2013: done, kept 2003 matching rows
2014: done, kept 1952 matching rows
2015: done, kept 2203 matching rows
2016: done, kept 2567 matching rows
2017: done, kept 2804 matching rows
2018: done, kept 3179 matching rows
2019: done, kept 3896 matching rows
2020: done, kept 5198 matching rows
2021: done, kept 6312 matching rows
2022: done, kept 6226 matching rows
2023: done, kept 5958 matching rows

Total matching rows: 49035
After dedup: 31209 rows

Final shape: (48075, 44)

Coverage of new columns:
CAE_NATIONALID     1485
B_MULTIPLE_CAE    32022
dtype: int64


In [65]:
# Understand NUTS granularity in the data
print("=== NUTS value lengths ===")
nuts_lengths = data_clean_['TAL_LOCATION_NUTS'].dropna().str.len().value_counts().sort_index()
print(nuts_lengths)

print("\n=== NUTS prefix examples (first 3 chars) ===")
print(data_clean_['TAL_LOCATION_NUTS'].dropna().str[:3].value_counts().head(15))

print("\n=== Missing NUTS ===")
print(f"Missing: {data_clean_['TAL_LOCATION_NUTS'].isna().sum()} / {len(data_clean_)}")

=== NUTS value lengths ===
TAL_LOCATION_NUTS
2       3724
3       4912
4       2712
5      28581
6        141
7        129
8        250
9        214
10       109
11       830
12        32
13        95
14        52
15        58
16        22
17       239
18        36
19        11
20        11
21         7
22        11
23        70
24         2
25         7
26         4
27         2
28        20
29        39
31         2
32         3
33        17
34         4
35        30
38         6
39         3
41        26
42         1
43         3
44         1
47         5
52         1
53        11
58         9
59         6
60         1
62         1
63         4
65         4
66         1
69         4
71        13
77         3
83         3
89         1
95         1
107        1
118        1
125        1
191        1
Name: count, dtype: int64

=== NUTS prefix examples (first 3 chars) ===
TAL_LOCATION_NUTS
DEA    7879
DE2    7044
DE1    4607
DE7    3973
DE     3679
DE3    3605
DED    1757
DE9    1739
DE

In [66]:
# Find how many distinct NUTS regions the most common buyer names appear in
# This directly shows where NUTS blocking would cause false negatives
name_nuts = (data_clean_
    .groupby('CAE_NAME')['TAL_LOCATION_NUTS']
    .nunique()
    .sort_values(ascending=False)
)

print("=== Buyer names appearing in most NUTS regions ===")
print(name_nuts.head(30))

print("\n=== Distribution: how many names span >1 NUTS region? ===")
print(name_nuts.value_counts().sort_index().head(10))

=== Buyer names appearing in most NUTS regions ===
CAE_NAME
Informationstechnikzentrum Bund (ITZBund)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               59
BWI GmbH                                                                                                                                                         

### Joint contracting authorities

Some notices are issued by several public bodies at once, which TED records as a
single concatenated `CAE_NAME`. The following cells identify these cases and
resolve them into individual buyers, so that the buyer side can be analysed at
the level of the actual contracting authority.

In [67]:
# Diagnosis Multi-Buyer-Entries

# 1. How many notices have B_MULTIPLE_CAE flagged?
print("=== B_MULTIPLE_CAE distribution ===")
print(data_clean_['B_MULTIPLE_CAE'].value_counts(dropna=False))

# 2. How many CAE_NAME entries contain '---'?
multi_sep = data_clean_['CAE_NAME'].str.contains('---', na=False)
print(f"\nCAE_NAME entries with '---': {multi_sep.sum()}")

# 3. How many contain ';'?
semi_sep = data_clean_['CAE_NAME'].str.contains(';', na=False)
print(f"CAE_NAME entries with ';':   {semi_sep.sum()}")

# 4. Show a few examples
print("\n=== Examples with '---' ===")
print(data_clean_.loc[multi_sep, 'CAE_NAME'].head(5).tolist())

print("\n=== Examples with ';' ===")
print(data_clean_.loc[semi_sep, 'CAE_NAME'].head(5).tolist())

# 5. Cross-check with B_MULTIPLE_CAE flag
print("\n=== '---' entries vs B_MULTIPLE_CAE ===")
print(pd.crosstab(multi_sep, data_clean_['B_MULTIPLE_CAE'].fillna('missing')))

=== B_MULTIPLE_CAE distribution ===
B_MULTIPLE_CAE
N      31024
NaN    16053
Y        998
Name: count, dtype: int64

CAE_NAME entries with '---': 998
CAE_NAME entries with ';':   4733

=== Examples with '---' ===
['Bundesarbeitsgemeinschaft für Sicherheit und Gesundheit bei der Arbeit (Basi) e. V.---Deutsche Gesetzliche Unfallversicherung e. V.---Verein zur Förderung der Arbeitssicherheit e. V. (VfA)', 'Bayerische Landesbodenkreditanstalt---Bremer Aufbau-Bank GmbH---Hamburgische Investitions- und Förderbank---Investitions- und Förderbank Niedersachsen GmbH---Investitions- und Strukturbank Rheinland-Pfalz GmbH---Investitionsbank Berlin---Investitionsbank Schleswig-Holstein---Investitionsbank des Landes Brandenburg---Wirtschafts- und Infrastrukturbank Hessen', 'Land Hessen; vertreten durch das Regierungspräsidium Kassel---Land Rheinland-Pfalz; vertreten durch das Landesamt für Finanzen', 'Das Land Hessen; vertreten durch die Hessische Zentrale für Datenverarbeitung---ekom21', 'Das Land H

In [68]:
# check how many cases would be affected by "double"-splitting
mask_multi_cae = data_clean_['CAE_NAME'].str.contains('---', na=False)
print(pd.crosstab(data_clean_['IS_SPLIT'], mask_multi_cae.rename('multi_cae')))

multi_cae  False  True 
IS_SPLIT               
False      45566    948
True        1511     50


In [69]:
# Inspect the 50 cases that are both supplier-split and multi-buyer
mask_multi_cae = data_clean_['CAE_NAME'].str.contains('---', na=False)
double_split = data_clean_[data_clean_['IS_SPLIT'] & mask_multi_cae]

print(f"Double-split cases: {len(double_split)}")
print(f"Unique ID_NOTICE_CAN: {double_split['ID_NOTICE_CAN'].nunique()}")

# Show relevant columns for manual inspection
cols_inspect = ['ID_NOTICE_CAN', 'CAE_NAME', 'WIN_NAME', 'IS_SPLIT']
print(double_split[cols_inspect].to_string())

Double-split cases: 50
Unique ID_NOTICE_CAN: 13
       ID_NOTICE_CAN                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          CAE_NAME                                        WIN_NAME  IS_SPLIT
46398     2016275497                                                                                                                                                                                                                                   

In [70]:
# From here on, data_clean_ holds the BUYER frame: one row per
# buyer-supplier-award pair. The supplier-award frame is preserved as
# data_supplier_ and must be used for any supplier-level figure.
# TED concatenates joint contracting authorities with '---' in CAE_NAME;
# the separator matches B_MULTIPLE_CAE == 'Y' exactly (998/998), so it is
# a reliable delimiter rather than a heuristic.
# No value column is rescaled - monetary fields are not used in this thesis.
if 'data_supplier_' not in globals():
    data_supplier_ = data_clean_.copy()

assert len(data_supplier_) == 48075, "supplier frame changed upstream"

mask_multi_cae = data_supplier_['CAE_NAME'].str.contains('---', na=False)
assert mask_multi_cae.sum() == 998, "multi-buyer row count changed"

data_clean_ = data_supplier_.copy()
data_clean_['IS_SPLIT_BUYER'] = mask_multi_cae
data_clean_['CAE_NAME'] = (data_clean_['CAE_NAME']
                           .str.split('---')
                           .apply(lambda parts: [p.strip() for p in parts if p.strip()]))
data_clean_ = data_clean_.explode('CAE_NAME', ignore_index=True)

assert not data_clean_['CAE_NAME'].str.contains('---').any()
assert data_clean_['CAE_NAME'].str.strip().eq(data_clean_['CAE_NAME']).all()
assert len(data_clean_) == 56277, "buyer frame row count changed"
assert data_clean_['IS_SPLIT_BUYER'].sum() == 9200, "buyer split count changed"
assert data_clean_['CAE_NAME'].nunique() == 8393, "distinct buyer names changed"

print(f"Supplier frame (data_supplier_): {len(data_supplier_)} rows")
print(f"Buyer frame (data_clean_):       {len(data_clean_)} rows")
print(f"Distinct raw buyer names:        {data_clean_['CAE_NAME'].nunique()}")

Supplier frame (data_supplier_): 48075 rows
Buyer frame (data_clean_):       56277 rows
Distinct raw buyer names:        8393


The following cells try to identify further cases with several suppliers, explicitly looking out for patterns already identified on the supplier side

In [71]:
# Diagnosis semicolon cases in CAE_NAME
mask_semi = data_clean_['CAE_NAME'].str.contains(';', na=False)
print(f"CAE_NAME entries with ';': {mask_semi.sum()}")

# Show a sample of 30 for manual inspection
sample = (data_clean_.loc[mask_semi, 'CAE_NAME']
          .drop_duplicates()
          .head(30)
          .tolist())

for i, name in enumerate(sample):
    print(f"{i+1:2d}. {name}")

CAE_NAME entries with ';': 4740
 1. ZDF; Zweites Deutsches Fernsehen; Anstalt des öffentlichen Rechts; HA Rechtemanagement und Zentraleinkauf; Abt. Produktions-; Verwaltungs- und Baueinkauf; HH 907
 2. AOK PLUS - Die Gesundheitskasse für Sachsen und Thüringen; Stabsbereich Recht; Zentrale Vergabestelle
 3. Dataport; Anstalt des öffentlichen Rechts
 4. Sächsisches Staatsministerium des Innern; Referat 15; Zentrale Vergabestelle
 5. Stadt Köln; 27/Zentrales Vergabeamt
 6. Bayerischer Rundfunk; Anstalt des öffentlichen Rechts
 7. Freistaat Bayern; vertreten durch die Universität Würzburg
 8. Freistaat Bayern; vertreten durch das Bayerische Landesamt für Steuern; Dienststelle Nürnberg
 9. Stadt Bielefeld; Amt für Personal; Organisation und Zentrale Leistungen
10. Bundesministerium für Familie; Senioren; Frauen und Jugend
11. Das Land Hessen; vertreten durch die Hessische Zentrale für Datenverarbeitung
12. Landeshauptstadt Dresden; Geschäftsbereich Allgemeine Verwaltung
13. IT-Dienstleistun

In [72]:
# check for 1) 2) etc. patterns
patterns = {
    'parenthesis': r'\b\d+\)',       # 1) 2) 3)
    'dot':         r'\b\d+\.',       # 1. 2. 3.
    'slash':       r'\b\d+/',        # 1/ 2/
    'diverse':     r'\bdiverse[:\s]', # diverse: / diverse 
}

for pat_name, pat in patterns.items():
    mask = data_clean_['CAE_NAME'].str.contains(pat, na=False, regex=True)
    print(f"Pattern '{pat_name}': {mask.sum()} cases")
    if mask.sum() > 0:
        print(data_clean_.loc[mask, 'CAE_NAME'].drop_duplicates().head(5).tolist())
    print()

Pattern 'parenthesis': 175 cases
['Bundesstadt Bonn; Liegenschaftsamt; Zentraler Vergabeservice (Amt 23-02)', 'Deutsche Bahn AG; Vorstandsressort Technik; Systemverbund und Dienstleistungen - Beschaffung; Strategischer Einkauf IT Hardware (TEA 6)', 'Deutsche Bahn AG; Vorstandsressort Technik; Systemverbund und Dienstleistungen – Beschaffung; Strategischer Einkauf IT Hardware (TEA 6)', 'Bundesstadt Bonn; Zentrales Vergabeamt (Amt 25)', 'Der AOK-Bundesverband GbR im Namen der im I.1) genannten Auftraggeberinnen']

Pattern 'dot': 148 cases
['Stadt Minden; Zentralbereich 0.4; Personal und TUIV', 'Universität Paderborn Dez. 1.6 - Beschaffung', 'Stadtschulamt Frankfurt; 40.22.1 IT-Koordination & Beschaffung', 'FernUniversität in Hagen; Dezernat 5.4', 'Kreisausschuss des Lahn-Dill-Kreises; Fachdienst 11.2 Gebäude und Technik']

Pattern 'slash': 129 cases
['Stadt Köln; 27/Zentrales Vergabeamt', '24/7 IT Services GmbH', 'Landeshauptstadt Düsseldorf; Der Oberbürgermeister; Hauptamt 10/11', 'Stad

In [73]:
# Check for uninformative CAE_NAME entries
# Inspect first, then decide what to drop/fix
uninformative_patterns = [
    'keine angabe', 'aufgehoben', 'nicht bekannt', 'unbekannt',
    'entfaellt', 'entfallt', 'siehe',
    'wurde', 'losweise', 'wettbewerb',
    's. pkt', 'sh. pkt'   # additional abbreviation variants
]

mask_uninformative = data_clean_['CAE_NAME'].str.lower().apply(
    lambda x: any(p in x for p in uninformative_patterns)
    if isinstance(x, str) else False
)

print(f"Potentially uninformative CAE_NAME: {mask_uninformative.sum()}")
print("\nUnique values flagged:")
flagged = (data_clean_.loc[mask_uninformative, 'CAE_NAME']
           .value_counts()
           .head(30))
print(flagged.to_string())

Potentially uninformative CAE_NAME: 13

Unique values flagged:
CAE_NAME
Stadtverwaltung Filderstadt und 3 weitere Kommunen (siehe VI.3)                                                           7
Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (siehe Punkt VI.3)     2
Teilnahmewettbewerb mit anschließendem Verhandlungsverfahren "Dokumentenerstellung und Outputmanagement" (DOM)            1
Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (s. Pkt. VI.2))        1
Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (sh. Pkt VI.2))        1
Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (siehe Punkt VI.3))    1


In [74]:
# Explicit fix dictionary
CAE_FIX_DICT = {
    # Filderstadt: multi-buyer suffix stripped, kept as legitimate buyer
    'Stadtverwaltung Filderstadt und 3 weitere Kommunen (siehe VI.3)': 
        'Stadtverwaltung Filderstadt',
    
    # Bayerische Versorgungskammer: parenthetical suffix stripped
    'Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (siehe Punkt VI.3)': 
        'Bayerische Versorgungskammer',
    'Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (s. Pkt. VI.2))': 
        'Bayerische Versorgungskammer',
    'Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (sh. Pkt VI.2))': 
        'Bayerische Versorgungskammer',
    'Bayerische Versorgungskammer (im Namen der von ihr gesetzlich vertretenen Versorgungseinrichtungen (siehe Punkt VI.3))': 
        'Bayerische Versorgungskammer',
}

# Drop dictionary
CAE_DROP_SET = {
    # Procedure title, not a contracting authority - the buyer is unrecoverable.
    'Teilnahmewettbewerb mit anschließendem Verhandlungsverfahren "Dokumentenerstellung und Outputmanagement" (DOM)',
}

n_to_fix = data_clean_['CAE_NAME'].isin(CAE_FIX_DICT.keys()).sum()
n_to_drop = data_clean_['CAE_NAME'].isin(CAE_DROP_SET).sum()

if n_to_fix or n_to_drop:
    assert n_to_fix == 12, f"expected 12 rows to fix, found {n_to_fix}"
    assert n_to_drop == 1, f"expected 1 row to drop, found {n_to_drop}"
    data_clean_['CAE_NAME'] = data_clean_['CAE_NAME'].replace(CAE_FIX_DICT)
    data_clean_ = data_clean_[~data_clean_['CAE_NAME'].isin(CAE_DROP_SET)].copy()
    data_clean_ = data_clean_.reset_index(drop=True)

# End-state checks run on every execution
assert len(data_clean_) == 56276, "buyer frame row count changed"
assert data_clean_['CAE_NAME'].nunique() == 8387, "distinct buyer names changed"

print(f"Rows fixed:   {n_to_fix}")
print(f"Rows dropped: {n_to_drop}")
print(f"Remaining rows:   {len(data_clean_)}")
print(f"Unique CAE_NAME:  {data_clean_['CAE_NAME'].nunique()}")

Rows fixed:   12
Rows dropped: 1
Remaining rows:   56276
Unique CAE_NAME:  8387


In [75]:
# How much consolidation would head-truncation at ';' actually buy?
mask_semi = data_clean_['CAE_NAME'].str.contains(';', na=False)
names_semi = data_clean_.loc[mask_semi, 'CAE_NAME']

print(f"Rows with ';':            {mask_semi.sum()}")
print(f"Distinct names with ';':  {names_semi.nunique()}")

heads = names_semi.str.split(';').str[0].str.strip()
print(f"Distinct heads after truncation: {heads.nunique()}")

# Which heads absorb the most distinct full names?
collapse = (pd.DataFrame({'head': heads, 'full': names_semi})
            .groupby('head')['full'].nunique()
            .sort_values(ascending=False))
print(f"\nHeads absorbing >1 distinct name: {(collapse > 1).sum()}")
print(collapse.head(20).to_string())

Rows with ';':            4740
Distinct names with ';':  1020
Distinct heads after truncation: 574

Heads absorbing >1 distinct name: 155
head
Freistaat Bayern                                                  32
Landeshauptstadt München                                          23
Land Baden-Württemberg                                            21
Das Land Hessen                                                   17
Max-Planck-Gesellschaft zur Förderung der Wissenschaften e. V.    14
Landeshauptstadt Düsseldorf                                       13
Bundesamt für Ausrüstung                                          12
Landeshauptstadt Dresden                                          12
Land Berlin                                                       11
Land Hessen                                                       10
Freie und Hansestadt Hamburg                                       8
Bundesrepublik Deutschland                                         8
ZDF                          

### Semicolons in `CAE_NAME`: left untreated

Unlike on the supplier side, semicolons here carry three incompatible meanings:
authority vs. department, legal entity vs. the body procuring on its behalf, and
a plain substitute for a comma inside one name. No single rule resolves all
three, so these names are kept as they are.

### Departmental identifiers

Numeric patterns in `CAE_NAME` (`Amt 23-02`, `Dezernat 5.4`, `27/Zentrales
Vergabeamt`) mark administrative sub-units of the same authority. They are part
of the semicolon structure described above and are treated the same way: left
intact.

In [76]:
data_clean_.to_csv("data_checkpoints/data_pre_buyer_er.csv")

In [77]:
data_clean_ = pd.read_csv("data_checkpoints/data_pre_buyer_er.csv")

## Entity Resolution - Buyer Side
Same four-step approach as on supplier side: 1. normalization, 2. blocking, 3. similarity scores, 4. network-based clustering.
Again, both false positives and false negatives were checked

In [78]:
# step I

# Buyer-specific stopwords:
# Generic administrative terms with low discriminative value.
# Crucially, proper nouns (munich, niedersachsen etc.)
# and meaningful identifiers (bundesamt, landesamt) that distinguish entities are kept
BUYER_STOPWORDS = {
    # organisational units
    'amt', 'abteilung', 'referat', 'dezernat', 'sachgebiet', 'fachbereich',
    'fachamt', 'stabsbereich', 'bereich', 'stelle', 'dienststelle',
    'zentralbereich', 'zentraler', 'zentrale', 'hauptamt',
    # procurement-specific
    'vergabestelle', 'vergabeamt', 'vergabe', 'beschaffung', 'beschaffungsstelle',
    'einkauf', 'zentraleinkauf', 'strategischer',
    # legal form designations for public entities
    'anstalt', 'oeffentlichen', 'rechts', 'aoer', 'koerperschaft',
    'zweckverband', 'eigenbetrieb',
    # representation language
    'vertreten', 'durch', 'dieser', 'diese', 'dieses',
    # generic geographic/organisational qualifiers
    'deutschland', 'bundesrepublik', 'germany',
}

# Legal form patterns specific to public entities
PUBLIC_LEGAL_FORMS = r'\b(gmbh|ag|kg|ev|e\.v\.|aoer|aör|gbr|ggmbh|mbh|co)\b'

def normalize_cae_name(name):
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    # Umlaut replacement
    name = (name.replace('ä', 'ae').replace('ö', 'oe')
                .replace('ü', 'ue').replace('ß', 'ss'))
    # Replace ALL punctuation with space before the ASCII fold. Otherwise
    # non-ASCII punctuation (en dash, em dash) is deleted rather than
    # replaced, gluing tokens together: 'Max–Planck' -> 'maxplanck'.
    name = re.sub(r'[^\w\s]', ' ', name, flags=re.UNICODE)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode()
    # Remove legal forms
    name = re.sub(PUBLIC_LEGAL_FORMS, '', name)
    # Replace all non-alphanumeric (incl. semicolons, dashes, brackets) with space
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    # Tokenize, remove stopwords and single characters
    tokens = [t for t in name.split()
              if t not in BUYER_STOPWORDS and len(t) > 1]
    return ' '.join(tokens).strip()

# Apply normalization
data_clean_['CAE_NAME_NORM'] = data_clean_['CAE_NAME'].apply(normalize_cae_name)

# Token-sorted variant for blocking (invariant to word order)
data_clean_['CAE_NAME_NORM_SORTED'] = data_clean_['CAE_NAME_NORM'].apply(
    lambda x: ' '.join(sorted(x.split())) if isinstance(x, str) else x
)

# Sanity check
print(f"Empty CAE_NAME_NORM: {(data_clean_['CAE_NAME_NORM'] == '').sum()}")
print(f"Unique CAE_NAME_NORM_SORTED: {data_clean_['CAE_NAME_NORM_SORTED'].nunique()}")

# Spot check: compare raw vs normalized for a sample of cases
print("\n=== Spot check: raw vs. normalized ===")
spot = data_clean_[['CAE_NAME', 'CAE_NAME_NORM']].drop_duplicates().sample(15, random_state=42)
for _, row in spot.iterrows():
    print(f"  RAW:  {row['CAE_NAME']}")
    print(f"  NORM: {row['CAE_NAME_NORM']}")
    print()

Empty CAE_NAME_NORM: 6
Unique CAE_NAME_NORM_SORTED: 6869

=== Spot check: raw vs. normalized ===
  RAW:  Klinikum Chemnitz gGmbH c/o Klinikum Chemnitz Logistik- und Wirtschaftsgesellschaft mbH Zentrales Beschaffungsmanagement
  NORM: klinikum chemnitz klinikum chemnitz logistik und wirtschaftsgesellschaft zentrales beschaffungsmanagement

  RAW:  Universitätsstadt Tübingen
  NORM: universitaetsstadt tuebingen

  RAW:  Universität Siegen
  NORM: universitaet siegen

  RAW:  Das Land Hessen; vertreten durch das Regierungspräsidium Gießen
  NORM: das land hessen das regierungspraesidium giessen

  RAW:  Studierendenwerk Karlsruhe AöR
  NORM: studierendenwerk karlsruhe

  RAW:  Markt Bechhofen
  NORM: markt bechhofen

  RAW:  SWU Energie GmbH
  NORM: swu energie

  RAW:  Kreis Herzogtum Lauenburg
  NORM: kreis herzogtum lauenburg

  RAW:  Bundesamt für Ausrüstung; Informationstechnik und Ausrüstung der Bundeswehr; Referat E2.4
  NORM: bundesamt fuer ausruestung informationstechnik und ausr

In [79]:
# 1. Inspect the 6 empty CAE_NAME_NORM cases
print("=== Empty CAE_NAME_NORM ===")
empty_mask = data_clean_['CAE_NAME_NORM'] == ''
print(data_clean_.loc[empty_mask, ['ID_NOTICE_CAN', 'CAE_NAME', 'CAE_TYPE', 'WIN_NAME']]
      .to_string())

# 2. Inspect the suspicious case (looks like a contract description, not a buyer)
print("\n=== Suspicious CAE_NAME entries (look like contract descriptions) ===")
# Heuristic: entries starting with a verb or containing procurement action words
# Heuristic: CAE_NAME that reads like a contract title rather than an authority.
# Word boundary required, 'Betrieb' excluded (matches legitimate public entities
# such as 'Betrieb für Bau und Liegenschaften M-V').
suspicious_mask = data_clean_['CAE_NAME'].str.contains(
    r'^(?:Lieferung|Beschaffung|Kauf|Bereitstellung|Erstellung|Wartung|Planung)\b',
    na=False, regex=True
)
print(f"Potentially suspicious entries: {suspicious_mask.sum()}")
print(data_clean_.loc[suspicious_mask, 'CAE_NAME'].drop_duplicates().to_string())

=== Empty CAE_NAME_NORM ===
       ID_NOTICE_CAN                    CAE_NAME CAE_TYPE                                                                                                          WIN_NAME
30296     2020583295      Zentrale Vergabestelle        3                                                                                                      Bechtle GmbH
30297     2020583295      Zentrale Vergabestelle        3                                                                                                      Bechtle GmbH
49135     2023210220                  Dezernat 2        6                                                                                                   PeGo Consulting
52551     2023515160          Beschaffungsstelle        1                                                                                                         SINC GmbH
54877     2020217856  Bundesrepublik Deutschland        1               Rechtsanwalt Ottmar Hermann als Insolven

In [80]:
# Inspect empty cases more carefully across all available columns
print("=== Empty CAE_NAME_NORM - full context ===")
empty_mask = data_clean_['CAE_NAME_NORM'] == ''
print(data_clean_.loc[empty_mask, [
    'ID_NOTICE_CAN', 'CAE_NAME', 'CAE_TYPE', 'MAIN_ACTIVITY',
    'TAL_LOCATION_NUTS', 'WIN_NAME', 'YEAR'
]].to_string())

=== Empty CAE_NAME_NORM - full context ===
       ID_NOTICE_CAN                    CAE_NAME CAE_TYPE            MAIN_ACTIVITY TAL_LOCATION_NUTS                                                                                                          WIN_NAME  YEAR
30296     2020583295      Zentrale Vergabestelle        3  General public\services             DEA5C                                                                                                      Bechtle GmbH  2020
30297     2020583295      Zentrale Vergabestelle        3  General public\services             DEA5C                                                                                                      Bechtle GmbH  2020
49135     2023210220                  Dezernat 2        6                Education             DE916                                                                                                   PeGo Consulting  2023
52551     2023515160          Beschaffungsstelle        1  General public

In [81]:
# Drop rows where CAE_NAME_NORM is empty after normalization
empty_mask = data_clean_['CAE_NAME_NORM'] == ''
print(f"Dropping {empty_mask.sum()} rows with empty CAE_NAME_NORM")
data_clean_ = data_clean_[~empty_mask].copy().reset_index(drop=True)
print(f"Remaining rows: {len(data_clean_)}")

Dropping 6 rows with empty CAE_NAME_NORM
Remaining rows: 56270


In [82]:
# Contract titles recorded in the buyer field. Word boundaries keep legitimate
# authorities ('Beschaffungsamt', 'Beschaffungsstelle') out of the match.
CAE_DESCRIPTION_PATTERN = (
    r'^(?:Lieferung|Wartung|Beschaffung\b|Erstellung|'
    r'Bereitstellung|Installation|Kauf von|Betrieb und)'
)

description_mask = data_clean_['CAE_NAME'].str.match(CAE_DESCRIPTION_PATTERN, na=False)
print(f"Likely contract descriptions in CAE_NAME: {description_mask.sum()}")
print(f"Unique cases: {data_clean_.loc[description_mask, 'CAE_NAME'].nunique()}")
print(data_clean_.loc[description_mask, 'CAE_NAME'].drop_duplicates().to_string())

Likely contract descriptions in CAE_NAME: 28
Unique cases: 18
1021     Betrieb und Betreuung technischer IT-Anlagen u...
3158     Beschaffung u. Betreuung der IT-Infrastruktur ...
6171        Beschaffung von Arbeitsplatzdruckern (Kyocera)
8079     Lieferung; Aufstellung; Wartung inkl. Systemse...
8288               Lieferung von Kopierpapier im Jahr 2014
15216    Beschaffung, Installation und Betrieb von WLAN...
21439    Beschaffung mobiler Ticketautomaten (MTA 1.1) ...
28758                                 Beschaffung Schul-IT
29582    Lieferung von Touchdisplays inkl. Zubehör für ...
30507    Lieferung von Büromaterial, Breifhüllen, Versa...
30508    Lieferung von Büromaterial, Breifhüllen, Versa...
30509    Lieferung von Büromaterial, Breifhüllen, Versa...
30510    Lieferung von Büromaterial, Breifhüllen, Versa...
30511    Lieferung von Büromaterial, Breifhüllen, Versa...
30512    Lieferung von Büromaterial, Briefhüllen, Versa...
31745    Beschaffung der Software für Planning, Analy

In [83]:
# dropping rows
description_mask = data_clean_['CAE_NAME'].str.match(CAE_DESCRIPTION_PATTERN, na=False)
n_drop = description_mask.sum()

if n_drop:
    assert n_drop == 28, f"expected 25 rows, found {n_drop}"
    data_clean_ = data_clean_[~description_mask].copy().reset_index(drop=True)

assert len(data_clean_) == 56242, "buyer frame row count changed"
print(f"Rows dropped: {n_drop}, remaining: {len(data_clean_)}")

Rows dropped: 28, remaining: 56242


In [84]:
# stept II
# Get unique normalized (token-sorted) buyer names for blocking
unique_cae_sorted = data_clean_['CAE_NAME_NORM_SORTED'].dropna().unique()
print(f"Unique CAE_NAME_NORM_SORTED for blocking: {len(unique_cae_sorted)}")

# TF-IDF vectorization at character n-gram level (same as supplier pipeline)
vec_cae = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_cae = vec_cae.fit_transform(unique_cae_sorted).toarray().astype('float32')

# Normalize for cosine similarity via inner product
faiss.normalize_L2(X_cae)

# Build FAISS index and retrieve top-10 candidates per name
index_cae = faiss.IndexFlatIP(X_cae.shape[1])
index_cae.add(X_cae)
D_cae, I_cae = index_cae.search(X_cae, k=10)

print(f"Blocking complete: {len(unique_cae_sorted)} x 10 candidate pairs generated")

Unique CAE_NAME_NORM_SORTED for blocking: 6850
Blocking complete: 6850 x 10 candidate pairs generated


In [85]:
# Is k=10 sufficient, or is the candidate list saturated by boilerplate?
tenth = D_cae[:, -1]   # similarity of the least similar retrieved neighbour
for thr in [0.7, 0.8, 0.9]:
    share = (tenth >= thr).mean()
    print(f"10th neighbour still >= {thr}: {share:.1%} of names")

10th neighbour still >= 0.7: 4.2% of names
10th neighbour still >= 0.8: 0.8% of names
10th neighbour still >= 0.9: 0.0% of names


Candidate generation retrieves the ten nearest neighbours per name. This is not
a binding constraint: for no name does the tenth neighbour reach a cosine
similarity of 0.9, and only 0.8 % reach 0.8, well below the matching threshold.

In [86]:
# step III
# Build lookup: normalized_sorted_name -> CAE_TYPE
# Shown for inspection only. CAE_TYPE is recorded inconsistently for the same
# authority (numeric codes mixed with N/R/Z), so it is not used to reject
# candidate pairs.
cae_type_lookup = (
    data_clean_
    .groupby('CAE_NAME_NORM_SORTED')['CAE_TYPE']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    .to_dict()
)

THRESHOLD_CAE = 92

pairs_cae = []
n_type_differs = 0

for i, (dists, idxs) in enumerate(zip(D_cae, I_cae)):
    for dist, j in zip(dists, idxs):
        if i >= j:
            continue

        name_i = unique_cae_sorted[i]
        name_j = unique_cae_sorted[j]

        score = fuzz.token_sort_ratio(name_i, name_j)
        if score < THRESHOLD_CAE:
            continue

        type_i = cae_type_lookup.get(name_i)
        type_j = cae_type_lookup.get(name_j)
        if type_i != type_j:
            n_type_differs += 1

        pairs_cae.append((name_i, name_j, score, type_i, type_j))

assert len(pairs_cae) == 893, f"expected 893 candidate pairs, found {len(pairs_cae)}"
assert n_type_differs == 173, f"expected 173 type-mismatched pairs, found {n_type_differs}"

print(f"Candidate pairs at threshold {THRESHOLD_CAE}: {len(pairs_cae)}")
print(f"  of which CAE_TYPE differs: {n_type_differs} (retained, see note above)")

# Quick distribution check
scores = [p[2] for p in pairs_cae]
print(f"\nScore distribution:")
print(f"  Min:  {min(scores):.1f}")
print(f"  Mean: {sum(scores)/len(scores):.1f}")
print(f"  Max:  {max(scores):.1f}")

# Show a sample of pairs for sanity check
print("\n=== Sample pairs (random) ===")
rng = random.Random(42)
for a, b, score, t_i, t_j in rng.sample(pairs_cae, min(15, len(pairs_cae))):
    flag = ' *' if t_i != t_j else ''
    print(f"  [{t_i}/{t_j}]{flag} {score:.0f} | {a} <-> {b}")

Candidate pairs at threshold 92: 893
  of which CAE_TYPE differs: 173 (retained, see note above)

Score distribution:
  Min:  92.0
  Mean: 95.4
  Max:  99.7

=== Sample pairs (random) ===
  [6/8] * 93 | hessischer rundfunk <-> hessischer hr rundfunk
  [1/1] 99 | bundesamt fuer hydrographie seeschifffahrt und <-> bundesamt fuer hydrographier seeschifffahrt und
  [6/6] 92 | deutsche pfalz rentenversicherung rheinland <-> deutsche rentenversicherung rheinland
  [3/3] 97 | karlsruhe lzfd oberfinanzdirektion <-> karlsrueh lzfd oberfinanzdirektion
  [6/6] 93 | aachen hochschule rheinisch technische westfaelische <-> hochschule rheinisch technische westfaelische
  [3/3] 93 | fuer geoinformation landesamt und vermessung <-> bayern fuer geoinformation landesamt und vermessung
  [6/6] 97 | bundesvereinigung gkv kassenaerztliche oe oe spitzenverband und <-> bundesvereinigung gkv kassenaerztliche oe oe spitzenverband
  [8/8] 93 | dachverband dienstleister it kdn kommunaler <-> dachverband der dien

In [87]:
# step IV
# Build graph from candidate pairs
G_cae = nx.Graph()
G_cae.add_edges_from([(p[0], p[1]) for p in pairs_cae])

# Default mapping: every normalized name resolves to its most frequent raw
# spelling. Without this, exact normalization duplicates that form no fuzzy
# pair would keep separate canonical names ('BITMARCK' vs 'Bitmarck').
mapping_cae = (
    data_clean_
    .assign(_raw=data_clean_['CAE_NAME'].str.strip())
    .groupby('CAE_NAME_NORM_SORTED')['_raw']
    .agg(lambda s: s.mode()[0])
    .to_dict()
)

# Clusters override the default with one canonical name per connected component
for component in nx.connected_components(G_cae):
    mask = data_clean_['CAE_NAME_NORM_SORTED'].isin(component)

    # Count by lowercased raw name to collapse case variants
    counts = Counter(
        data_clean_.loc[mask, 'CAE_NAME']
        .str.strip().str.lower()
    )
    canonical_lower = counts.most_common(1)[0][0]

    # Recover representative raw name with original capitalization
    canonical_raw = (
        data_clean_.loc[
            data_clean_['CAE_NAME'].str.strip().str.lower() == canonical_lower,
            'CAE_NAME'
        ].mode()[0]
    )
    for norm_name in component:
        mapping_cae[norm_name] = canonical_raw

# Apply mapping
data_clean_['CAE_NAME_CANONICAL'] = data_clean_['CAE_NAME_NORM_SORTED'].map(mapping_cae)
assert data_clean_['CAE_NAME_CANONICAL'].notna().all(), "unmapped normalized names"

# check
print(f"Unique CAE_NAME before resolution: {data_clean_['CAE_NAME'].nunique()}")
print(f"Unique CAE_NAME_CANONICAL:         {data_clean_['CAE_NAME_CANONICAL'].nunique()}")
print(f"Clusters formed: {nx.number_connected_components(G_cae)}")

print("\n=== Largest clusters (5+ variants) ===")
large_clusters = [c for c in nx.connected_components(G_cae) if len(c) >= 5]
print(f"Clusters with 5+ variants: {len(large_clusters)}")

for cluster in sorted(large_clusters, key=len, reverse=True)[:10]:
    print(f"\n  Size {len(cluster)}:")
    mask = data_clean_['CAE_NAME_NORM_SORTED'].isin(cluster)
    raw_names = data_clean_.loc[mask, 'CAE_NAME'].str.strip().unique()
    for name in sorted(raw_names):
        print(f"    {name}")

Unique CAE_NAME before resolution: 8365
Unique CAE_NAME_CANONICAL:         6108
Clusters formed: 569

=== Largest clusters (5+ variants) ===
Clusters with 5+ variants: 8

  Size 15:
    Der AOK Bundesverband führt das Vergabeverfahren im Auftrag der unten genannten Auftraggeberinnen durch
    Der AOK-Bundesverband GbR führt das Vergabeverfahren im Auftrag der unter Ziffer VI.3 genannten Auftraggeberin durch.
    Der AOK-Bundesverband GbR führt im Namen der unter VI.2 genannten Auftraggeberinnen das Vergabeverfahren durch
    Der AOK-Bundesverband führte das Vergabeverfahren im Namen der unter VI.3) genannten Auftraggeberinnen durch
    Der AOK-Bundesverband führte im Auftrag der unter VI.2 genannten Auftraggeberinnen das Vergabeverfahren durch.
    Der AOK-Bundesverband führte im Namen der unter Ziffer VI.2 genannten Auftraggeber das Vergabeverfahren durch
    Der AOK-Bundesverband führte im Namen der unter Ziffer VI.2 genannten Auftraggeber das Vergabeverfahren durch.
    Der AOK-Bund

False negative check

In [88]:
# --- False Negative Check: suspiciously similar canonical buyer names ---

canonical_cae_names = data_clean_["CAE_NAME_CANONICAL"].unique().tolist()

suspicious_pairs_cae = []
for i, name_a in enumerate(canonical_cae_names):
    for name_b in canonical_cae_names[i+1:]:
        score = fuzz.token_sort_ratio(name_a.lower(), name_b.lower())
        if score >= 95:
            suspicious_pairs_cae.append((name_a, name_b, score))

suspicious_df_cae = pd.DataFrame(
    suspicious_pairs_cae,
    columns=["name_a", "name_b", "score"]
)

print(f"Total suspicious pairs (>= 95): {len(suspicious_df_cae)}")
print(suspicious_df_cae.sort_values("score", ascending=False).to_string(index=False))

Total suspicious pairs (>= 95): 7
                                                                                                                                       name_a                                                                                                                                                 name_b  score
                                                                                                         VBG Verwaltungs-Berufsgenossenschaft                                                                                                                    VBG Verwaltungsberufsgenossenschaft  98.59
                                                                                                             Große Kreisstadt WaldshutTiengen                                                                                                                      Große Kreisstadt Waldshut-Tiengen  98.46
                                                                  

In [89]:
# Manual resolution of false negatives from the automated matching step.
# All pairs below scored 95+ on the raw canonical names but fell short of the
# token_sort_ratio threshold of 92 on the normalized form. Each was verified
# individually; the target is the officially correct spelling.
CAE_CANONICAL_FIX = {
    'VBG Verwaltungsberufsgenossenschaft':
        'VBG Verwaltungs-Berufsgenossenschaft',
    'Große Kreisstadt WaldshutTiengen':
        'Große Kreisstadt Waldshut-Tiengen',
    'Engagement Global gGmH':
        'Engagement Global gGmbH',
    'Öffentlich-rechtliche Rundfunkanstalten der BRD; ZDF; Deutschlandradio '
    'vertreten durch die Geschäftsführung der Gebühreneinzugszentrale (GEZ)':
        'Öffentlich-rechtliche Landesrundfunkanstalten der BRD; ZDF und '
        'Deutschlandradio vertreten durch die Geschäftsführung der '
        'Gebühreneinzugszentrale (GEZ)',
    'IT.Niedersachsen FG54- Zentrale Vergabestelle IT':
        'IT.Niedersachsen - Zentrale Vergabestelle IT',
    # MDK-IT was renamed MD-IT in 2021; the current name is used throughout.
    'MDK-IT GmbH':
        'MD-IT GmbH',
    'Land Baden-Württemberg vertretung durch Landtag von Baden-Württemberg':
        'Land Baden-Württemberg vertreten durch den Landtag von Baden-Württemberg',
}

n_before = data_clean_['CAE_NAME_CANONICAL'].nunique()
data_clean_['CAE_NAME_CANONICAL'] = (
    data_clean_['CAE_NAME_CANONICAL'].replace(CAE_CANONICAL_FIX)
)

# The AOK cluster consists of procedural sentences naming the AOK-Bundesverband
# as the body conducting the procedure. Following the rule applied on the
# supplier side, the named representative is treated as the buyer.
aok_mask = (
    data_clean_['CAE_NAME_CANONICAL'].str.contains('AOK', na=False)
    & data_clean_['CAE_NAME_CANONICAL'].str.contains('Vergabeverfahren', na=False)
)
print(f"AOK procedural rows: {aok_mask.sum()}")
print(f"Distinct canonical names collapsed: "
      f"{data_clean_.loc[aok_mask, 'CAE_NAME_CANONICAL'].nunique()}")
data_clean_.loc[aok_mask, 'CAE_NAME_CANONICAL'] = 'AOK-Bundesverband GbR'

assert data_clean_['CAE_NAME_CANONICAL'].nunique() == 6096, "canonical buyer count changed"

print(f"Unique CAE_NAME_CANONICAL before: {n_before}")
print(f"Unique CAE_NAME_CANONICAL after:  {data_clean_['CAE_NAME_CANONICAL'].nunique()}")

AOK procedural rows: 53
Distinct canonical names collapsed: 5
Unique CAE_NAME_CANONICAL before: 6108
Unique CAE_NAME_CANONICAL after:  6096


In [90]:
data_clean_.to_csv("data_checkpoints/data_cae_er.csv")

In [91]:
data_clean_ = pd.read_csv("data_checkpoints/data_cae_er.csv")

### Solving representation structures
Many cases include structures like "vertreten durch", indicating different patterns of represenation. The following chunks solve this issue.

In [92]:
# Most frequent tokens in canonical buyer names
stopwords = {'der', 'die', 'das', 'des', 'dem', 'den', 'und', 'für',
             'von', 'zu', 'zur', 'zum', 'am', 'im', 'in', 'an', 'auf',
             'bei', 'mit', 'nach', 'über', 'e.v.', 'e.v', 'gmbh', '-'}

tokens = Counter()
for name in data_clean_['CAE_NAME_CANONICAL'].unique():
    for tok in str(name).lower().split():
        if tok not in stopwords and len(tok) > 1:
            tokens[tok] += 1

print("\n--- Top 50 tokens in CAE_NAME_CANONICAL ---")
for tok, count in tokens.most_common(50):
    print(f"  {tok:<30} {count:>5}")


--- Top 50 tokens in CAE_NAME_CANONICAL ---
  stadt                            694
  durch                            412
  vertreten                        408
  landkreis                        240
  zentrale                         204
  land                             198
  klinikum                         187
  ggmbh                            163
  mbh                              159
  ministerium                      157
  landratsamt                      152
  asklepios                        150
  hochschule                       147
  baden-württemberg                143
  universität                      141
  vergabestelle                    136
  berlin                           120
  klinik                           119
  amt                              110
  e.                               109
  aör                              103
  stadtverwaltung                  103
  deutsche                          99
  v.                                99
  landesamt        

In [93]:
# Explore the "vertreten durch" pattern
vd_buyers = (data_clean_[['CAE_NAME_CANONICAL']]
             .drop_duplicates()
             .assign(has_vd=lambda df: df['CAE_NAME_CANONICAL']
                     .str.contains(r'vertreten\s+durch', case=False, na=False)))

vd_names = vd_buyers[vd_buyers['has_vd']]['CAE_NAME_CANONICAL']
print(f"Buyers with 'vertreten durch': {len(vd_names)}\n")

# Extract the part before "vertreten durch" as parent candidate
vd_parents = (vd_names.str.split(r'\s*vertreten\s+durch\s*', regex=True)
              .str[0].str.strip())

print("--- Most frequent parent candidates ---")
print(vd_parents.value_counts().head(20).to_string())

Buyers with 'vertreten durch': 364

--- Most frequent parent candidates ---
CAE_NAME_CANONICAL
Land Baden-Württemberg         34
Freistaat Bayern               21
Freistaat Bayern;              18
Das Land Hessen                18
Land Baden-Württemberg;        13
Freistaat Bayern,              12
Das Land Hessen;                9
Bundesrepublik Deutschland,     9
Land Rheinland-Pfalz,           9
Land Berlin;                    5
Freistaat Sachsen,              5
Land Baden-Württemberg,         5
Land Berlin                     5
Land Niedersachsen;             4
Land Hessen;                    4
Freistaat Thüringen             4
Land Hessen,                    4
Freistaat Thüringen;            3
Freistaat Sachsen;              3
Land Berlin,                    3


In [94]:
# Step 1: Strip prefix + articles
vd_pattern = r'^.+?\s*[,;]?\s*vertreten\s+durch\s+'
article_pattern = r'^(die|der|das|den|dem|einen?|eines)\s+'

canonical_series = data_clean_[['CAE_NAME_CANONICAL']].drop_duplicates().copy()
canonical_series['STRIPPED'] = (canonical_series['CAE_NAME_CANONICAL']
                                .str.replace(vd_pattern, '', regex=True, case=False)
                                .str.strip()
                                .str.strip(',;')
                                .str.replace(article_pattern, '', regex=True, case=False)
                                .str.strip())

changed = canonical_series[
    canonical_series['CAE_NAME_CANONICAL'] != canonical_series['STRIPPED']
].copy()

print(f"Buyers with prefix stripped: {len(changed)}")
print("\n--- Spot check ---")
for _, row in changed.head(10).iterrows():
    print(f"  {row['CAE_NAME_CANONICAL']}")
    print(f"    -> {row['STRIPPED']}\n")

Buyers with prefix stripped: 412

--- Spot check ---
  Freistaat Bayern vertreten durch die Universität Würzburg
    -> Universität Würzburg

  Freistaat Bayern; vertreten durch das Bayerische Landesamt für Steuern; Dienststelle Nürnberg
    -> Bayerische Landesamt für Steuern; Dienststelle Nürnberg

  Land Nordrhein-Westfalen vertreten durch das Gemeinsame Gebietsrechenzentrum Hagen
    -> Gemeinsame Gebietsrechenzentrum Hagen

  Das Land Hessen vertreten durch die Hessische Zentrale für Datenverarbeitung
    -> Hessische Zentrale für Datenverarbeitung

  Das Land Hessen; vertreten durch das Hessische Ministerium des Innern und für Sport; vertreten durch die Verwaltungsfachhochschule Wiesbaden
    -> Hessische Ministerium des Innern und für Sport; vertreten durch die Verwaltungsfachhochschule Wiesbaden

  Bundesrepublik Deutschland, vertreten durch das Beschaffungsamt des BMI
    -> Beschaffungsamt des BMI

  Stadt Barmstedt; Am Markt 1; 25355 Barmstedt vertreten durch die KUBUS Kommu

In [95]:
# How often does the tail resolve to a private service provider rather than
# a public body? Legal-form suffixes are the marker.
tail_private = changed['STRIPPED'].str.contains(
    r'\b(?:GmbH|AG|KG|mbH|GbR|OHG)\b', case=False, na=False, regex=True
)
print(f"Stripped names carrying a commercial legal form: {tail_private.sum()} of {len(changed)}")
print(changed.loc[tail_private, ['CAE_NAME_CANONICAL', 'STRIPPED']].to_string())

Stripped names carrying a commercial legal form: 40 of 412
                                                                                                                                                                                                                                                           CAE_NAME_CANONICAL                                                                                                                                                                                                                                                            STRIPPED
112                                                                                                                                                                  Stadt Barmstedt; Am Markt 1; 25355 Barmstedt vertreten durch die KUBUS Kommunalberatung und Service GmbH                                                                                                                                              

In [96]:
# Step 2: Fuzzy match stripped names against all existing canonicals
all_canonical = data_clean_['CAE_NAME_CANONICAL'].unique()
# Exclude the "vertreten durch" names themselves as match targets
target_canonical = [c for c in all_canonical if c not in changed['CAE_NAME_CANONICAL'].values]

results = []
for _, row in changed.iterrows():
    stripped = row['STRIPPED']
    best_score = 0
    best_match = None
    for target in target_canonical:
        score = fuzz.token_sort_ratio(stripped, target)
        if score > best_score:
            best_score = score
            best_match = target
    results.append({
        'CAE_NAME_CANONICAL': row['CAE_NAME_CANONICAL'],
        'STRIPPED': stripped,
        'BEST_MATCH': best_match,
        'SCORE': best_score
    })

match_df = pd.DataFrame(results)

# Show matches at established threshold
high_matches = match_df[match_df['SCORE'] >= 92].sort_values('SCORE', ascending=False)
print(f"Matches at score >= 92: {len(high_matches)}")
print(f"\n--- Proposed merges ---")
for _, row in high_matches.iterrows():
    print(f"  {row['CAE_NAME_CANONICAL']}")
    print(f"    stripped: {row['STRIPPED']}")
    print(f"    -> match: {row['BEST_MATCH']} (score: {row['SCORE']:.0f})\n")

Matches at score >= 92: 134

--- Proposed merges ---
  Das Land Hessen vertreten durch die Hessische Zentrale für Datenverarbeitung
    stripped: Hessische Zentrale für Datenverarbeitung
    -> match: Hessische Zentrale für Datenverarbeitung (score: 100)

  Land Baden-Württemberg vertreten durch das Ministerium für Wirtschaft; Arbeit und Wohnungsbau Baden-Württemberg
    stripped: Ministerium für Wirtschaft; Arbeit und Wohnungsbau Baden-Württemberg
    -> match: Ministerium für Wirtschaft; Arbeit und Wohnungsbau Baden-Württemberg (score: 100)

  Freistaat Bayern vertreten durch die Julius-Maximilians-Universität Würzburg
    stripped: Julius-Maximilians-Universität Würzburg
    -> match: Julius-Maximilians-Universität Würzburg (score: 100)

  Land Berlin, vertreten durch Tegel Projekt GmbH
    stripped: Tegel Projekt GmbH
    -> match: Tegel Projekt GmbH (score: 100)

  Freistaat Thüringen vertreten durch das Thüringer Finanzministerium
    stripped: Thüringer Finanzministerium
    -> 

In [97]:
# Also check: any stripped names that now match EACH OTHER (new clusters)
stripped_list = changed['STRIPPED'].tolist()
new_clusters = []
for a, b in combinations(range(len(stripped_list)), 2):
    score = fuzz.token_sort_ratio(stripped_list[a], stripped_list[b])
    if score >= 92:
        new_clusters.append((changed.iloc[a]['CAE_NAME_CANONICAL'],
                             changed.iloc[b]['CAE_NAME_CANONICAL'],
                             stripped_list[a], stripped_list[b], score))

print(f"\nNew inter-VD clusters (stripped names matching each other): {len(new_clusters)}")
for a_orig, b_orig, a_str, b_str, score in new_clusters[:15]:
    print(f"  {a_str} <-> {b_str} (score: {score:.0f})")
    print(f"    from: {a_orig}")
    print(f"    from: {b_orig}\n")


New inter-VD clusters (stripped names matching each other): 245
  KUBUS Kommunalberatung und Service GmbH <-> KUBUS Kommunalberatung und Service GmbH (score: 100)
    from: Stadt Barmstedt; Am Markt 1; 25355 Barmstedt vertreten durch die KUBUS Kommunalberatung und Service GmbH
    from: Stadt Bargteheide vertreten durch KUBUS Kommunalberatung und Service GmbH

  Logistikzentrum Baden-Württemberg <-> Logistikzentrum Baden-Württemberg (score: 100)
    from: Ministerium für Kultus; Jugend und Sport Baden-Württemberg; vertreten durch das Logistikzentrum Baden-Württemberg
    from: Oberlandesgericht Stuttgart; vertreten durch das Logistikzentrum Baden-Württemberg

  Logistikzentrum Baden-Württemberg <-> Logistikzentrum Baden-Württemberg (score: 100)
    from: Ministerium für Kultus; Jugend und Sport Baden-Württemberg; vertreten durch das Logistikzentrum Baden-Württemberg
    from: Pädagogische Hochschule Ludwigsburg vertreten durch das Logistikzentrum Baden-Württemberg

  Logistikzentrum B

In [98]:
# Identify central procurement agencies by frequency as "vertreten durch" target
vd_buyers = data_clean_['CAE_NAME_CANONICAL'].drop_duplicates()
vd_buyers = vd_buyers[vd_buyers.str.contains(r'vertreten\s+durch', case=False, na=False)]

# Extract part AFTER "vertreten durch" (= the representative)
article_pattern = r'^(die|der|das|den|dem|einen?|eines)\s+'
representatives = (vd_buyers.str.split(r'\s*[,;]?\s*vertreten\s+durch\s*', regex=True)
                   .str[-1]
                   .str.strip()
                   .str.strip(',;')
                   .str.replace(article_pattern, '', regex=True, case=False)
                   .str.strip())

# Also extract part BEFORE "vertreten durch" (= the principal)
principals = (vd_buyers.str.split(r'\s*[,;]?\s*vertreten\s+durch\s*', regex=True)
              .str[0]
              .str.strip()
              .str.strip(',;'))

vd_df = pd.DataFrame({
    'CAE_NAME_CANONICAL': vd_buyers.values,
    'principal': principals.values,
    'representative': representatives.values
})

# Count how often each representative appears
rep_counts = vd_df['representative'].value_counts()

print("--- Representatives appearing 2+ times (likely central procurement) ---")
print(rep_counts[rep_counts >= 2].to_string())
print(f"\n--- Representatives appearing only once ---")
print(f"Count: {(rep_counts == 1).sum()}")

--- Representatives appearing 2+ times (likely central procurement) ---
representative
Gebäudemanagement Schleswig-Holstein AöR            20
Logistikzentrum Baden-Württemberg                    9
Gebäudemanagement Schleswig-Holstein AöR (GMSH)      5
Direktion der Regens-Wagner-Stiftungen               5
Berliner Immobilienmanagement GmbH                   4
Staatsbetrieb Sächsische Informatik Dienste          3
Deutsche Bahn AG Beschaffung Informationssysteme     3
Präsidenten                                          3
Beschaffungsamt des BMI                              2
KUBUS Kommunalberatung und Service GmbH              2
Oberfinanzdirektion Frankfurt am Main                2
Bundesamt für Bauwesen und Raumordnung               2
Thüringer Landesfinanzdirektion                      2
Brandenburgischen IT-Dienstleister (ZIT-BB)          2
Landrat                                              2
Bayerische Staatsministerium der Justiz              2
Ministerium der Justiz           

In [99]:
# Check the parsing artifacts - what do the original names look like?
problem_reps = ['Präsidenten', 'Landrat', 'Kanzler',
                ': Landrat Stefan Sternberg',
                ': Gebäudemanagement Schleswig-Holstein AöR',
                ': Bayerisches Staatsministerium des Innern; für Bau und Verkehr (s. VI.2))']

print("--- Parsing artifact originals ---")
for rep in problem_reps:
    originals = vd_df[vd_df['representative'] == rep]['CAE_NAME_CANONICAL'].tolist()
    for o in originals:
        print(f"  REP: {rep}")
        print(f"  ORIG: {o}\n")

--- Parsing artifact originals ---
  REP: Präsidenten
  ORIG: Universität Kassel, vertreten durch den Präsidenten

  REP: Präsidenten
  ORIG: Hochschule Fulda; vertreten durch den Präsidenten

  REP: Präsidenten
  ORIG: Justus-Liebig-Universität Gießen; vertreten durch den Präsidenten

  REP: Landrat
  ORIG: Landkreis Neu-Ulm; dieser vertreten durch den Landrat

  REP: Landrat
  ORIG: Landkreis Gotha, vertreten durch den Landrat

  REP: Kanzler
  ORIG: Freistaat Bayern vertreten durch die Universität Regensburg, vertreten durch den Kanzler

  REP: : Landrat Stefan Sternberg
  ORIG: Landkreis Ludwigslust-Parchim – vertreten durch: Landrat Stefan Sternberg

  REP: : Gebäudemanagement Schleswig-Holstein AöR
  ORIG: Hamburgische Wohnungsbaukreditanstalt vertreten durch: Gebäudemanagement Schleswig-Holstein AöR

  REP: : Bayerisches Staatsministerium des Innern; für Bau und Verkehr (s. VI.2))
  ORIG: Freistaat Bayern sowie: Sozialversicherungsträger; vertreten durch: Bayerisches Staatsminis

In [100]:
# Define central procurement agencies
central_procurement = [
    'Gebäudemanagement Schleswig-Holstein',
    'Logistikzentrum Baden-Württemberg',
    'Berliner Immobilienmanagement',
    'Staatsbetrieb Sächsische Informatik Dienste',
    'Beschaffungsamt des BMI',
    'Brandenburgischen IT-Dienstleister',
    'KUBUS Kommunalberatung und Service',
    'IT-Servicezentrum der bayerischen Justiz',
    'Oberfinanzdirektion Frankfurt',
]

# Tag each VD case
vd_df['is_central_procurement'] = vd_df['representative'].apply(
    lambda r: any(cp.lower() in r.lower() for cp in central_procurement))

print(f"Total VD cases: {len(vd_df)}")
print(f"Central procurement (keep principal as buyer): {vd_df['is_central_procurement'].sum()}")
print(f"Regular (strip prefix, match representative):  {(~vd_df['is_central_procurement']).sum()}")

print("\n--- Central procurement cases (spot check) ---")
for _, row in vd_df[vd_df['is_central_procurement']].head(10).iterrows():
    print(f"  BUYER should be: {row['principal']}")
    print(f"  (not: {row['representative']})\n")

Total VD cases: 364
Central procurement (keep principal as buyer): 54
Regular (strip prefix, match representative):  310

--- Central procurement cases (spot check) ---
  BUYER should be: Bundesrepublik Deutschland
  (not: Beschaffungsamt des BMI)

  BUYER should be: Stadt Barmstedt; Am Markt 1; 25355 Barmstedt
  (not: KUBUS Kommunalberatung und Service GmbH)

  BUYER should be: Ministerium für Kultus; Jugend und Sport Baden-Württemberg
  (not: Logistikzentrum Baden-Württemberg)

  BUYER should be: Oberlandesgericht Stuttgart
  (not: Logistikzentrum Baden-Württemberg)

  BUYER should be: Hamburgische Wohnungsbaukreditanstalt
  (not: : Gebäudemanagement Schleswig-Holstein AöR)

  BUYER should be: Pferdesportverband Schleswig-Holstein
  (not: Gebäudemanagement Schleswig-Holstein AöR)

  BUYER should be: Pädagogische Hochschule Ludwigsburg
  (not: Logistikzentrum Baden-Württemberg)

  BUYER should be: Duale Hochschule Baden-Württemberg Villingen-Schwenningen
  (not: Logistikzentrum Baden-

In [101]:
# Improved split: handle "vertreten durch:" and "endvertreten durch"
vd_pattern_improved = r'\s*[,;]?\s*(?:end)?vertreten\s+durch[:\s]\s*'

# Also handle double "vertreten durch" (take first split only for principal)
canonical_vd = data_clean_['CAE_NAME_CANONICAL'].drop_duplicates()
canonical_vd = canonical_vd[canonical_vd.str.contains(r'vertreten\s+durch', case=False, na=False)]

article_pat = r'^(die|der|das|den|dem|einen?|eines)\s+'

rows = []
for name in canonical_vd:
    parts = re.split(vd_pattern_improved, name, flags=re.IGNORECASE)
    parts = [p.strip().strip(',;') for p in parts if p.strip()]
    
    if len(parts) >= 2:
        principal = parts[0]
        representative = re.sub(article_pat, '', parts[-1], flags=re.IGNORECASE).strip()
        # For double VD ("Bayern vd Uni Regensburg vd Kanzler"), middle part is the real buyer
        middle = re.sub(article_pat, '', parts[1], flags=re.IGNORECASE).strip() if len(parts) >= 3 else None
    else:
        principal = name
        representative = None
        middle = None
    
    rows.append({
        'CAE_NAME_CANONICAL': name,
        'principal': principal,
        'representative': representative,
        'middle_entity': middle
    })

vd_df = pd.DataFrame(rows)

# --- Categorize ---

# Title representatives (person, not org)
title_pattern = r'^(Präsident|Landrat|Kanzler|Bürgermeister|Oberbürgermeister|Geschäftsführ|Direktor|Vorstand)'
vd_df['is_title'] = vd_df['representative'].str.contains(title_pattern, case=False, na=False)

# Central procurement agencies
central_procurement = [
    'Gebäudemanagement Schleswig-Holstein',
    'Logistikzentrum Baden-Württemberg',
    'Berliner Immobilienmanagement',
    'Staatsbetrieb Sächsische Informatik Dienste',
    'Beschaffungsamt des BMI',
    'Brandenburgischen IT-Dienstleister',
    'KUBUS Kommunalberatung',
    'IT-Servicezentrum der bayerischen Justiz',
    'Oberfinanzdirektion Frankfurt',
]
vd_df['is_central'] = vd_df['representative'].apply(
    lambda r: any(cp.lower() in str(r).lower() for cp in central_procurement) if r else False)

# Overly generic principals (would need further resolution)
generic_principals = r'^(Bundesrepublik Deutschland|Freistaat|Land\s|Das Land\s|Saarland)'
vd_df['principal_is_generic'] = vd_df['principal'].str.contains(generic_principals, case=False, na=False)

# Assign category
def categorize(row):
    if row['is_title']:
        return 'TITLE (use principal)'
    if pd.notna(row['middle_entity']):
        return 'DOUBLE_VD (use middle entity)'
    if row['is_central'] and not row['principal_is_generic']:
        return 'CENTRAL_PROC (use principal)'
    if row['is_central'] and row['principal_is_generic']:
        return 'CENTRAL_PROC_GENERIC (needs manual resolution)'
    return 'REGULAR (strip prefix, use representative)'

vd_df['category'] = vd_df.apply(categorize, axis=1)

print("--- Category counts ---")
print(vd_df['category'].value_counts().to_string())

print("\n--- Spot check per category ---")
for cat in vd_df['category'].unique():
    subset = vd_df[vd_df['category'] == cat]
    print(f"\n=== {cat} ({len(subset)} cases) ===")
    for _, row in subset.head(5).iterrows():
        print(f"  ORIG: {row['CAE_NAME_CANONICAL']}")
        if pd.notna(row['middle_entity']):
            print(f"  -> USE: {row['middle_entity']}")
        elif 'principal' in cat.lower() or 'title' in cat.lower():
            print(f"  -> USE: {row['principal']}")
        else:
            print(f"  -> USE: {row['representative']}")
        print()

--- Category counts ---
category
REGULAR (strip prefix, use representative)        251
DOUBLE_VD (use middle entity)                      40
CENTRAL_PROC (use principal)                       35
TITLE (use principal)                              27
CENTRAL_PROC_GENERIC (needs manual resolution)     11

--- Spot check per category ---

=== REGULAR (strip prefix, use representative) (251 cases) ===
  ORIG: Freistaat Bayern vertreten durch die Universität Würzburg
  -> USE: Universität Würzburg

  ORIG: Freistaat Bayern; vertreten durch das Bayerische Landesamt für Steuern; Dienststelle Nürnberg
  -> USE: Bayerische Landesamt für Steuern; Dienststelle Nürnberg

  ORIG: Land Nordrhein-Westfalen vertreten durch das Gemeinsame Gebietsrechenzentrum Hagen
  -> USE: Gemeinsame Gebietsrechenzentrum Hagen

  ORIG: Das Land Hessen vertreten durch die Hessische Zentrale für Datenverarbeitung
  -> USE: Hessische Zentrale für Datenverarbeitung

  ORIG: Land Niedersachsen; vertreten durch den Landesbe

C:\Users\stoeh\AppData\Local\Temp\ipykernel_23364\2476854845.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  vd_df['is_title'] = vd_df['representative'].str.contains(title_pattern, case=False, na=False)
C:\Users\stoeh\AppData\Local\Temp\ipykernel_23364\2476854845.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  vd_df['principal_is_generic'] = vd_df['principal'].str.contains(generic_principals, case=False, na=False)


In [102]:
def categorize_v2(row):
    rep = str(row['representative']) if pd.notna(row['representative']) else ''
    principal = str(row['principal']) if pd.notna(row['principal']) else ''
    middle = row['middle_entity']
    
    # Double VD: use the LAST entity (representative), not the middle
    if pd.notna(middle):
        return 'DOUBLE_VD (use last entity)'
    
    # Title representatives: "Präsidenten des OLG München" -> extract org from title
    title_pattern = r'^(Präsident\w*|Landrat\w*|Kanzler\w*|Bürgermeister\w*|Oberbürgermeister\w*|Geschäftsführ\w*|Direktor\w*|Vorstand\w*)\s'
    if re.match(title_pattern, rep, re.IGNORECASE):
        # Check if principal is generic (Land/Freistaat)
        if re.match(r'^(Bundesrepublik|Freistaat|Land\s|Das Land|Saarland)', principal, re.IGNORECASE):
            return 'TITLE_GENERIC_PRINCIPAL (extract org from title or keep full rep)'
        else:
            return 'TITLE (use principal)'
    
    # Central procurement with specific principal -> use principal
    if row['is_central'] and not row['principal_is_generic']:
        return 'CENTRAL_PROC (use principal)'
    
    # Everything else (including former CENTRAL_PROC_GENERIC) -> use representative
    return 'REGULAR (strip prefix, use representative)'

vd_df['category_v2'] = vd_df.apply(categorize_v2, axis=1)

print("--- Category counts v2 ---")
print(vd_df['category_v2'].value_counts().to_string())

print("\n--- Spot check per category ---")
for cat in vd_df['category_v2'].unique():
    subset = vd_df[vd_df['category_v2'] == cat]
    print(f"\n=== {cat} ({len(subset)} cases) ===")
    for _, row in subset.head(8).iterrows():
        print(f"  ORIG: {row['CAE_NAME_CANONICAL']}")
        if cat == 'DOUBLE_VD (use last entity)':
            print(f"  -> USE: {row['representative']}")
        elif 'use principal' in cat.lower():
            print(f"  -> USE: {row['principal']}")
        elif 'TITLE_GENERIC' in cat:
            print(f"  -> FULL REP: {row['representative']}")
            print(f"     (principal too generic: {row['principal']})")
        else:
            print(f"  -> USE: {row['representative']}")
        print()

--- Category counts v2 ---
category_v2
REGULAR (strip prefix, use representative)                           269
DOUBLE_VD (use last entity)                                           43
CENTRAL_PROC (use principal)                                          35
TITLE_GENERIC_PRINCIPAL (extract org from title or keep full rep)      9
TITLE (use principal)                                                  8

--- Spot check per category ---

=== REGULAR (strip prefix, use representative) (269 cases) ===
  ORIG: Freistaat Bayern vertreten durch die Universität Würzburg
  -> USE: Universität Würzburg

  ORIG: Freistaat Bayern; vertreten durch das Bayerische Landesamt für Steuern; Dienststelle Nürnberg
  -> USE: Bayerische Landesamt für Steuern; Dienststelle Nürnberg

  ORIG: Land Nordrhein-Westfalen vertreten durch das Gemeinsame Gebietsrechenzentrum Hagen
  -> USE: Gemeinsame Gebietsrechenzentrum Hagen

  ORIG: Das Land Hessen vertreten durch die Hessische Zentrale für Datenverarbeitung
  -> US

In [103]:
# Step 1: Build resolved buyer name for clear cases, flag problem cases

def resolve_buyer(row):
    cat = row['category_v2']
    rep = str(row['representative']) if pd.notna(row['representative']) else ''
    principal = str(row['principal']) if pd.notna(row['principal']) else ''
    
    if cat == 'REGULAR (strip prefix, use representative)':
        return rep, 'auto'
    elif cat == 'CENTRAL_PROC (use principal)':
        return principal, 'auto'
    elif cat == 'TITLE (use principal)':
        # Check for generic principals that slipped through
        if re.match(r'^(Die\s+)?Bundesrepublik', principal, re.IGNORECASE):
            return None, 'manual'
        return principal, 'auto'
    elif cat == 'DOUBLE_VD (use last entity)':
        # Flag if last entity starts with a person title
        title_pat = r'^(Präsident|Landrat|Kanzler|Bürgermeister|Geschäftsführ|Direktor|Vorstand|Leiter|Generaldirektor)'
        if re.match(title_pat, rep, re.IGNORECASE):
            return None, 'manual'
        return rep, 'auto'
    else:
        return None, 'manual'

vd_df[['resolved_buyer', 'resolution_type']] = vd_df.apply(
    resolve_buyer, axis=1, result_type='expand')

print(f"Auto-resolved: {(vd_df['resolution_type'] == 'auto').sum()}")
print(f"Needs manual:  {(vd_df['resolution_type'] == 'manual').sum()}")

# Step 2: Strip leftover articles from auto-resolved names
article_pat = r'^(die|der|das|den|dem|einen?|eines)\s+'
vd_df.loc[vd_df['resolution_type'] == 'auto', 'resolved_buyer'] = (
    vd_df.loc[vd_df['resolution_type'] == 'auto', 'resolved_buyer']
    .str.replace(article_pat, '', regex=True, case=False)
    .str.strip().str.strip(',;–-').str.strip()
)

# Step 3: Fuzzy match resolved names against existing canonicals
auto_df = vd_df[vd_df['resolution_type'] == 'auto'].copy()
all_canonical = set(data_clean_['CAE_NAME_CANONICAL'].unique())
# Exclude VD names themselves as match targets
vd_names = set(vd_df['CAE_NAME_CANONICAL'].values)
target_list = [c for c in all_canonical if c not in vd_names]

matches = []
for _, row in auto_df.iterrows():
    resolved = row['resolved_buyer']
    best_score, best_match = 0, None
    for target in target_list:
        score = fuzz.token_sort_ratio(resolved, target)
        if score > best_score:
            best_score = score
            best_match = target
    matches.append({'best_match': best_match, 'best_score': best_score})

auto_df = auto_df.assign(**pd.DataFrame(matches, index=auto_df.index))

# Split into confident matches and uncertain ones
confident = auto_df[auto_df['best_score'] >= 92].copy()
uncertain = auto_df[auto_df['best_score'] < 92].copy()

print(f"Confident matches (score >= 92): {len(confident)}")
print(f"Uncertain (score < 92):          {len(uncertain)}")
print(f"Manual cases:                    {(vd_df['resolution_type'] == 'manual').sum()}")

Auto-resolved: 348
Needs manual:  16
Confident matches (score >= 92): 126
Uncertain (score < 92):          222
Manual cases:                    16


In [104]:
# Step 4: Build the merge dictionary for confident cases
vd_merge_dict = dict(zip(confident['CAE_NAME_CANONICAL'], confident['best_match']))

print(f"VD merge dictionary: {len(vd_merge_dict)} entries")
print("\n--- Sample merges ---")
for orig, target in list(vd_merge_dict.items())[:10]:
    print(f"  {orig}")
    print(f"    -> {target}\n")

VD merge dictionary: 126 entries

--- Sample merges ---
  Das Land Hessen vertreten durch die Hessische Zentrale für Datenverarbeitung
    -> Hessische Zentrale für Datenverarbeitung

  Land Niedersachsen; vertreten durch den Landesbetrieb für Statistik und Kommunikationstechnologie Niedersachsen (LSKN)
    -> Landesbetrieb für Statistik und Kommunikationstechnologie Niedersachsen (LSKN)

  Land Baden-Württemberg; vertreten durch das Innenministerium Baden-Württemberg
    -> Innenminsiterium Baden-Württemberg

  Ministerium für Kultus; Jugend und Sport Baden-Württemberg; vertreten durch das Logistikzentrum Baden-Württemberg
    -> Ministerium für Kultus; Jugend und Sport Baden-Württemberg

  Oberlandesgericht Stuttgart; vertreten durch das Logistikzentrum Baden-Württemberg
    -> Oberlandesgericht Stuttgart

  Land Berlin; vertreten durch das Landesverwaltungsamt Berlin; LS
    -> Landesverwaltungsamt Berlin

  Land Berlin; vertreten durch das Bezirksamt Neukölln von Berlin
    -> Bezi

In [105]:
# Step 5: Export manual cases + uncertain cases for review
manual_cases = vd_df[vd_df['resolution_type'] == 'manual'][
    ['CAE_NAME_CANONICAL', 'principal', 'representative', 'category_v2']
].copy()
manual_cases['RESOLVED_BUYER'] = ''  # placeholder for you to fill in

uncertain_cases = uncertain[
    ['CAE_NAME_CANONICAL', 'resolved_buyer', 'best_match', 'best_score']
].copy()
uncertain_cases['ACCEPT_MATCH'] = ''  # placeholder: yes/no

print(f"\n=== MANUAL CASES ({len(manual_cases)}) ===")
for _, row in manual_cases.iterrows():
    print(f"  ORIG: {row['CAE_NAME_CANONICAL']}")
    print(f"  PRINCIPAL: {row['principal']}")
    print(f"  REPRESENTATIVE: {row['representative']}")
    print(f"  CATEGORY: {row['category_v2']}\n")

print(f"\n=== UNCERTAIN MATCHES ({len(uncertain_cases)}) ===")
for _, row in uncertain_cases.iterrows():
    print(f"  ORIG: {row['CAE_NAME_CANONICAL']}")
    print(f"  RESOLVED: {row['resolved_buyer']}")
    print(f"  BEST MATCH: {row['best_match']} (score: {row['best_score']:.0f})\n")


=== MANUAL CASES (16) ===
  ORIG: Stiftung Preußischer Kulturbesitz; vertreten durch den Präsidenten; dieser vertreten durch die Generaldirektorin der Staatsbibliothek zu Berlin-Preußischer Kulturbesitz
  PRINCIPAL: Stiftung Preußischer Kulturbesitz
  REPRESENTATIVE: Generaldirektorin der Staatsbibliothek zu Berlin-Preußischer Kulturbesitz
  CATEGORY: DOUBLE_VD (use last entity)

  ORIG: Freistaat Bayern, vertreten durch den Präsidenten des Oberlandesgerichts München
  PRINCIPAL: Freistaat Bayern
  REPRESENTATIVE: Präsidenten des Oberlandesgerichts München
  CATEGORY: TITLE_GENERIC_PRINCIPAL (extract org from title or keep full rep)

  ORIG: Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz - (federführend für die Landesjustizverwaltungen in der Bundesrepublik Deutschland)
  PRINCIPAL: Freistaat Bayern
  REPRESENTATIVE: Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz -

In [106]:
# --- CATEGORY A: confident merges (already in vd_merge_dict) ---
# --- CATEGORY B: uncertain cases ---

title_pat = r'^(Präsident|Landrat|Kanzler|Bürgermeister|Geschäftsführ|Direktor|Vorstand|Leiter|Herrn|Frau|Senator|Betriebsleitung$)'

uncertain['is_broken'] = uncertain['resolved_buyer'].str.contains(
    title_pat, case=False, na=False)

broken = uncertain[uncertain['is_broken']]
clean_renames = uncertain[~uncertain['is_broken']]

# Clean renames: just use the resolved name as new canonical (no merge needed)
rename_dict = dict(zip(clean_renames['CAE_NAME_CANONICAL'], clean_renames['resolved_buyer']))

print(f"Clean renames (VD name -> stripped name, no merge): {len(rename_dict)}")
print(f"Broken resolutions (need manual fix): {len(broken)}")

print("\n--- Broken resolutions ---")
for _, row in broken.iterrows():
    print(f"  ORIG: {row['CAE_NAME_CANONICAL']}")
    print(f"  BROKEN RESOLVED: {row['resolved_buyer']}")
    print(f"  PRINCIPAL: {vd_df.loc[vd_df['CAE_NAME_CANONICAL']==row['CAE_NAME_CANONICAL'], 'principal'].iloc[0]}")
    print()

Clean renames (VD name -> stripped name, no merge): 206
Broken resolutions (need manual fix): 16

--- Broken resolutions ---
  ORIG: Universitätsklinikum Freiburg; vertreten durch den Leiter des Klinikrechenzentrums; Herrn Udo Timmermann
  BROKEN RESOLVED: Leiter des Klinikrechenzentrums; Herrn Udo Timmermann
  PRINCIPAL: Universitätsklinikum Freiburg

  ORIG: Deutsches Museum; vertreten durch Herrn Prof. Dr. rer. nat. Wolfgang Heckl
  BROKEN RESOLVED: Herrn Prof. Dr. rer. nat. Wolfgang Heckl
  PRINCIPAL: Deutsches Museum

  ORIG: Universität Kassel, vertreten durch den Präsidenten
  BROKEN RESOLVED: Präsidenten
  PRINCIPAL: Universität Kassel

  ORIG: Freie und Hansestadt Hamburg; vertreten durch die Behörde für Schule und Berufsbildung; vertreten durch Herrn Senator Ties Rabe
  BROKEN RESOLVED: Herrn Senator Ties Rabe
  PRINCIPAL: Freie und Hansestadt Hamburg

  ORIG: Hochschule Fulda; vertreten durch den Präsidenten
  BROKEN RESOLVED: Präsidenten
  PRINCIPAL: Hochschule Fulda

  ORI

C:\Users\stoeh\AppData\Local\Temp\ipykernel_23364\2692481106.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  uncertain['is_broken'] = uncertain['resolved_buyer'].str.contains(


In [107]:
# --- CATEGORY C: manual cases + broken cases -> combined manual dictionary ---
# For these, the correct buyer needs to be set by hand

manual_all = pd.concat([
    vd_df[vd_df['resolution_type'] == 'manual'][['CAE_NAME_CANONICAL', 'principal', 'representative']],
    broken[['CAE_NAME_CANONICAL']].merge(
        vd_df[['CAE_NAME_CANONICAL', 'principal', 'representative']], on='CAE_NAME_CANONICAL')
])

print(f"\n=== TOTAL MANUAL CASES: {len(manual_all)} ===")
print("manual_override_dict = {")
for _, row in manual_all.iterrows():
    print(f"    # PRINCIPAL: {row['principal']}")
    print(f"    # REPRESENTATIVE: {row['representative']}")
    print(f"    '{row['CAE_NAME_CANONICAL']}': '',  # <-- fill in")
    print()
print("}")


=== TOTAL MANUAL CASES: 32 ===
manual_override_dict = {
    # PRINCIPAL: Stiftung Preußischer Kulturbesitz
    # REPRESENTATIVE: Generaldirektorin der Staatsbibliothek zu Berlin-Preußischer Kulturbesitz
    'Stiftung Preußischer Kulturbesitz; vertreten durch den Präsidenten; dieser vertreten durch die Generaldirektorin der Staatsbibliothek zu Berlin-Preußischer Kulturbesitz': '',  # <-- fill in

    # PRINCIPAL: Freistaat Bayern
    # REPRESENTATIVE: Präsidenten des Oberlandesgerichts München
    'Freistaat Bayern, vertreten durch den Präsidenten des Oberlandesgerichts München': '',  # <-- fill in

    # PRINCIPAL: Freistaat Bayern
    # REPRESENTATIVE: Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz - (federführend für die Landesjustizverwaltungen in der Bundesrepublik Deutschland)
    'Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz - (federführend für die Landesju

In [108]:
# --- SUMMARY ---
print("=== VD RESOLUTION SUMMARY ===")
print(f"Total VD cases:                          {len(vd_df)}")
print(f"A) Confident merges (auto, score >= 92): {len(vd_merge_dict)}")
print(f"B) Clean renames (no existing match):    {len(rename_dict)}")
print(f"C) Manual resolution needed:             {len(manual_all)}")
print(f"   Sum:                                  {len(vd_merge_dict) + len(rename_dict) + len(manual_all)}")

=== VD RESOLUTION SUMMARY ===
Total VD cases:                          364
A) Confident merges (auto, score >= 92): 126
B) Clean renames (no existing match):    206
C) Manual resolution needed:             32
   Sum:                                  364


In [109]:
manual_override_dict = {
    # --- OLG / Gemeinsame IT-Stelle Bayern (all -> same entity) ---
    'Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz - (federführend für die Landesjustizverwaltungen in der Bundesrepublik Deutschland)':
        'Gemeinsame IT-Stelle der bayerischen Justiz',
    'Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz -':
        'Gemeinsame IT-Stelle der bayerischen Justiz',
    'Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz - (federführend für die Landesjustizverwaltungen der Länder Baden-Württemberg; Berlin; Brandenburg; Freie und Hansestadt Hamburg; Mecklenburg-Vorpommern; Rheinland-Pfalz; Schleswig-Holstein und Thüringen)':
        'Gemeinsame IT-Stelle der bayerischen Justiz',
    'Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts München - Gemeinsame IT-Stelle der bayerischen Justiz - (federführend für die Landesjustizverwaltungen Baden-Württemberg; Bremen; Niedersachsen; Rheinland-Pfalz; Saarland; Sachsen-Anhalt; Sachsen; Thüringen)':
        'Gemeinsame IT-Stelle der bayerischen Justiz',
    'Freistaat Bayern, vertreten durch den Präsidenten des Oberlandesgerichts München':
        'Gemeinsame IT-Stelle der bayerischen Justiz',

    # --- Other courts (generic principal -> extract court) ---
    'Freistaat Bayern; vertreten durch den Präsidenten des Oberlandesgerichts Bamberg':
        'Oberlandesgericht Bamberg',
    'Freistaat Thüringen, endvertreten durch den Präsidenten des Thüringer Oberlandesgerichts':
        'Thüringer Oberlandesgericht',
    'Freistaat Bayern, vertreten durch den Präsidenten des Landgerichts Traunstein':
        'Landgericht Traunstein',
    'Freistaat Bayern, vertreten durch den Direktor des Amtsgerichts Rosenheim':
        'Amtsgericht Rosenheim',

    # --- Federal institutions (generic principal -> extract org) ---
    'Die Bundesrepublik Deutschland, vertreten durch den Präsidenten des Bundesrechnungshofes':
        'Bundesrechnungshof',
    'Bundesrepublik Deutschland, vertreten durch den Bundesminister der Justiz, dieser vertreten durch den Präsidenten des Bundesverwaltungsgerichts':
        'Bundesverwaltungsgericht',
    'Bundesrepublik Deutschland, diese vertreten durch den Bundesminister der Justiz, dieser vertreten durch die Präsidentin des Deutschen Patent- und Markenamtes':
        'Deutsches Patent- und Markenamt',

    # --- Double VD where principal is the correct buyer ---
    'Stiftung Preußischer Kulturbesitz; vertreten durch den Präsidenten; dieser vertreten durch die Generaldirektorin der Staatsbibliothek zu Berlin-Preußischer Kulturbesitz':
        'Staatsbibliothek zu Berlin',
    'Bundesagentur für Arbeit (BA), vertreten durch den Vorstand, hier vertreten durch die Leiterin des Geschäftsbereiches Einkauf im BA-Service-Haus':
        'Bundesagentur für Arbeit',
    'Land Nordrhein Westfalen; vertreten durch das Ministerium des Innern des Landes NRW; dieses vertreten durch den Leiter der Bezirksregierung Arnsberg':
        'Bezirksregierung Arnsberg',
    'Freie und Hansestadt Hamburg; vertreten durch die Behörde für Schule und Berufsbildung; vertreten durch Herrn Senator Ties Rabe':
        'Behörde für Schule und Berufsbildung Hamburg',

    # --- Uni Regensburg via Kanzler ---
    'Freistaat Bayern vertreten durch die Universität Regensburg, vertreten durch den Kanzler':
        'Universität Regensburg',

    # --- Simple title cases (principal = buyer) ---
    'Universitätsklinikum Freiburg; vertreten durch den Leiter des Klinikrechenzentrums; Herrn Udo Timmermann':
        'Universitätsklinikum Freiburg',
    'Deutsches Museum; vertreten durch Herrn Prof. Dr. rer. nat. Wolfgang Heckl':
        'Deutsches Museum',
    'Deutsches Museum von Meisterwerken der Naturwissenschaft und Technik (AdöR) vertreten durch Herrn Prof. Dr. rer. nat. Wolfgang M. Heckl':
        'Deutsches Museum',
    'Universität Kassel, vertreten durch den Präsidenten':
        'Universität Kassel',
    'Hochschule Fulda; vertreten durch den Präsidenten':
        'Hochschule Fulda',
    'Philipps-Universität Marburg; vertreten durch die Präsidentin':
        'Philipps-Universität Marburg',
    'Justus-Liebig-Universität Gießen; vertreten durch den Präsidenten':
        'Justus-Liebig-Universität Gießen',
    'Magistrat der Wissenschaftsstadt Darmstadt Eigenbetrieb für kommunale Aufgaben und Dienstleistungen (EAD) vertreten durch die Betriebsleitung':
        'Eigenbetrieb für kommunale Aufgaben und Dienstleistungen Darmstadt',

    # --- Landkreise via Landrat ---
    'Landkreis Neu-Ulm; dieser vertreten durch den Landrat':
        'Landkreis Neu-Ulm',
    'Landkreis Ansbach; Landratsamt Ansbach; vertreten durch Herrn Landrat Dr. Jürgen Ludwig':
        'Landratsamt Ansbach',
    'Landkreis Altötting, vertreten durch Herrn Landrat Erwin Schneider':
        'Landkreis Altötting',
    'Landkreis Gotha, vertreten durch den Landrat':
        'Landkreis Gotha',

    # --- Central procurement with title principal ---
    'Der Präsident des Schleswig-Holsteinischen Landtages vertreten durch die Gebäudemanagement Schleswig-Holstein AöR':
        'Schleswig-Holsteinischer Landtag',

    # --- Remaining special cases ---
    'Land Mecklenburg-Vorpommern, vertreten duch das Finanzministerium Mecklenburg-Vorpommern, endvertreten durch die Direktorin, Landesamt für Finanzen Mecklenburg-Vorpommern':
        'Landesamt für Finanzen Mecklenburg-Vorpommern',
    'Freie Hansestadt Bremen, vertreten durch den Senator für Justiz und Verfassung der Freien Hansestadt Bremen':
        'Senator für Justiz und Verfassung Bremen',
}

assert len(manual_override_dict) == 32, "manual override count changed"
assert set(manual_override_dict) == set(manual_all['CAE_NAME_CANONICAL']), \
    "manual override keys no longer match the generated manual cases"

print(f"Manual override dictionary: {len(manual_override_dict)} entries")

Manual override dictionary: 32 entries


In [110]:
# Combine all three dictionaries into one mapping
vd_full_mapping = {}
vd_full_mapping.update(rename_dict)
vd_full_mapping.update(vd_merge_dict)
vd_full_mapping.update(manual_override_dict)

# If any key appeared in more than one dictionary, the combined length would
# fall short of the sum - later updates would silently overwrite earlier ones.
assert len(vd_full_mapping) == len(rename_dict) + len(vd_merge_dict) + len(manual_override_dict), \
    "overlapping keys across the three dictionaries"
assert len(vd_full_mapping) == 364, "VD case count changed"

n_before = data_clean_['CAE_NAME_CANONICAL'].nunique()
data_clean_['CAE_NAME_CANONICAL'] = data_clean_['CAE_NAME_CANONICAL'].replace(vd_full_mapping)
n_after = data_clean_['CAE_NAME_CANONICAL'].nunique()

assert n_after == 5940, "canonical buyer count changed"

print(f"Total VD mappings: {len(vd_full_mapping)}")
print(f"Unique CAE_NAME_CANONICAL before: {n_before}")
print(f"Unique CAE_NAME_CANONICAL after:  {n_after}")
print(f"Reduction:                        {n_before - n_after}")

Total VD mappings: 364
Unique CAE_NAME_CANONICAL before: 6096
Unique CAE_NAME_CANONICAL after:  5940
Reduction:                        156


In [111]:
# Verify: spot check that merges landed correctly
print("--- Spot check: merged entries ---")
for orig, target in list(vd_merge_dict.items())[:5]:
    count = (data_clean_['CAE_NAME_CANONICAL'] == target).sum()
    print(f"  {target}: {count} contracts")

print("\n--- Spot check: renamed entries ---")
for orig, target in list(rename_dict.items())[:5]:
    count = (data_clean_['CAE_NAME_CANONICAL'] == target).sum()
    print(f"  {target}: {count} contracts")

print("\n--- Spot check: manual overrides ---")
for orig, target in list(manual_override_dict.items())[:5]:
    count = (data_clean_['CAE_NAME_CANONICAL'] == target).sum()
    print(f"  {target}: {count} contracts")

# Confirm no originals remain
remaining = data_clean_[data_clean_['CAE_NAME_CANONICAL'].isin(vd_full_mapping.keys())]
print(f"\nRows still containing old VD names: {len(remaining)}")

--- Spot check: merged entries ---
  Hessische Zentrale für Datenverarbeitung: 900 contracts
  Landesbetrieb für Statistik und Kommunikationstechnologie Niedersachsen (LSKN): 51 contracts
  Innenminsiterium Baden-Württemberg: 7 contracts
  Ministerium für Kultus; Jugend und Sport Baden-Württemberg: 12 contracts
  Oberlandesgericht Stuttgart: 7 contracts

--- Spot check: renamed entries ---
  Universität Würzburg: 26 contracts
  Bayerische Landesamt für Steuern; Dienststelle Nürnberg: 18 contracts
  Gemeinsame Gebietsrechenzentrum Hagen: 1 contracts
  Verwaltungsfachhochschule Wiesbaden: 1 contracts
  Beschaffungsamt des BMI: 491 contracts

--- Spot check: manual overrides ---
  Gemeinsame IT-Stelle der bayerischen Justiz: 16 contracts
  Gemeinsame IT-Stelle der bayerischen Justiz: 16 contracts
  Gemeinsame IT-Stelle der bayerischen Justiz: 16 contracts
  Gemeinsame IT-Stelle der bayerischen Justiz: 16 contracts
  Gemeinsame IT-Stelle der bayerischen Justiz: 16 contracts

Rows still con

In [112]:
# Would normalization have merged any of the renamed VD names with an
# existing buyer? Compares normalized forms rather than raw strings.
norm_existing = set(data_clean_.loc[
    ~data_clean_['CAE_NAME_CANONICAL'].isin(rename_dict.values()),
    'CAE_NAME_CANONICAL'
].map(normalize_cae_name))

hits = {new: normalize_cae_name(new) for new in set(rename_dict.values())}
overlap = {k: v for k, v in hits.items() if v in norm_existing}

print(f"Renamed names whose normalized form matches an existing buyer: {len(overlap)}")
for k in list(overlap)[:20]:
    print(f"  {k}")

Renamed names whose normalized form matches an existing buyer: 3
  IT-Verbund Stormarn
  bremenports GmbH &Co. KG
  Landtag Rheinland-Pfalz


In [113]:
# Representation formulas produce new buyer names that did not pass through
# the normalization step. Where such a name normalizes to exactly the same
# form as an existing buyer, the two are the same authority and are merged.
norm_to_existing = {}
for name in data_clean_['CAE_NAME_CANONICAL'].unique():
    if name in set(rename_dict.values()):
        continue
    norm_to_existing.setdefault(normalize_cae_name(name), name)

post_vd_merge = {}
for new_name in set(rename_dict.values()):
    target = norm_to_existing.get(normalize_cae_name(new_name))
    if target and target != new_name:
        post_vd_merge[new_name] = target

assert len(post_vd_merge) == 3, f"expected 3 post-VD merges, found {len(post_vd_merge)}"
for src, tgt in post_vd_merge.items():
    print(f"  {src}  ->  {tgt}")

data_clean_['CAE_NAME_CANONICAL'] = data_clean_['CAE_NAME_CANONICAL'].replace(post_vd_merge)
print(f"\nUnique CAE_NAME_CANONICAL: {data_clean_['CAE_NAME_CANONICAL'].nunique()}")

  IT-Verbund Stormarn  ->  IT-Verbund Stormarn AöR
  bremenports GmbH &Co. KG  ->  Bremenports GmbH & Co. KG
  Landtag Rheinland-Pfalz  ->  Landtag Rheinland - Pfalz; Z 2-2

Unique CAE_NAME_CANONICAL: 5937


In [114]:
data_clean_.to_csv("data_checkpoints/data_cae_2.csv")

In [115]:
data_clean = pd.read_csv("data_checkpoints/data_cae_2.csv")

### Parent Mapping Buyer-Side

In [116]:
# Identify corporate-family candidates on buyer side
# Based on token analysis: hospital chains, research networks, federal agency groups

family_patterns = {
    # Hospital chains
    'Asklepios': r'asklepios',
    'Helios': r'helios',
    'Sana': r'sana\s+klinik',
    'Rhön-Klinikum': r'rhön',
    'Vivantes': r'vivantes',
    'Charité': r'charité',
    
    # Research networks
    'Fraunhofer': r'fraunhofer',
    'Max-Planck': r'max[\-\s]planck',
    'Leibniz': r'leibniz',
    'Helmholtz': r'helmholtz',
    
    # Federal agencies with sub-units
    'Deutsche Bahn': r'deutsche\s+bahn',
    'Bundeswehr/BAAINBw': r'baainbw|bundeswehr|bundesamt.*ausrüstung',
    'Autobahn GmbH': r'autobahn\s+gmbh',
    'Deutsche Rentenversicherung': r'deutsche\s+rentenversicherung',
    'AOK': r'^aok[\s\-]',
    'Dataport': r'dataport',
    'ITZBund': r'itzbund|informationstechnikzentrum\s+bund',
    'BWI': r'^bwi\s',
    'Regens-Wagner': r'regens[\-\s]wagner',
    
    # Municipal utilities
    'Stadtwerke': r'^stadtwerke\s',
}

canonical = data_clean_.groupby('CAE_NAME_CANONICAL').size().reset_index(name='n_contracts')

print(f"{'Family':<30} {'Buyers':>6} {'Contracts':>10}  Examples")
print("-" * 100)

for label, pat in family_patterns.items():
    matches = canonical[canonical['CAE_NAME_CANONICAL'].str.contains(pat, case=False, na=False)]
    if len(matches) > 0:
        total_c = matches['n_contracts'].sum()
        examples = matches.nlargest(3, 'n_contracts')['CAE_NAME_CANONICAL'].tolist()
        print(f"{label:<30} {len(matches):>6} {total_c:>10}  {examples[0]}")
        for ex in examples[1:]:
            print(f"{'':>48}  {ex}")
        print()

Family                         Buyers  Contracts  Examples
----------------------------------------------------------------------------------------------------
Asklepios                         125        629  Asklepios Kliniken Hamburg GmbH
                                                  Asklepios Westklinikum Hamburg GmbH
                                                  Asklepios Klinik Fürstenhof Bad Wildungen GmbH

Helios                              2          4  Helios Amper-Klinikum Dachau
                                                  Helios Kliniken GmbH

Sana                               29        374  Sana Kliniken Düsseldorf GmbH
                                                  Sana Kliniken Leipziger Land GmbH
                                                  Sana Klinikum Remscheid GmbH

Rhön-Klinikum                       4          7  Landkreis Rhön-Grabfeld
                                                  RHÖN-KLINIKUM Campus Bad Neustadt
                     

In [117]:
# For families with >1 buyer: show all members with contract counts
# to decide which ones actually warrant a parent mapping

interesting = ['Asklepios', 'Fraunhofer', 'Max-Planck', 'Leibniz', 'Helmholtz',
               'Deutsche Bahn', 'Autobahn GmbH', 'Regens-Wagner', 'AOK',
               'Deutsche Rentenversicherung', 'Helios', 'Sana']

for label in interesting:
    pat = family_patterns[label]
    matches = canonical[canonical['CAE_NAME_CANONICAL'].str.contains(pat, case=False, na=False)]
    if len(matches) > 1:
        matches = matches.sort_values('n_contracts', ascending=False)
        print(f"\n=== {label} ({len(matches)} buyers, {matches['n_contracts'].sum()} contracts) ===")
        for _, row in matches.iterrows():
            print(f"  [{row['n_contracts']:>4}] {row['CAE_NAME_CANONICAL']}")


=== Asklepios (125 buyers, 629 contracts) ===
  [  42] Asklepios Kliniken Hamburg GmbH
  [  22] Asklepios Westklinikum Hamburg GmbH
  [  20] Asklepios Klinik Fürstenhof Bad Wildungen GmbH
  [  19] Asklepios Schwalm-Eder-Kliniken GmbH
  [  19] Asklepios Südpfalzkliniken GmbH
  [  18] Asklepios Klinik Lich GmbH
  [  18] Asklepios Klinik Gauting GmbH
  [  18] Asklepios Klinik Wiesbaden GmbH
  [  17] Asklepios Klinik Bad Wildungen GmbH
  [  17] Asklepios Fachklinikum Stadtroda GmbH
  [  17] Asklepios Klinik Alsbach GmbH
  [  17] Asklepios Klinikum Bad Abbach GmbH
  [  16] Asklepios Klinik Lindenlohe GmbH
  [  16] Asklepios Nordseeklinik Westerland GmbH
  [  16] Asklepios Klinik Pasewalk GmbH
  [  16] Asklepios Klinik Sankt Augustin GmbH
  [  16] Asklepios Klinik Lindau GmbH
  [  15] Asklepios Psychiatrie Niedersachsen GmbH
  [  15] Asklepios Klinik Weißenfels GmbH
  [  15] Asklepios Stadtklinik Bad Tölz GmbH
  [  15] Asklepios Klinik Sobernheim GmbH
  [  15] Asklepios Klinikum Uckermark G

In [118]:
# Check: large buyers not captured by any family pattern
all_patterns = '|'.join(family_patterns.values())
canonical_with_contracts = data_clean_.groupby('CAE_NAME_CANONICAL').size().reset_index(name='n')

unmatched_large = canonical_with_contracts[
    (~canonical_with_contracts['CAE_NAME_CANONICAL'].str.contains(all_patterns, case=False, na=False))
    & (canonical_with_contracts['n'] >= 20)
].sort_values('n', ascending=False)

print(f"Large buyers (>=20 contracts) not in any family pattern: {len(unmatched_large)}\n")
for _, row in unmatched_large.head(30).iterrows():
    print(f"  [{row['n']:>4}] {row['CAE_NAME_CANONICAL']}")

Large buyers (>=20 contracts) not in any family pattern: 484

  [ 900] Hessische Zentrale für Datenverarbeitung
  [ 645] Bundesagentur für Arbeit
  [ 491] Beschaffungsamt des BMI
  [ 476] Deutsche Bundesbank, Beschaffungszentrum
  [ 385] Beschaffungsamt des Bundesministeriums des Innern
  [ 367] Techniker Krankenkasse
  [ 331] Generalzolldirektion Zentrale Beschaffungsstelle der Bundesfinanzverwaltung
  [ 303] IT-Dienstleistungszentrum Berlin (AÖR)
  [ 271] Zentrale Vergabestelle im Bayerischen Landesamt für Steuern
  [ 248] Deutsches Zentrum für Luft- und Raumfahrt e. V.
  [ 239] Staatsbetrieb Sächsische Informatik Dienste
  [ 232] gkv informatik
  [ 230] Berliner Wasserbetriebe
  [ 226] Deutsche Bundesbank; Zentrale Beschaffungen
  [ 219] Logistikzentrum Baden-Württemberg
  [ 218] Bundesdruckerei GmbH
  [ 214] Westdeutscher Rundfunk Köln (WDR); Anstalt des öffentlichen Rechts
  [ 208] Landesbetrieb Information und Technik NRW
  [ 206] Vertragseinrichtungen der Prospitalia GmbH, Ulm
 

In [119]:
# Check for missed families and duplicates
check_patterns = {
    'Beschaffungsamt BMI': r'beschaffungsamt.*(bmi|bundesministerium.*innern)',
    'Deutsche Bundesbank': r'deutsche\s+bundesbank',
    'AMEOS': r'ameos',
    'Prospitalia': r'prospitalia',
    'EnBW': r'enbw|energie\s+baden.württemberg',
}

canonical_counts = data_clean_.groupby('CAE_NAME_CANONICAL').size().reset_index(name='n')

for label, pat in check_patterns.items():
    matches = canonical_counts[canonical_counts['CAE_NAME_CANONICAL'].str.contains(
        pat, case=False, na=False)].sort_values('n', ascending=False)
    if len(matches) > 0:
        print(f"\n=== {label} ({len(matches)} buyers, {matches['n'].sum()} contracts) ===")
        for _, row in matches.iterrows():
            print(f"  [{row['n']:>4}] {row['CAE_NAME_CANONICAL']}")


=== Beschaffungsamt BMI (4 buyers, 879 contracts) ===
  [ 491] Beschaffungsamt des BMI
  [ 385] Beschaffungsamt des Bundesministeriums des Innern
  [   2] Beschaffungsamt des BMI Brühler
  [   1] Kontaktstelle: Beschaffungsamt des BMI; (verantwortlicher Auftraggeber: ZVS; Dortmund)

=== Deutsche Bundesbank (5 buyers, 768 contracts) ===
  [ 476] Deutsche Bundesbank, Beschaffungszentrum
  [ 226] Deutsche Bundesbank; Zentrale Beschaffungen
  [  37] Deutsche Bundesbank
  [  26] Deutsche Bundesbank Hauptverwaltung Frankfurt am Main
  [   3] Deutsche Bundesbank Beschaffungszentrum; Vertrags- und Lizenzmanagement (400)

=== AMEOS (23 buyers, 1902 contracts) ===
  [ 202] AMEOS Krankenhausgesellschaft Baden mbH
  [ 129] AMEOS Klinikum Bad Salzuflen GmbH & Co. KG
  [ 129] AMEOS Klinikum Halberstadt GmbH
  [ 119] AMEOS Klinikum Geestland GmbH
  [ 109] AMEOS Krankenhausgesellschaft Niedersachsen mbH
  [ 109] AMEOS Klinikum Aschersleben-Staßfurt GmbH
  [ 109] AMEOS Klinikum Bernburg GmbH
  [ 109] 

C:\Users\stoeh\AppData\Local\Temp\ipykernel_23364\1333220478.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = canonical_counts[canonical_counts['CAE_NAME_CANONICAL'].str.contains(


In [120]:
# === STEP 1: Full-aggregate families (all variants -> one canonical name) ===

full_aggregate = {
    'BWI GmbH': r'^bwi\s',
    'Autobahn GmbH des Bundes': r'autobahn\s+gmbh',
    'Dataport AöR': r'dataport',
    'Informationstechnikzentrum Bund (ITZBund)': r'itzbund|informationstechnikzentrum\s+bund',
    'Fraunhofer-Gesellschaft': r'fraunhofer',
    'Deutsche Bahn AG': r'deutsche\s+bahn',
    'Asklepios': r'asklepios|^akg\s+klinik|^gkb\s+klinikbetriebe|sächsische\s+schweiz\s+kliniken.*asklepios|fachklinikum\s+wiesen',
    'Sana Kliniken': r'^sana\s',
    'Helios Kliniken': r'^helios\s',
    'Charité': r'charité',
    'AMEOS': r'^ameos\s',
    'Max-Planck-Gesellschaft': r'max[\-\s]planck',
    'EnBW': r'^enbw\s',
    'Beschaffungsamt des BMI': r'beschaffungsamt.*(?:bmi|bundesministerium.*innern)',
    'Deutsche Bundesbank': r'deutsche\s+bundesbank',
    'Vivantes': r'^vivantes|charité\s+vivantes',
}

parent_dict = {}
for parent_name, pattern in full_aggregate.items():
    matches = data_clean_['CAE_NAME_CANONICAL'].unique()
    matches = [m for m in matches if re.search(pattern, m, re.IGNORECASE)]
    for m in matches:
        if m != parent_name:
            parent_dict[m] = parent_name

print(f"Full-aggregate mappings: {len(parent_dict)}")
for parent in full_aggregate.keys():
    count = sum(1 for v in parent_dict.values() if v == parent)
    print(f"  {parent}: {count} variants merged")

# === STEP 2: Deutsche Rentenversicherung ===
# Only the sub-units of DRV Bund are consolidated. The regional carriers are
# legally independent institutions that procure separately, as are the shared
# IT companies (NOW IT, Rechenzentrum Würzburg), and are therefore left as
# separate contracting authorities.

drv_all = [c for c in data_clean_['CAE_NAME_CANONICAL'].unique()
           if re.search(r'deutsche\s+rentenversicherung', c, re.IGNORECASE)]

drv_bund_pattern = r'deutsche\s+rentenversicherung\s+bund'
for name in drv_all:
    if re.search(drv_bund_pattern, name, re.IGNORECASE):
        parent_dict[name] = 'Deutsche Rentenversicherung Bund'

n_drv = sum(1 for v in parent_dict.values() if v == 'Deutsche Rentenversicherung Bund')
print(f"\nDeutsche Rentenversicherung Bund: {n_drv} variants merged")
print(f"Regional carriers left untouched: {len(drv_all) - n_drv}")

# === STEP 3: AOK ===
# Same rule: the Bundesverband and its service companies are consolidated,
# the eleven regional AOKs are independent public-law corporations and remain
# separate contracting authorities.

aok_all = [c for c in data_clean_['CAE_NAME_CANONICAL'].unique()
           if re.search(r'^aok', c, re.IGNORECASE)]

for name in aok_all:
    if re.search(r'bundesverband|aok\s+systems|aok.*finanzmanagement', name, re.IGNORECASE):
        parent_dict[name] = 'AOK-Bundesverband'

# Same carrier, name extended by a note that it also acts for a shared
# computing centre. Consistent with the rule applied to consortia and to the
# AOK-Bundesverband: the named authority is the buyer.
for name in aok_all:
    if re.search(r'gesundheitskasse für niedersachsen', name, re.IGNORECASE):
        parent_dict[name] = 'AOK - Die Gesundheitskasse für Niedersachsen'

n_aok = sum(1 for v in parent_dict.values() if v == 'AOK-Bundesverband')
print(f"\nAOK-Bundesverband: {n_aok} variants merged")
print(f"Regional AOKs left untouched: {len(aok_all) - n_aok}")

print(f"\nTotal parent mappings: {len(parent_dict)}")

Full-aggregate mappings: 319
  BWI GmbH: 3 variants merged
  Autobahn GmbH des Bundes: 6 variants merged
  Dataport AöR: 1 variants merged
  Informationstechnikzentrum Bund (ITZBund): 1 variants merged
  Fraunhofer-Gesellschaft: 10 variants merged
  Deutsche Bahn AG: 16 variants merged
  Asklepios: 131 variants merged
  Sana Kliniken: 54 variants merged
  Helios Kliniken: 2 variants merged
  Charité: 3 variants merged
  AMEOS: 23 variants merged
  Max-Planck-Gesellschaft: 50 variants merged
  EnBW: 9 variants merged
  Beschaffungsamt des BMI: 3 variants merged
  Deutsche Bundesbank: 4 variants merged
  Vivantes: 3 variants merged

Deutsche Rentenversicherung Bund: 8 variants merged
Regional carriers left untouched: 24

AOK-Bundesverband: 8 variants merged
Regional AOKs left untouched: 19

Total parent mappings: 339


In [121]:
# Names caught by more than one family pattern. The loop above assigns them to
# whichever pattern is checked last, silently. Multi-brand hits were excluded
# on the supplier side and are inspected here as well.
hits = {m: [p for p, pat in full_aggregate.items() if re.search(pat, m, re.IGNORECASE)]
        for m in data_clean_['CAE_NAME_CANONICAL'].unique()}
multi = {m: p for m, p in hits.items() if len(p) > 1}

print(f"Names matching more than one family pattern: {len(multi)}")
for m, p in multi.items():
    print(f"  {m}  ->  {p}")

Names matching more than one family pattern: 1
  Labor Berlin - Charité Vivantes GmbH  ->  ['Charité', 'Vivantes']


In [122]:
# === SUMMARY & APPLY ===
print(f"\nTotal parent mappings before exclusions: {len(parent_dict)}")

# Joint venture of two families; multi-brand hits are not mapped, consistent
# with the rule applied on the supplier side.
for name in list(parent_dict):
    if len([p for p, pat in full_aggregate.items() if re.search(pat, name, re.IGNORECASE)]) > 1:
        del parent_dict[name]

print(f"Total parent mappings after exclusions:  {len(parent_dict)}")

print(f"\n--- By family ---")
family_counts = Counter(parent_dict.values())
for label, count in family_counts.most_common():
    print(f"  {label}: {count} mappings")

# Apply
n_before = data_clean_['CAE_NAME_CANONICAL'].nunique()
assert n_before == 5937, "buyer count before parent mapping changed"

data_clean_['CAE_NAME_CANONICAL'] = data_clean_['CAE_NAME_CANONICAL'].replace(parent_dict)
n_after = data_clean_['CAE_NAME_CANONICAL'].nunique()

print(f"\nUnique buyers before: {n_before}")
print(f"Unique buyers after:  {n_after}")
print(f"Reduction:            {n_before - n_after}")


Total parent mappings before exclusions: 339
Total parent mappings after exclusions:  338

--- By family ---
  Asklepios: 131 mappings
  Sana Kliniken: 54 mappings
  Max-Planck-Gesellschaft: 50 mappings
  AMEOS: 23 mappings
  Deutsche Bahn AG: 16 mappings
  Fraunhofer-Gesellschaft: 10 mappings
  EnBW: 9 mappings
  Deutsche Rentenversicherung Bund: 8 mappings
  AOK-Bundesverband: 8 mappings
  Autobahn GmbH des Bundes: 6 mappings
  Deutsche Bundesbank: 4 mappings
  AOK - Die Gesundheitskasse für Niedersachsen: 4 mappings
  BWI GmbH: 3 mappings
  Charité: 3 mappings
  Beschaffungsamt des BMI: 3 mappings
  Helios Kliniken: 2 mappings
  Vivantes: 2 mappings
  Dataport AöR: 1 mappings
  Informationstechnikzentrum Bund (ITZBund): 1 mappings

Unique buyers before: 5937
Unique buyers after:  5610
Reduction:            327


In [123]:
# final check
n_unique = data_clean_['CAE_NAME_CANONICAL'].nunique()
print(f"Final unique buyers: {n_unique}")

# Threshold analysis for buyers active in both periods (split at 2017)
contracts_all = data_clean_.groupby('CAE_NAME_CANONICAL').size().rename('n_contracts_all')
contracts_post = (data_clean_[data_clean_['YEAR'] >= 2017]
                  .groupby('CAE_NAME_CANONICAL').size().rename('n_contracts_post'))

buyer_stats_new = pd.concat([contracts_all, contracts_post], axis=1).fillna(0).astype(int)
buyer_stats_new['active_pre'] = buyer_stats_new.index.isin(
    data_clean_[data_clean_['YEAR'] < 2017]['CAE_NAME_CANONICAL'].unique())
buyer_stats_new['active_post'] = buyer_stats_new['n_contracts_post'] > 0
buyer_stats_new['active_both'] = buyer_stats_new['active_pre'] & buyer_stats_new['active_post']

print(f"\nActive only pre:  {(buyer_stats_new['active_pre'] & ~buyer_stats_new['active_post']).sum()}")
print(f"Active only post: {(~buyer_stats_new['active_pre'] & buyer_stats_new['active_post']).sum()}")
print(f"Active in both:   {buyer_stats_new['active_both'].sum()}")

# Threshold table for both-period buyers
both = buyer_stats_new[buyer_stats_new['active_both']]
post_both = both['n_contracts_post']
total = post_both.sum()

print(f"\n--- Both-period buyers: contracts post 2017 ---")
print(post_both.describe().to_string())

print(f"\nThreshold | Buyers  | % both-active | Contracts | % contracts")
print("-" * 70)
for t in [1, 2, 3, 5, 10, 15, 20, 50]:
    n = (post_both >= t).sum()
    c = post_both[post_both >= t].sum()
    print(f"  >={t:<6}| {n:>6}  | {n/len(post_both)*100:>12.1f}% | {c:>9,} | {c/total*100:.1f}%")

Final unique buyers: 5610

Active only pre:  1183
Active only post: 3234
Active in both:   1193

--- Both-period buyers: contracts post 2017 ---
count   1,193.00
mean       20.12
std        76.90
min         1.00
25%         2.00
50%         6.00
75%        17.00
max     2,093.00

Threshold | Buyers  | % both-active | Contracts | % contracts
----------------------------------------------------------------------
  >=1     |   1193  |        100.0% |    24,002 | 100.0%
  >=2     |    978  |         82.0% |    23,787 | 99.1%
  >=3     |    841  |         70.5% |    23,513 | 98.0%
  >=5     |    661  |         55.4% |    22,896 | 95.4%
  >=10    |    445  |         37.3% |    21,423 | 89.3%
  >=15    |    333  |         27.9% |    20,111 | 83.8%
  >=20    |    253  |         21.2% |    18,745 | 78.1%
  >=50    |     93  |          7.8% |    13,967 | 58.2%


In [124]:
# Full distribution across all periods
all_buyers = data_clean_.groupby('CAE_NAME_CANONICAL').agg(
    n_contracts=('CAE_NAME_CANONICAL', 'size'),
    n_years=('YEAR', 'nunique'),
    first_year=('YEAR', 'min'),
    last_year=('YEAR', 'max')
).sort_values('n_contracts', ascending=False)

print(f"Total unique buyers: {len(all_buyers)}")
print(f"\n--- Contract distribution (all buyers, all years) ---")
print(all_buyers['n_contracts'].describe().to_string())

print(f"\n--- Years active ---")
print(all_buyers['n_years'].describe().to_string())

total_contracts = all_buyers['n_contracts'].sum()
print(f"\nThreshold | Buyers  | % of all   | Contracts | % contracts")
print("-" * 70)
for t in [1, 2, 3, 5, 10, 15, 20, 50, 100]:
    n = (all_buyers['n_contracts'] >= t).sum()
    c = all_buyers.loc[all_buyers['n_contracts'] >= t, 'n_contracts'].sum()
    print(f"  >={t:<6}| {n:>6}  | {n/len(all_buyers)*100:>9.1f}% | {c:>9,} | {c/total_contracts*100:.1f}%")

Total unique buyers: 5610

--- Contract distribution (all buyers, all years) ---
count   5,610.00
mean       10.03
std        52.17
min         1.00
25%         1.00
50%         2.00
75%         6.00
max     2,210.00

--- Years active ---
count   5,610.00
mean        2.44
std         2.57
min         1.00
25%         1.00
50%         1.00
75%         3.00
max        15.00

Threshold | Buyers  | % of all   | Contracts | % contracts
----------------------------------------------------------------------
  >=1     |   5610  |     100.0% |    56,242 | 100.0%
  >=2     |   3348  |      59.7% |    53,980 | 96.0%
  >=3     |   2540  |      45.3% |    52,364 | 93.1%
  >=5     |   1766  |      31.5% |    49,718 | 88.4%
  >=10    |   1004  |      17.9% |    44,778 | 79.6%
  >=15    |    687  |      12.2% |    41,067 | 73.0%
  >=20    |    499  |       8.9% |    37,891 | 67.4%
  >=50    |    186  |       3.3% |    28,423 | 50.5%
  >=100   |     74  |       1.3% |    20,772 | 36.9%


In [125]:
# Same split by period for comparison
for period, label in [(range(2009, 2017), '2009-2016'), (range(2017, 2024), '2017-2023')]:
    subset = data_clean_[data_clean_['YEAR'].isin(period)]
    buyers = subset.groupby('CAE_NAME_CANONICAL').size()
    total = buyers.sum()
    print(f"\n--- {label}: {len(buyers)} buyers, {total} contracts ---")
    for t in [1, 3, 5, 10, 20]:
        n = (buyers >= t).sum()
        c = buyers[buyers >= t].sum()
        print(f"  >={t:<4} | {n:>5} buyers | {c:>7,} contracts ({c/total*100:.1f}%)")


--- 2009-2016: 2376 buyers, 15404 contracts ---
  >=1    |  2376 buyers |  15,404 contracts (100.0%)
  >=3    |   905 buyers |  13,541 contracts (87.9%)
  >=5    |   586 buyers |  12,459 contracts (80.9%)
  >=10   |   302 buyers |  10,599 contracts (68.8%)
  >=20   |   144 buyers |   8,485 contracts (55.1%)

--- 2017-2023: 4427 buyers, 40838 contracts ---
  >=1    |  4427 buyers |  40,838 contracts (100.0%)
  >=3    |  2023 buyers |  37,790 contracts (92.5%)
  >=5    |  1382 buyers |  35,608 contracts (87.2%)
  >=10   |   761 buyers |  31,530 contracts (77.2%)
  >=20   |   368 buyers |  26,113 contracts (63.9%)


In [126]:
data_clean_.to_csv("data_checkpoints/data_final.csv")